# ネットワーク構築を一歩ずつ確認するノートブック

オーナーの指示を1ステップずつセルにしていきます。各ステップは、(1) 指示の引用と **動作** / **出力** を書いた Markdown セル、(2) 結果を確認するコードセル、の組です。

## このノートブックのルール
- **プログラムはすべてこのノートブックの中に書き、日本語で説明する。** `targetnet` パッケージの関数に処理を隠さない（標準的なライブラリ pandas / numpy / networkx などは使う）。
- 1つの指示 = 1ステップ = 1つのコードセル（読み込み・加工・確認・図は分ける）。
- 指示を受けてからセルを追加する。先回りで作らない。
- 指示を元の考え方どおりに実現できない場合（データ不足、あいまい、元の方針と矛盾など）は、そのセルの **Note** に書いて知らせる。
- パラメータは最初のコードセルにまとめる。データベースの場所は `target_net.yaml` から読む。

## 0. パラメータ
- 動作: 設定ファイルの場所を指定する
- 出力: なし（設定のみ）

In [ ]:
# パラメータ（最初のセルに集約）
CONFIG = "../target_net.yaml"   # データベースの場所を書いた YAML
MAX_HOPS = 6                    # リガンドから転写因子までの最大ステップ数（枝刈りで使う）
TARGET = "AKT1"                  # 試験に使う標的遺伝子（ここから上流へ向かう）
UP_HOPS = 6                      # 上流へ何ステップまで調べるか
N_LIGANDS = 15                   # 図に描くリガンドの数（標的への経路スコアが高い順）
K_IN = 5                         # 各ノードで残す入ってくる辺の数（重みの大きい順）
REL_MIN = 0.8                    # そのノードの最大の重みに対する割合がこれ以上の辺だけ残す
X_PER_NODE = 10                  # Step 8: 各遺伝子で残す上流の数（直接の辺のスコア E の大きい順）
REF_RECEPTOR = "KDR"             # Step 9: 基準にする受容体（あなたのイメージ: KDR のシグナル）
REF_LIGANDS = ["VEGFA", "VEGFC", "VEGFD"]   # KDR に結合するリガンド
THRESHOLD_PATTERNS = [("geo", 0.2), ("geo", 0.25), ("geo", 0.4), ("geo", 0.6), ("mean", 0.3), ("mean", 0.5), ("min", 0.3), ("prod", 0.08), ("prod", 0.2), ("prod", 0.3)]   # Step 10: (合成スコアの種類, しきい値) の組。geo = √(weight×E) / mean = (weight+E)/2 / min = 小さいほう / prod = weight×E
FIG_PATTERN = "geo >= 0.25"      # Step 11: 図を描くパターン（Step 10 の名前）
TOP_N = 100                      # Step 13: 図に描く遺伝子の数（受容体 -> 標的の経路の流れが大きい順）
MAX_LIG_PER_REC = 3              # Step 14: 図に添える、受容体1つあたりのリガンドの最大数（辺のスコア E が大きい順）
N_REC = 15                       # Step 15: 図に描く受容体の数（標的への経路スコアが大きい順）
N_LIG = 15                       # Step 15: 図に描くリガンドの数（標的への経路スコアが大きい順）
N_MID = 60                       # Step 15: 図に描く途中のタンパク質の数（流れが大きい順。受容体の経路上のものは必ず入る）
MAX_OUT = 3                      # Step 15: 各遺伝子から標的の側へ描く辺の最大数（E が大きい順）
DECAY = 0.5                      # Step 16: 間に1つ遺伝子が入るごとに、影響にかける係数（直接の辺 = 1、間に1つ = 0.5、2つ = 0.25 ...）
N_PER_HOP = 20                   # Step 17: 途中のタンパク質を、hop（1〜5）ごとに何個ずつ描くか
RESTRICT_RECEPTOR = "KDR"        # Step 19: 受容体をこの1つに限る
MAX_STEPS_KDR = 5                # Step 19: その受容体から AKT1 までの最大ステップ数
TOP_CODEP = 30                   # Step 20: 図に出す、相関が高い遺伝子の数
N_TF = 15                        # Step 21 (下流): 図に描く転写因子（TF）の数（影響の大きい順）
N_HUBS = 10                      # Step 22: 描くハブ遺伝子の数（その先の TF の数が多い順）
TF_PER_HUB = 6                   # Step 22: ハブごとに描く TF の数（経路スコアの大きい順）
N_DIRECT = 8                     # Step 22: AKT1 の直接の標的の TF を描く数（経路スコアの大きい順）
FOCUS_TFS = ["HIF1A", "FOXO1", "SMAD2", "SMAD3"]   # Step 23: 下流で注目する転写因子（AKT1 からこの TF までの経路だけを見る）
MAX_STEPS_TF = 4                 # Step 23: AKT1 からその TF までの最大ステップ数
CORR_MIN = 0.1                   # Step 24: DepMap の相関のしきい値（|r| がこれ以上。FDR < 0.05、細胞株 1,000 以上も条件）
N_MID_SIDE = 40                  # Step 25: 上流・下流それぞれで、流れの大きい順に描く途中のタンパク質の数

## 0b. 設定ファイルの読み込み
- 動作: YAML を読み込み、データベースのパスを辞書 `DB` にまとめる（`~` は展開する）
- 出力: 各データベースのパスと、ファイルが存在するかどうかの表

In [ ]:
# YAML を読み込み、パスを展開して一覧にする
import os
import re
import yaml
import numpy as np
import pandas as pd

cfg = yaml.safe_load(open(CONFIG))
dbs = cfg["databases"]

# 各データベースの path（または dir）を取り出す。設定がなければ None
DB = {}
for name, d in dbs.items():
    p = d.get("path") or d.get("dir") if isinstance(d, dict) else None
    DB[name] = os.path.expanduser(p) if p else None

# 存在確認の表
table = pd.DataFrame({"path": DB})
table["exists"] = [(p is not None) and os.path.exists(p) for p in DB.values()]
table

## 1a. データベースの読み込み
> 指示: step 1: network development — read signor and reactome and the other database and show the structure

- 動作: SIGNOR、Reactome FI、OmniPath（キャッシュ）、UniProt、Lambert TF 一覧、DoRothEA、HPA、DepMap を読み込み、辞書 `raw` に入れる。DepMap は巨大（数百 MB）なので先頭5行だけ読む
- 出力: なし（次のセルで構造を確認する）

**Note**
- STRING は、以前の指示（STRING は解析から除外）に従い読み込まない。
- OmniPath は API から取得してキャッシュしたファイルを使う。これはライセンスフィルタ**前**の生データ（約14万行）。

In [ ]:
# 各データベースを読み込む（DepMap は先頭5行だけ）
import glob

raw = {}   # 名前 -> DataFrame
raw["signor"] = pd.read_csv(DB["signor"], sep="\t", dtype=str, quoting=3, on_bad_lines="skip")   # SIGNOR 全データ（タブ区切り）
raw["reactome_fi"] = pd.read_csv(DB["reactome_fi"], sep="\t")                                    # Reactome の機能的相互作用
raw["uniprot"] = pd.read_csv(DB["uniprot"], sep="\t")                                            # UniProt（受容体・局在の注釈）
raw["tf_list"] = pd.read_csv(DB["tf_list"])                                                      # Lambert の転写因子一覧
raw["regulon"] = pd.read_csv(DB["regulon"], sep=None, engine="python")                           # DoRothEA（TF -> 標的遺伝子）
raw["hpa_tissue"] = pd.read_csv(DB["hpa_tissue"], sep="\t")                                      # HPA 組織別発現（縦持ち）

# OmniPath: API から取得してキャッシュした生ファイル（ライセンスフィルタ前）
cache = os.path.expanduser(cfg["cache_dir"])
omni_file = sorted(glob.glob(os.path.join(cache, "omnipath_signed_*.txt")))[0]
raw["omnipath"] = pd.read_csv(omni_file, sep="\t", dtype=str)

# DepMap: 3 ファイルとも先頭5行だけ（構造確認用）
for key, fname in [("depmap_effect", "CRISPRGeneEffect.csv"), ("depmap_mutations", "OmicsSomaticMutations.csv"),
                   ("depmap_expression", "OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv")]:
    raw[key] = pd.read_csv(os.path.join(DB["depmap"], fname), nrows=5)

## 1b. 構造の一覧
- 動作: 各データベースの行数・列数・ファイルの大きさ・列名（先頭8つ）を表にする
- 出力: 1行1データベースの表（DepMap は先頭5行だけ読んだので行数は参考値ではなく `5`）

In [ ]:
# 行数・列数・列名の一覧表
rows = []
for name, df in raw.items():
    rows.append(dict(database=name, rows=len(df), columns=df.shape[1], first_columns=", ".join(map(str, df.columns[:8]))))
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
pd.DataFrame(rows).set_index("database")

## 1c. 中身の例（1行ずつ）
- 動作: 各データベースの1行目を縦に並べて、列ごとの値の例を見る
- 出力: データベースごとに、列名と値の例（20列まで）

In [ ]:
# 各データベースの1行目を縦に表示する
for name, df in raw.items():
    print(f"=== {name}  ({df.shape[1]} columns) ===")
    print(df.iloc[0].head(20).to_string()[:1500], "\n")

## 1d. 全データベースの列の一覧（完全版）
> 指示: Please show me the structure of all database in the notebook

- 動作: 読み込んだ全データベースについて、**すべての列**の名前・型（dtype）・値が入っている行数・値の例・種類数を表にして表示する。列が非常に多い DepMap の effect / expression（約1.8万・1.9万列）は、先頭10列と最後の3列だけ表示し、列数を書く
- 出力: データベースごとの表（見出し付き）。全部で10個

**Note**
- DepMap の3ファイルは先頭5行だけ読んでいるので、「値が入っている行数」は最大5。
- STRING は指示により読まない。`receptors` / `ligands` / `compartments` / `custom_edges` は YAML で未設定のため、ファイルがない（UniProt から作る、または使わない）。

In [ ]:
# 全データベースの全列を表にして表示する
from IPython.display import display, Markdown

WIDE = 100   # 列数がこれを超える表は、先頭10列と最後の3列だけ表示する
for name, df in raw.items():
    info = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),                                 # 値が入っている行数
        "n_unique": df.nunique(),                                      # 値の種類数
        "example": df.apply(lambda s: str(s.dropna().iloc[0])[:50] if s.notna().any() else ""),   # 最初の非欠損値
    })
    note = ""
    if len(info) > WIDE:
        note = f"（列が多いので先頭10列と最後の3列のみ表示。全 {len(info):,} 列）"
        info = pd.concat([info.head(10), info.tail(3)])
    display(Markdown(f"**{name}** — {len(df):,} 行 × {df.shape[1]:,} 列 {note}"))
    display(info)

## 1e. 各データベースの列の説明
> 指示: please add the explanation of the columns of each database.

- 動作: 各データベースの列の意味を日本語で表にする。説明の過不足を自動で確認し、説明がない列があれば警告を出す。遺伝子名が列名になっている DepMap の effect / expression は、固定の列だけ説明し、遺伝子の列は1行にまとめる
- 出力: データベースごとの「列名 | 説明」の表（全10個）

**Note**
- 説明は、各データベースの公式の説明とファイル中身の例から私が日本語にしたもの。意味に自信がない列には「（要確認）」と付けた（DepMap の `PS`、`Rescue`、`RescueReason`）。
- 値の意味を実際のデータで確かめたもの: SIGNOR の `EFFECT` / `DIRECT`、Reactome の `Direction`、Lambert の `Is TF?`。

In [ ]:
# 各データベースの列の説明（日本語）。(列名, 説明) のリスト
doc = {}

doc["signor"] = [   # SIGNOR: 手作業で整理された「A が B にこう作用する」という因果関係
    ("ENTITYA", "作用する側（上流）の名前。遺伝子名のことが多い"),
    ("TYPEA", "ENTITYA の種類（protein / complex / proteinfamily / chemical / smallmolecule / mirna / stimulus / phenotype など）"),
    ("IDA", "ENTITYA の ID（タンパク質なら UniProt ID）"),
    ("DATABASEA", "IDA がどのデータベースの ID か（UNIPROT など）"),
    ("ENTITYB", "作用される側（下流）の名前"),
    ("TYPEB", "ENTITYB の種類"),
    ("IDB", "ENTITYB の ID"),
    ("DATABASEB", "IDB がどのデータベースの ID か"),
    ("EFFECT", "A の B への効果。up-regulates（促進）/ down-regulates（抑制）/ form complex（複合体形成）/ unknown。activity（活性）、quantity（量）などの細分あり。符号（+/-）はここから決める"),
    ("MECHANISM", "作用のしくみ（phosphorylation, binding, transcriptional regulation, ubiquitination, dephosphorylation など）"),
    ("RESIDUE", "修飾されるアミノ酸残基（例 Ser15）。リン酸化などのとき"),
    ("SEQUENCE", "その残基の周辺アミノ酸配列"),
    ("TAX_ID", "生物種（9606 = ヒト）。複数種が `;` で並ぶこともある"),
    ("CELL_DATA", "根拠が得られた細胞（BTO の ID）"),
    ("TISSUE_DATA", "根拠が得られた組織（BTO の ID）"),
    ("MODULATOR_COMPLEX", "A が複合体のとき、その SIGNOR 複合体 ID"),
    ("TARGET_COMPLEX", "B が複合体のとき、その SIGNOR 複合体 ID"),
    ("MODIFICATIONA", "A 自身の修飾状態（例 リン酸化されている）"),
    ("MODASEQ", "A の修飾部位周辺の配列"),
    ("MODIFICATIONB", "B 自身の修飾状態"),
    ("MODBSEQ", "B の修飾部位周辺の配列"),
    ("PMID", "根拠となる論文の PubMed ID"),
    ("DIRECT", "直接の作用か（YES）、経路をまとめた間接的な記述か（NO）。このツールでは NO を除く"),
    ("NOTES", "キュレーターのメモ"),
    ("ANNOTATOR", "登録した人（キュレーター）"),
    ("SENTENCE", "根拠となった論文の文（要約）"),
    ("SCORE", "根拠の強さを表すスコア（0〜1）。論文数・実験の種類などから SIGNOR が計算"),
    ("SIGNOR_ID", "SIGNOR 内の行の ID"),
]

doc["reactome_fi"] = [   # Reactome FI: 機能的相互作用（向きは弱い）
    ("Gene1", "遺伝子1"),
    ("Gene2", "遺伝子2"),
    ("Annotation", "関係の種類（complex = 同じ複合体、input = 入力、catalyze = 触媒、catalyzed by = 触媒される、predicted = 機械学習による予測）。`;` で複数"),
    ("Direction", "向き。`->` は 1→2、`<-` は 2→1、`<->` は双方向、`-` は向きなし。`-|` / `|-` は抑制を表す"),
    ("Score", "信頼度（0〜1）。1.0 は経路（pathway）データベースの注釈あり、それ未満は予測"),
]

doc["uniprot"] = [   # UniProt: タンパク質の注釈
    ("Entry", "UniProt のアクセッション（ID）"),
    ("Gene Names (primary)", "代表の遺伝子名（HGNC 記号）"),
    ("Keywords", "UniProt のキーワード（`;` 区切り）。Receptor、Membrane、Secreted、Cytokine などから受容体・リガンドを判定する"),
    ("Subcellular location [CC]", "細胞内の局在の文章。膜・核・細胞質などの判定に使う"),
]

doc["tf_list"] = [   # Lambert らの転写因子一覧（2018）
    ("Unnamed: 0", "行番号（不要な列）"),
    ("Ensembl ID", "Ensembl の遺伝子 ID"),
    ("HGNC symbol", "遺伝子名（HGNC 記号）"),
    ("DBD", "DNA 結合ドメインの種類（例 AP-2、C2H2 ZF）"),
    ("Is TF?", "転写因子と判定されたか（Yes / No）。Yes が 1,639 個で、下流の終点に使う"),
    ("TF assessment", "判定の根拠の種類（Known motif など）"),
    ("Binding mode", "DNA への結合様式（単量体 / ホモ多量体 など）"),
    ("Motif status", "結合モチーフの確かさ・取得方法"),
    ("Final Notes", "最終判定のメモ"),
    ("Final Comments", "最終判定のコメント"),
    ("Interpro ID(s)", "InterPro のドメイン ID"),
    ("EntrezGene ID", "NCBI Entrez の遺伝子 ID"),
    ("EntrezGene Description", "Entrez の遺伝子説明"),
    ("PDB ID", "立体構造（PDB）の ID"),
    ("TF tested by HT-SELEX?", "HT-SELEX 実験で調べられたか"),
    ("TF tested by PBM?", "PBM（タンパク質結合マイクロアレイ）で調べられたか"),
    ("Conditional Binding Requirements ", "結合に必要な条件（共因子など）。列名の末尾に空白あり"),
    ("Original Comments", "元のコメント"),
    ("Vaquerizas 2009 classification", "Vaquerizas ら (2009) の分類"),
    ("CisBP considers it a TF?", "CisBP が転写因子と見なすか"),
    ("TFCat classification", "TFCat の分類"),
    ("Is a GO TF?", "GO で転写因子とされているか"),
    ("Initial assessment ", "最初の判定。列名の末尾に空白あり"),
    ("Curator 1", "判定したキュレーター1"),
    ("Curator 2", "判定したキュレーター2"),
    ("TFclass considers it a TF? ", "TFclass が転写因子と見なすか。列名の末尾に空白あり"),
    ("Go Evidence", "GO の根拠"),
    ("Pfam Domains (By ENSP ID)", "Pfam ドメイン（タンパク質 ID ごと）"),
    ("Is C2H2 ZF(KRAB)?", "KRAB 型の C2H2 ジンクフィンガーか"),
]

omni_cols = [   # OmniPath（API からの相互作用表）と DoRothEA は同じ形
    ("source", "上流の UniProt ID"),
    ("target", "下流の UniProt ID"),
    ("source_genesymbol", "上流の遺伝子名"),
    ("target_genesymbol", "下流の遺伝子名"),
    ("is_directed", "向きがあるか（True / False）"),
    ("is_stimulation", "促進を示す記録があるか"),
    ("is_inhibition", "抑制を示す記録があるか。促進・抑制の両方が True なら符号が割れている"),
    ("consensus_direction", "複数のリソースの意見を合わせた「向き」"),
    ("consensus_stimulation", "合意としての促進"),
    ("consensus_inhibition", "合意としての抑制。符号の矛盾を解くのに使う"),
    ("sources", "この記録を載せているリソース名（`;` 区切り）。ライセンスの確認に使う"),
    ("references", "根拠の論文（リソース名:PubMed ID の `;` 区切り）。数が多いほど確かと見なす"),
]
doc["omnipath"] = omni_cols + [("type", "相互作用の種類（post_translational = 翻訳後の調節。ここではすべて同じ値）")]
doc["regulon"] = omni_cols + [("dorothea_level", "DoRothEA の信頼度（A が最も高い、B、C）。B;D のような複合表記が少数ある")]

doc["hpa_tissue"] = [   # HPA: 組織ごとの RNA 発現（縦持ち）
    ("Gene", "Ensembl の遺伝子 ID"),
    ("Gene name", "遺伝子名"),
    ("Tissue", "組織名（breast, liver など約50種）"),
    ("nTPM", "正規化した発現量（nTPM）。1 以上を「発現している」とみなしてフィルタに使う"),
]

doc["depmap_effect"] = [   # DepMap CRISPR: 細胞株×遺伝子の行列（横持ち）
    ("Unnamed: 0", "細胞株の ID（ACH-xxxxxx）。1行 = 1細胞株"),
    ("（遺伝子ごとの列）", "列名は `遺伝子名 (Entrez ID)`、例 `A1BG (1)`。値は CRISPR の遺伝子効果（Chronos）。0 付近は影響なし、−1 付近は必須遺伝子（ノックアウトで増殖が落ちる）。このツールの共必須性（co-essentiality）の計算に使う"),
]
doc["depmap_expression"] = [   # DepMap の RNA 発現（横持ち）
    ("Unnamed: 0", "行番号"),
    ("SequencingID", "シーケンスの ID"),
    ("ModelConditionID", "細胞株の培養条件の ID（MC-…）"),
    ("ModelID", "細胞株の ID（ACH-…）"),
    ("IsDefaultEntryForMC", "その培養条件の代表データか（Yes / No）"),
    ("IsDefaultEntryForModel", "その細胞株の代表データか（Yes / No）"),
    ("（遺伝子ごとの列）", "列名は `遺伝子名 (Entrez ID)`。値は log2(TPM+1) の発現量。合成致死の代替判定（発現が下位25%）に使う"),
]
doc["depmap_mutations"] = [   # DepMap の体細胞変異（1行 = 1変異）
    ("Unnamed: 0", "行番号"), ("SequencingID", "シーケンスの ID"), ("ModelID", "細胞株の ID（ACH-…）"),
    ("ModelConditionID", "培養条件の ID"), ("IsDefaultEntryForModel", "細胞株の代表データか"), ("IsDefaultEntryForMC", "培養条件の代表データか"),
    ("Chrom", "染色体"), ("Pos", "染色体上の位置"), ("Ref", "参照配列の塩基"), ("Alt", "変異後の塩基"),
    ("AF", "変異アレルの割合（アレル頻度）"), ("DP", "読み取りの深さ（リード数）"), ("RefCount", "参照アレルのリード数"), ("AltCount", "変異アレルのリード数"),
    ("GT", "遺伝子型（0/1 = ヘテロ、1/1 = ホモ）"), ("PS", "フェージング（変異同士が同じ染色体か）に関する値（要確認）"),
    ("VariantType", "変異の型（SNP、挿入、欠失など）"), ("VariantInfo", "変異の影響の種類（missense、frameshift、nonsense など）"),
    ("DNAChange", "DNA レベルの変化（HGVS 表記）"), ("ProteinChange", "アミノ酸の変化（例 p.G12D）"),
    ("HugoSymbol", "遺伝子名（HUGO 記号）。遺伝子ごとの絞り込みに使う"), ("Exon", "変異のあるエクソン番号"), ("Intron", "変異のあるイントロン番号"),
    ("EnsemblGeneID", "Ensembl の遺伝子 ID"), ("EnsemblFeatureID", "Ensembl の転写産物 ID"), ("HgncName", "HGNC の遺伝子名称"),
    ("HgncFamily", "HGNC の遺伝子ファミリー"), ("UniprotID", "UniProt ID"), ("DbsnpRsID", "dbSNP の rs 番号"), ("GcContent", "周辺の GC 含量"),
    ("VepImpact", "VEP による影響の大きさ（HIGH / MODERATE / LOW / MODIFIER）"), ("VepBiotype", "遺伝子の種類（protein_coding など）"),
    ("VepHgncID", "VEP が付けた HGNC ID"), ("VepExistingVariation", "既知の変異 ID（dbSNP、COSMIC など）"), ("VepManeSelect", "MANE Select 転写産物の ID"),
    ("VepENSP", "Ensembl のタンパク質 ID"), ("VepSwissprot", "SwissProt の ID"),
    ("Sift", "SIFT によるアミノ酸変化の影響予測"), ("Polyphen", "PolyPhen によるアミノ酸変化の影響予測"),
    ("GnomadeAF", "gnomAD（エクソーム）での集団アレル頻度"), ("GnomadgAF", "gnomAD（ゲノム）での集団アレル頻度"),
    ("VepClinSig", "ClinVar の臨床的意義"), ("VepSomatic", "体細胞変異かどうかの注釈"),
    ("VepPliGeneValue", "遺伝子の pLI（機能喪失への耐性）の値"), ("VepLofTool", "機能喪失（LoF）変異予測ツールの判定"),
    ("OncogeneHighImpact", "がん遺伝子での影響大の変異か"), ("TumorSuppressorHighImpact", "がん抑制遺伝子での影響大の変異か"),
    ("CivicID", "CIViC データベースの ID"), ("CivicDescription", "CIViC の説明"), ("CivicScore", "CIViC のスコア"),
    ("LikelyLoF", "機能喪失（LoF）の可能性が高い変異か。このツールの合成致死で「遺伝子を失った細胞株」の判定に使う"),
    ("AMClass", "AlphaMissense の分類（likely_pathogenic など）"), ("AMPathogenicity", "AlphaMissense の病原性スコア"),
    ("Rescue", "DepMap の変異フィルタで救済（残された）変異か（要確認）"), ("RescueReason", "救済の理由（要確認）"),
    ("Hotspot", "がんで繰り返し見つかるホットスポット変異か"), ("EntrezGeneID", "Entrez の遺伝子 ID"),
]

# 各データベースを表にして表示。実際の列と説明の過不足も確認する
for name, items in doc.items():
    tab = pd.DataFrame(items, columns=["column", "説明"]).set_index("column")
    cols = set(map(str, raw[name].columns)) - {c for c in raw[name].columns if " (" in str(c) and str(c).endswith(")")}
    missing = sorted(cols - set(tab.index))                       # 説明がない実際の列
    display(Markdown(f"**{name}** — 説明あり {len(tab)} 行" + (f" ／ ⚠ 説明がない列: {missing}" if missing else " ／ 全列に説明あり")))
    display(tab.style.set_properties(subset=["説明"], **{"text-align": "left", "white-space": "normal"}))


### Step 1 のメモ（構造を見て気づいた点）
- **SIGNOR**: 43,315 行・28 列。`ENTITYA/B`（遺伝子名）、`TYPEA/B`（protein / complex / family など）、`EFFECT`（up/down-regulates）、`MECHANISM`、`RESIDUE`、`DIRECT`（直接か間接か）、`SCORE`（根拠スコア、平均 0.51）を使う。
- **Reactome FI**: 269,789 行・5 列（`Gene1, Gene2, Annotation, Direction, Score`）。向き（`Direction`）は記号（`<-` など）で、符号（活性化/抑制）は `Annotation` の文章から読む必要がある。
- **OmniPath / DoRothEA**: 同じ13列の形（UniProt ID と遺伝子名、`is_stimulation`、`is_inhibition`、`sources`、`references`）。DoRothEA 側には `dorothea_level`（A/B/C。`B;D` のような複合表記が17行ある）がある。
- **UniProt**: 4 列だけ（`Entry, Gene Names (primary), Keywords, Subcellular location [CC]`）。受容体・局在はここからキーワードで判定する。
- **Lambert TF 一覧**: 先頭に不要な列 `Unnamed: 0`（行番号）がある。`Is TF?` が Yes の行が転写因子。
- **HPA**: 縦持ち（遺伝子×組織の1行1値、102万行）。**DepMap**: 横持ち（列が遺伝子、列名は `A1BG (1)` のように遺伝子名＋Entrez ID）なので、遺伝子名の取り出しが必要。
- **Note（元の方針とのずれ）**: 今のところ、指示を実現できない点はない。ただし STRING は指示により読んでいない。

## Step 2. ネットワーク構築に使う列の選択
> 指示: step2 — select columns of all database for development of PPI network for the analysis. Is it single PPI or separate ones?

### 先に答え: 1つのネットワークではなく、役割別に分ける

| 層 | データベース | 中身 | 使い方 |
|---|---|---|---|
| **A. 主ネットワーク（1つ）** | SIGNOR + OmniPath | 向きと符号（促進/抑制）のあるタンパク質間の作用 | 2つを**1つの有向・符号付きグラフ**に統合し、ここで経路を探索する |
| **B. 文脈層（別）** | Reactome FI | 向きも符号も弱い機能的関係 | 探索には使わず、「同じ複合体か」「物理的に近いか」の注釈とシード救済だけ |
| **C. 転写層（別）** | Lambert TF 一覧 + DoRothEA | 転写因子 → 標的遺伝子 | 主ネットワークの終点（TF）から先の層。別表のまま接続する |
| **D. ノード注釈** | UniProt、HPA | 受容体・リガンド・局在、組織での発現 | ネットワークの辺ではなく、ノードの属性とフィルタ |
| **E. 遺伝学（別）** | DepMap 3 ファイル | 細胞株ごとの必須性・変異・発現 | ネットワークではなく、共必須性と合成致死の計算 |

**分ける理由**: (1) 辺の性質が違う。SIGNOR / OmniPath は向きと符号があり、Reactome は向きも符号も弱い。混ぜると符号の計算が壊れる。(2) 転写因子 → 遺伝子の辺（約3万）を主ネットワークに混ぜると、ハブが増えて探索があふれる。(3) 辺ごとの出典・ライセンスを別々に管理できる。(4) 各層を別々に点検でき、あとから入れ替えやすい。

**Note（元の方針との違い）**: 「PPI」という名前は物理的な結合（タンパク質どうしがくっつく）を連想させるが、層 A は**シグナル伝達の因果関係**（リン酸化、抑制など）で、物理的な結合そのものではない。物理的な結合は層 B（Reactome）と STRING だが、STRING は指示により使わない。したがって、このネットワークは「PPI」ではなく「シグナル伝達ネットワーク」と呼ぶのが正確。

## 2a. 選ぶ列の計画
- 動作: データベースごとに、役割（どの層か）・使う列・使う理由・捨てる列の主な理由を表にする
- 出力: 1行1データベースの計画表

In [ ]:
# データベースごとの役割と使う列（計画）
plan = {
    "signor": dict(layer="A 主ネットワーク",
        use=["ENTITYA", "TYPEA", "IDA", "ENTITYB", "TYPEB", "IDB", "EFFECT", "MECHANISM", "RESIDUE", "TAX_ID", "DIRECT", "SCORE", "PMID"],
        why="上流・下流・種類・符号(EFFECT)・仕組み・残基・ヒト判定(TAX_ID)・直接性(DIRECT)・辺の信頼度(SCORE)・論文(PMID)。文章やメモの列は不要"),
    "omnipath": dict(layer="A 主ネットワーク",
        use=["source", "target", "source_genesymbol", "target_genesymbol", "is_directed", "is_stimulation", "is_inhibition",
             "consensus_stimulation", "consensus_inhibition", "sources", "references"],
        why="上流・下流(ID と遺伝子名)・向き・符号・合意符号・出典(ライセンス確認)・論文数(信頼度)。type は全行同じ値なので不要"),
    "reactome_fi": dict(layer="B 文脈層",
        use=["Gene1", "Gene2", "Annotation", "Direction", "Score"],
        why="5列すべて。向きと関係の種類を注釈に使う（符号は使えない）"),
    "regulon": dict(layer="C 転写層",
        use=["source_genesymbol", "target_genesymbol", "is_stimulation", "is_inhibition", "consensus_stimulation", "consensus_inhibition", "sources", "dorothea_level"],
        why="TF・標的・促進/抑制(mode of regulation)・出典・信頼度(A-C)"),
    "tf_list": dict(layer="C 転写層",
        use=["HGNC symbol", "Is TF?"],
        why="転写因子の判定に必要なのは遺伝子名と Is TF? だけ。残り27列は根拠のメモ"),
    "uniprot": dict(layer="D ノード注釈",
        use=["Entry", "Gene Names (primary)", "Keywords", "Subcellular location [CC]"],
        why="4列すべて。受容体・リガンド・局在の判定"),
    "hpa_tissue": dict(layer="D ノード注釈",
        use=["Gene name", "Tissue", "nTPM"],
        why="遺伝子名・組織・発現量。Ensembl ID は使わない"),
    "depmap_effect": dict(layer="E 遺伝学",
        use=["Unnamed: 0"],
        why="細胞株 ID の列に加えて、遺伝子の列はすべて使う（列名から遺伝子名を取り出す）。共必須性の計算用"),
    "depmap_mutations": dict(layer="E 遺伝学",
        use=["ModelID", "HugoSymbol", "LikelyLoF", "VariantInfo"],
        why="細胞株・遺伝子・機能喪失の判定・変異の種類。合成致死で「遺伝子を失った細胞株」を見つけるのに使う"),
    "depmap_expression": dict(layer="E 遺伝学",
        use=["ModelID"],
        why="細胞株 ID に加えて、遺伝子の列はすべて使う。変異が足りないときの代替判定（発現が下位25%）用"),
}
table = pd.DataFrame({k: dict(層=v["layer"], 列数=len(v["use"]), 使う列=", ".join(v["use"]), 理由=v["why"]) for k, v in plan.items()}).T
pd.set_option("display.max_colwidth", 200)
table

## 2b. 選んだ列だけを取り出す
- 動作: 計画どおりに列を絞った表を `sel` に作り、計画の列が実際のファイルにすべてあるか確認する。DepMap の遺伝子の列は、遺伝子名（括弧の前）に直して数える
- 出力: データベースごとの行数・列数と、計画にあるのにファイルにない列（なければ空）

In [ ]:
# 計画した列だけを取り出す（列が足りなければ警告）
sel = {}
rows = []
for name, v in plan.items():
    df = raw[name]
    absent = [c for c in v["use"] if c not in df.columns]            # 計画にあるのにファイルにない列
    gene_cols = [c for c in df.columns if name.startswith("depmap") and " (" in str(c) and str(c).endswith(")")]   # DepMap の「遺伝子名 (Entrez ID)」の列だけ
    cols = [c for c in v["use"] if c in df.columns] + gene_cols
    sel[name] = df[cols]
    rows.append(dict(database=name, 層=v["layer"], 行数=len(df), 選んだ列数=len(cols), 元の列数=df.shape[1],
                     遺伝子の列=len(gene_cols), ファイルにない列=", ".join(absent)))
pd.DataFrame(rows).set_index("database")

## Step 3. 主シグナル伝達ネットワーク（ライセンスが必要なデータと曖昧なデータを除く）
> 指示: please exclude all data required license of commercial use. Please note which databases are licence required, which ones are not and which are ambiguous. You should exclude ambiguous. Please compare before and after the exclusion by showing the number of data

### 方針（厳しい規則）
1. すべてのデータベースとリソースを **要ライセンス / 不要 / 曖昧** の3つに分ける（3a, 3b）。
2. **曖昧は除く**。ネットワークには「不要」と判定されたリソースだけを使う。
3. OmniPath の記録は、載せているリソースが**1つでも**「要ライセンス」か「曖昧」なら、その記録ごと捨てる（前回は商用可のリソースだけを残していたが、符号が学術のみのリソースの影響を受けるので、今回は厳しくする）。
4. 除く前と後で、記録・辺・ノードの数を比べる（3d）。

### 判定の基準
| 区分 | 基準 | 例 |
|---|---|---|
| **要ライセンス** | OmniPath が `academic` / `non_profit` と判定（非営利のみ） | PhosphoSite, HPRD, Wang, KEA, iPTMnet, Cellinker, HINT |
| **不要** | 商用可で、標準的な許諾（CC BY、CC0、MIT、BSD）。CC BY-SA は商用可だが「同じ条件で共有」が付くので**不要（共有条件あり）**として使う | SIGNOR, IntAct, BioGRID, Reactome 系、SignaLink3 |
| **曖昧** | 区分が `composite` / 不明、または許諾の名前が標準的でない（GPL、LGPL、NAR Open Access、DSL、CC BY-ND、Artistic など。データへの適用や改変の可否がはっきりしない） | CellTalkDB, LRdb, talklr, CellChatDB, dbPTM, InnateDB, DIP |

**Note**
- 判定の基準は私の判断。コードの `CLEAR` / `SHARE_ALIKE` の2つの集合を変えれば、基準を変えて再計算できる。
- ローカルファイル（SIGNOR、Reactome FI など）の許諾はファイルの中に書かれていないので、プロジェクトのメモ（CLAUDE.md）と私の知識による。確認できないものは「曖昧」にした。

## 3a. データベースごとのライセンス
- 動作: 使っている8つのデータベースについて、ライセンスの名前と3区分を表にする（根拠も書く）
- 出力: データベース別の表。OmniPath の中のリソース別判定は 3b

In [ ]:
# データベース単位のライセンス判定（根拠はファイルの外の情報なので、根拠の列に書く）
db_lic = pd.DataFrame([
    ("SIGNOR",      "CC BY 4.0",           "不要",   "SIGNOR の公開条件（OmniPath のリソース一覧にも CC BY 4.0 と記載）"),
    ("OmniPath",    "リソースごとに異なる", "曖昧",   "約100の元データベースの集合。リソース単位で判定する（3b）"),
    ("Reactome FI", "CC0 1.0",             "不要",   "プロジェクトのメモ（CLAUDE.md）による。ファイルには記載なし"),
    ("UniProt",     "CC BY 4.0",           "不要",   "UniProt の公開条件"),
    ("Lambert TF 一覧", "記載なし",         "曖昧",   "ファイルに許諾の記載がない（メモには CC-BY とあるが未確認）。→ 除く対象"),
    ("DoRothEA",    "構成リソースごとに異なる", "曖昧", "複数のデータベースの集約（メモにも商用再配布の前に確認とある）。→ 構成リソースごとに判定が必要"),
    ("HPA",         "CC BY-SA 3.0",        "不要（共有条件あり）", "商用可。ただし派生物は同じ条件で共有。フィルタとしてのみ使う"),
    ("DepMap",      "CC BY 4.0",           "不要",   "DepMap の公開条件"),
], columns=["database", "ライセンス", "区分", "根拠"]).set_index("database")
db_lic

## 3b. OmniPath のリソース別ライセンス判定
- 動作: OmniPath のリソース一覧から、各リソースのライセンスの名前と区分（`purpose`）を読み、3区分に分ける。リソースごとの記録数と、区分別の合計を出す
- 出力: リソース別の表（記録数の多い順に40件）と、区分別のリソース数・記録数

In [ ]:
# リソースごとに「要ライセンス / 不要 / 曖昧」を判定する
import json
from collections import Counter

res = json.load(open(os.path.join(cache, "omnipath_resources.txt")))      # OmniPath が公開しているリソース一覧（ライセンス付き）

CLEAR = {"CC BY 4.0", "CC BY 3.0", "CC BY 2.5", "CC0 1.0", "MIT", "BSD"}   # 標準的で商用可の許諾 -> 不要
SHARE_ALIKE = {"CC BY-SA 3.0", "CC BY-SA 4.0"}                              # 商用可だが同条件で共有 -> 不要（共有条件あり）

def licence_of(s):
    # リソース名がそのまま一覧にあればそれ、なければ "_" の前の名前
    d = res.get(s) or res.get(s.split("_")[0]) or {}
    return d.get("license") or {}

def classify(s):
    l = licence_of(s); purpose = l.get("purpose"); name = l.get("name")
    if purpose in ("academic", "non_profit", "nonprofit"): return "要ライセンス"
    if purpose == "commercial" and name in CLEAR: return "不要"
    if purpose == "commercial" and name in SHARE_ALIKE: return "不要（共有条件あり）"
    return "曖昧"                                                           # composite・不明・標準的でない許諾

count = Counter(s for x in sel["omnipath"].sources.dropna() for s in x.split(";") if s)     # リソースごとの記録数
lic = pd.DataFrame([dict(resource=s, records=n, ライセンス名=licence_of(s).get("name"), purpose=licence_of(s).get("purpose"),
                         区分=classify(s)) for s, n in count.items()]).set_index("resource").sort_values("records", ascending=False)
lic["使う"] = lic.区分.isin(["不要", "不要（共有条件あり）"])
summary = lic.groupby("区分").agg(リソース数=("records", "size"), 記録数の延べ=("records", "sum"))
display(summary)
lic.head(40)

## 3c. OmniPath の記録の除外（記録単位）
- 動作: 各記録が載せているリソースを調べ、(1) 「要ライセンス」が1つでもある、(2) 「曖昧」が1つでもある、(3) 全部「不要」、の3つに分ける。(3) だけを残す
- 出力: 記録の数（除く前 → 除いた後）とその内訳

In [ ]:
# 記録ごとに、載せているリソースの区分を調べる
omni_all = sel["omnipath"].copy()
src_list = omni_all.sources.fillna("").map(lambda x: [s for s in x.split(";") if s])
cls = lic.区分.to_dict()
has_req = src_list.map(lambda l: any(cls[s] == "要ライセンス" for s in l))      # 要ライセンスのリソースを1つでも含む
has_amb = src_list.map(lambda l: any(cls[s] == "曖昧" for s in l))               # 曖昧なリソースを1つでも含む
omni_all["sources_ok"] = src_list                                                 # 残すとき、全部「不要」なので全リソースが出典

keep_mask = ~has_req & ~has_amb
omni_strict = omni_all[keep_mask]
pd.Series({
    "元の記録（除く前）": len(omni_all),
    "要ライセンスのリソースを含む（除く）": int(has_req.sum()),
    "曖昧なリソースだけが原因で除く（要ライセンスは含まない）": int((~has_req & has_amb).sum()),
    "全部「不要」のリソース（残す）": len(omni_strict),
}, name="記録数").to_frame()

## 3d. 除く前と後のネットワークを作る
- 動作: OmniPath の記録を「除く前（全部）」と「除いた後（3c で残したもの）」の2通りで、(1) 向きと符号がある記録だけにし、(2) 辺の表にし（複合体は構成タンパク質に展開）、(3) SIGNOR と統合する。SIGNOR は CC BY 4.0 なので両方で同じ。統合は各データベース内の百分位 + noisy-OR
- 出力: 辺の数・ノード数など（次のセルで比較する）

In [ ]:
# 除く前と後で、同じ手順で主ネットワークを作る
# --- SIGNOR（両方で共通）---
sg = sel["signor"]
sg = sg[(sg.TAX_ID.astype(str).str.strip() == "9606") & (sg.TYPEA.str.lower() == "protein") & (sg.TYPEB.str.lower() == "protein") & (sg.DIRECT.str.upper() == "YES")]
eff = sg.EFFECT.str.lower().map(lambda e: 1 if "up-regulates" in e else -1 if "down-regulates" in e else 0)
edges_sg = pd.DataFrame({"src": sg.ENTITYA.str.upper().str.strip(), "dst": sg.ENTITYB.str.upper().str.strip(), "effect": eff,
                         "mechanism": sg.MECHANISM.fillna(""), "residue": sg.RESIDUE.fillna(""), "source": "SIGNOR",
                         "score": pd.to_numeric(sg.SCORE, errors="coerce")})
edges_sg = edges_sg[(edges_sg.src != edges_sg.dst) & (edges_sg.src != "") & (edges_sg.dst != "")]
edges_sg = edges_sg.sort_values(["effect", "score"], key=lambda s: (s == 0) if s.name == "effect" else -s.fillna(0)).drop_duplicates(["src", "dst"])

members = lambda x: [m for m in x.replace("COMPLEX:", "").split("_") if m]       # COMPLEX:A_B -> [A, B]
def merge_group(g):
    e = g.effect[g.effect != 0]
    return pd.Series(dict(effect=0 if len(set(e)) != 1 else int(e.iloc[0]), mechanism=";".join(sorted({m for m in g.mechanism if m})),
                          residue=";".join(sorted({r for r in g.residue if r})), source="+".join(sorted(set(g.source)))))

nets, counts, omni_edges = {}, {}, {}
for label, om in [("除く前", omni_all), ("除いた後", omni_strict)]:
    om = om[om.is_directed.isin(["True", "true", "1"])].copy()
    T = lambda col: om[col].isin(["True", "true", "1"])
    stim, inh, cstim, cinh = T("is_stimulation"), T("is_inhibition"), T("consensus_stimulation"), T("consensus_inhibition")
    om["effect"] = 0
    om.loc[(stim & ~inh) | (cstim & ~cinh), "effect"] = 1
    om.loc[(inh & ~stim) | (cinh & ~cstim), "effect"] = -1
    om = om[stim | inh]                                                           # 符号の情報がある記録だけ
    e = pd.DataFrame({"src": om.source_genesymbol.str.upper(), "dst": om.target_genesymbol.str.upper(), "effect": om.effect,
                      "mechanism": "omnipath:" + om.sources_ok.map(lambda l: "|".join(sorted(set(l)))), "residue": "", "source": "OmniPath",
                      "score": om.references.fillna("").map(lambda r: len([x for x in r.split(";") if x]))})   # 元のスコア = 論文の数
    is_cx = e.src.str.contains("_") | e.dst.str.contains("_")                     # 複合体を構成タンパク質に展開
    rows = [dict(src=a, dst=b, effect=r.effect, mechanism=r.mechanism + ";complex", residue="", source="OmniPath", score=r.score)
            for r in e[is_cx].itertuples() for a in members(r.src) for b in members(r.dst)]
    e = pd.concat([e[~is_cx], pd.DataFrame(rows)], ignore_index=True)
    e = e[(e.src != e.dst) & (e.src != "") & (e.dst != "")]
    e = e.sort_values(["effect", "score"], key=lambda s: (s == 0) if s.name == "effect" else -s).drop_duplicates(["src", "dst"])

    omni_edges[label] = e.copy()                                                  # 統合前の OmniPath の辺（score = 論文の数）を保存（Step 8 で使う）
    a, b = edges_sg.copy(), e.copy()
    for d in (a, b): d["nscore"] = d.score.rank(pct=True).fillna(0.5)             # データベース内の順位 -> 0〜1
    allr = pd.concat([a, b], ignore_index=True)
    net = allr.groupby(["src", "dst"], sort=False).apply(merge_group, include_groups=False)
    per = allr.groupby(["src", "dst", "source"], sort=False).nscore.max().reset_index()
    per["l"] = np.log1p(-per.nscore.clip(upper=0.999))
    net["weight"] = (1 - np.exp(per.groupby(["src", "dst"], sort=False).l.sum())).round(3)   # noisy-OR
    nets[label] = net.reset_index()
    counts[label] = dict(OmniPath記録=len(om), OmniPath辺=len(e), SIGNOR辺=len(edges_sg))
    print(label, ": 辺", len(nets[label]), " ノード", len(set(nets[label].src) | set(nets[label].dst)))

## 3e. 除く前と後の比較
- 動作: 除く前と後の数を並べ、差と減った割合を出す。出典別・符号別の辺の数も並べる
- 出力: 比較表

In [ ]:
# 除く前と後の数を比べる
cmp = {}
for label, net in nets.items():
    d = dict(counts[label])
    d.update({"統合後の辺": len(net), "ノード": len(set(net.src) | set(net.dst)),
              "辺: SIGNOR のみ": int((net.source == "SIGNOR").sum()), "辺: OmniPath のみ": int((net.source == "OmniPath").sum()),
              "辺: 両方": int((net.source == "OmniPath+SIGNOR").sum()),
              "符号 +1 促進": int((net.effect == 1).sum()), "符号 -1 抑制": int((net.effect == -1).sum()), "符号 0 不明": int((net.effect == 0).sum())})
    cmp[label] = d
cmp = pd.DataFrame(cmp)
cmp["差"] = cmp["除いた後"] - cmp["除く前"]
cmp["減った割合 %"] = (100 * cmp["差"] / cmp["除く前"]).round(1)
cmp

## 3f. 除いた後の主シグナル伝達ネットワーク（表）
- 動作: 除いた後のネットワーク（`net`）を重みの大きい順に並べ、先頭20行を見る。`net` はこの後の段階で使う
- 出力: 辺の表（上流・下流・符号・仕組み・残基・出典・重み）

In [ ]:
# 除いた後のネットワークを以降の段階で使う
net = nets["除いた後"]
print("辺:", len(net), " ノード:", len(set(net.src) | set(net.dst)))
net.sort_values("weight", ascending=False).head(20)

## Step 4. リガンド → 転写因子（TF）の経路だけに剪定する
> 指示: Prune to ligand（リガンドから TF への経路に剪定）

### 方針
- **始点 = リガンド、終点 = TF**。リガンドから出発して TF に届く経路の上にあるノードと辺だけを残し、それ以外を捨てる。
- リガンドは**出発点だけ**（リガンドへ入る辺は使わない）、TF は**終点だけ**（TF から出る辺は使わない）。
- 経路の長さの上限は `MAX_HOPS`。辺 u→v は「リガンドから u までの最短距離 + 1 + v から TF までの最短距離 ≦ MAX_HOPS」のとき残す（そのような経路が1本はある、という意味）。
- 辺の符号・重みは変えない。辺を捨てるだけ。

**Note**
- リガンド・受容体は UniProt（CC BY 4.0）のキーワードから、TF は Lambert の一覧から決める。3a の表のとおり **Lambert の一覧はライセンスが「曖昧」**なので、厳密に守るなら TF の定義を変える必要がある（例えば UniProt の転写調節のキーワード）。まず Lambert で進め、あとで差し替えられるようにしてある。

## 4a. 始点（リガンド）と終点（TF）の集合
- 動作: UniProt から、細胞膜の受容体、分泌される増殖因子・サイトカイン・ホルモン（リガンド）を取り出し、Lambert の一覧から TF を取り出す。ネットワークに含まれる数も数える
- 出力: 集合ごとの数（全体 / ネットワーク内）

In [ ]:
# リガンド・受容体・TF の集合を作る
MAX_HOPS = 6                                                         # リガンドから TF までの経路の長さの上限（辺の数）

u = raw["uniprot"].dropna(subset=["Gene Names (primary)"])
kw = u.Keywords.fillna(""); loc = u["Subcellular location [CC]"].fillna("")
gene = u["Gene Names (primary)"].str.upper()
receptors = set(gene[kw.str.contains("Receptor") & loc.str.contains("Cell membrane")])                    # 細胞膜の受容体
ligands = set(gene[loc.str.contains("Secreted") & kw.str.contains("Cytokine|Growth factor|Hormone")]) - receptors   # 分泌される因子（受容体は除く）
tfs = set(raw["tf_list"].loc[raw["tf_list"]["Is TF?"] == "Yes", "HGNC symbol"].str.upper())              # 転写因子
ligands -= tfs                                                       # TF でもあるものはリガンドから外す

in_net = set(net.src) | set(net.dst)
pd.DataFrame({"全体": {"リガンド": len(ligands), "受容体": len(receptors), "TF": len(tfs)},
              "ネットワーク内": {"リガンド": len(ligands & in_net), "受容体": len(receptors & in_net), "TF": len(tfs & in_net)}})

## 4b. 剪定
- 動作: (1) リガンドへ入る辺とTFから出る辺を除く。(2) リガンドから各ノードまでの最短距離 `dL` と、各ノードから TF までの最短距離 `dT` を求める。(3) `dL(u) + 1 + dT(v) ≦ MAX_HOPS` を満たす辺だけを残す
- 出力: 剪定したネットワーク `net_lt` の辺数・ノード数

In [ ]:
# リガンド -> TF の経路の上にある辺だけを残す
import networkx as nx

use = net[~net.dst.isin(ligands) & ~net.src.isin(tfs)]               # リガンドは出発点だけ、TF は終点だけ
G = nx.DiGraph(); G.add_edges_from(zip(use.src, use.dst))
L = [g for g in ligands if g in G]; T = [g for g in tfs if g in G]

dL = nx.multi_source_dijkstra_path_length(G, L, cutoff=MAX_HOPS)                     # リガンドからの最短距離（辺の数）
dT = nx.multi_source_dijkstra_path_length(G.reverse(copy=False), T, cutoff=MAX_HOPS)  # TF までの最短距離（辺の向きを逆にして求める）

keep = [(a in dL) and (b in dT) and dL[a] + 1 + dT[b] <= MAX_HOPS for a, b in zip(use.src, use.dst)]
net_lt = use[keep].reset_index(drop=True)
print("剪定後: 辺", len(net_lt), " ノード", len(set(net_lt.src) | set(net_lt.dst)))

## 4c. 剪定の前と後の比較
- 動作: 剪定の前（ステップ3の `net`）と後（`net_lt`）で、辺・ノード、リガンド・受容体・TF・その他の数を並べる。さらに、リガンドが TF に届くまでの最短の辺の数の分布を見る
- 出力: 比較表と、最短経路の長さの分布

In [ ]:
# 剪定の前と後を比べる
def summary(n):
    nodes = set(n.src) | set(n.dst)
    return {"辺": len(n), "ノード": len(nodes), "リガンド": len(nodes & ligands), "受容体": len(nodes & receptors),
            "TF": len(nodes & tfs), "その他（リガンド・受容体・TF 以外）": len(nodes - ligands - receptors - tfs)}

cmp = pd.DataFrame({"剪定前": summary(net), "剪定後": summary(net_lt)})
cmp["減った割合 %"] = (100 * (cmp["剪定後"] - cmp["剪定前"]) / cmp["剪定前"]).round(1)
display(cmp)

# 各リガンドから、最も近い TF までの辺の数（剪定後のネットワークで）
G2 = nx.DiGraph(); G2.add_edges_from(zip(net_lt.src, net_lt.dst))
tf_in = [t for t in tfs if t in G2]
d2 = nx.multi_source_dijkstra_path_length(G2.reverse(copy=False), tf_in)
dist = pd.Series({g: d2[g] for g in ligands if g in d2}).value_counts().sort_index()
dist.rename_axis("最短の辺の数").to_frame("リガンドの数")

## 4d. 剪定したネットワークの表
- 動作: 剪定後のネットワーク `net_lt` を、重みの大きい順に並べて先頭20行を見る。上流・下流が「リガンド / 受容体 / TF / その他」のどれかも付ける
- 出力: 辺の表。`net_lt` は次の段階で使う

In [ ]:
# 剪定したネットワークの表（役割つき）
role = lambda g: "リガンド" if g in ligands else "TF" if g in tfs else "受容体" if g in receptors else "その他"
view = net_lt.copy()
view.insert(1, "src_role", view.src.map(role)); view.insert(3, "dst_role", view.dst.map(role))
view.sort_values("weight", ascending=False).head(20)

## Step 4. リガンド → 転写因子（TF）の経路だけに枝刈りする
> 指示: Prune to paths（最初に作るシグナル伝達ネットワークは、リガンドから TF まで。DoRothEA のような転写の層は使わない）

### 方針
1. **ノードの役割を決める**: リガンド（UniProt で分泌・サイトカイン・増殖因子・ホルモン）、受容体（UniProt で受容体かつ細胞膜）、TF（Lambert 一覧で `Is TF? = Yes`）。
2. **層の向きを強制する**: リガンドは出発点だけ（リガンドへ入る辺は捨てる）、リガンドから出る辺は**受容体へ向かうものだけ**、TF は終点だけ（TF から出る辺は捨てる＝TF の先へは進まない）。
3. **経路上の辺だけを残す**: リガンドから辺 `u→v` の手前の `u` までの最短ステップ数 + 1 + `v` から最も近い TF までの最短ステップ数 が `MAX_HOPS` 以下の辺だけを残す。

**Note**
- **TF の一覧のライセンスは「曖昧」**（Step 3 の 3a）。この段階では TF の定義にどうしても必要なので使うが、ネットワークの辺そのものはライセンス判定を通ったデータだけ。TF の一覧を別のものに替えたいときは知らせてほしい。
- 辺が経路上にあるかは「最短ステップで測った歩行」で判定するので、同じノードを通り直す歩行も数える（実際の経路よりやや多めに残る）。

## 4a. ノードの役割（リガンド・受容体・TF）
- 動作: UniProt と Lambert 一覧から、リガンド・受容体・TF の集合を作り、ネットワークに何個含まれるか数える
- 出力: 役割ごとの、一覧の個数・ネットワークにある個数（除く前と除いた後のネットワーク両方）

In [ ]:
# リガンド・受容体・TF の集合を作る
u = raw["uniprot"].dropna(subset=["Gene Names (primary)"])
kw = u.Keywords.fillna(""); loc = u["Subcellular location [CC]"].fillna("")
gene = u["Gene Names (primary)"].str.upper()
receptors = set(gene[kw.str.contains("Receptor") & loc.str.contains("Cell membrane")])               # 受容体: キーワード Receptor かつ細胞膜
ligands = set(gene[loc.str.contains("Secreted") & kw.str.contains("Cytokine|Growth factor|Hormone")]) - receptors   # リガンド: 分泌タンパク質
tf = raw["tf_list"]
tfs = set(tf.loc[tf["Is TF?"] == "Yes", "HGNC symbol"].str.upper()) - ligands                       # TF: Lambert 一覧
receptors = receptors - tfs                                                                          # 重なりは TF を優先

rows = [dict(役割="リガンド", 一覧の個数=len(ligands)), dict(役割="受容体", 一覧の個数=len(receptors)), dict(役割="TF", 一覧の個数=len(tfs))]
for r, s in zip(rows, [ligands, receptors, tfs]):
    for label, n in nets.items():
        r[f"ネットワークにある（{label}）"] = len(s & (set(n.src) | set(n.dst)))
pd.DataFrame(rows).set_index("役割")

## 4b. 経路上の辺だけを残す
- 動作: 除く前と除いた後のそれぞれで、層の向きの規則（リガンドは出発点、TF は終点）を適用し、最短ステップ数を計算して、`MAX_HOPS` 以内の経路上にある辺だけを残す
- 出力: 各段階の辺の数（除く前 / 除いた後）

In [ ]:
# ligand -> TF の経路上の辺だけを残す
import networkx as nx

pruned, stages, dist_tf = {}, {}, {}
for label, n in nets.items():
    e = n[["src", "dst"]]
    st = {"元の辺": len(e)}
    e = e[~e.dst.isin(ligands)]; st["リガンドへ入る辺を捨てる"] = len(e)
    e = e[~(e.src.isin(ligands) & ~e.dst.isin(receptors))]; st["リガンドからは受容体へ向かう辺だけ"] = len(e)
    e = e[~e.src.isin(tfs)]; st["TF から出る辺を捨てる"] = len(e)

    G = nx.DiGraph(); G.add_edges_from(zip(e.src, e.dst))                                    # 向きのある辺だけのグラフ
    src_l = [g for g in ligands if g in G]; src_t = [g for g in tfs if g in G]
    d_from = nx.multi_source_dijkstra_path_length(G, src_l, cutoff=MAX_HOPS)                # 最も近いリガンドからのステップ数
    d_to = nx.multi_source_dijkstra_path_length(G.reverse(copy=False), src_t, cutoff=MAX_HOPS)   # 最も近い TF までのステップ数
    on_path = [(a in d_from) and (b in d_to) and d_from[a] + 1 + d_to[b] <= MAX_HOPS for a, b in zip(e.src, e.dst)]
    keep = e[on_path]
    st[f"{MAX_HOPS} ステップ以内の経路上にある辺"] = len(keep)
    stages[label] = st
    pruned[label] = n.merge(keep, on=["src", "dst"])                                         # 重み・符号・出典を元の表から戻す
    dist_tf[label] = pd.Series({t: d_from[t] for t in src_t if t in d_from and t in set(keep.dst)})   # 各 TF のリガンドからの最短ステップ数
pd.DataFrame(stages)

## 4c. 枝刈りの前と後の比較
- 動作: 枝刈り前（Step 3 の `net`）と枝刈り後を比べ、辺・ノードの数、役割別のノード数、リガンドと TF の数、TF へ届く最短ステップ数の分布を出す
- 出力: 比較表と、ステップ数の分布

In [ ]:
# 枝刈りの前後を比べる
cmp = {}
for label in nets:
    a, b = nets[label], pruned[label]
    nodes_b = set(b.src) | set(b.dst)
    cmp[f"{label}: 枝刈り前"] = dict(辺=len(a), ノード=len(set(a.src) | set(a.dst)))
    cmp[f"{label}: 枝刈り後"] = dict(辺=len(b), ノード=len(nodes_b), リガンド=len(ligands & nodes_b), 受容体=len(receptors & nodes_b),
                                   TF=len(tfs & nodes_b), 途中のタンパク質=len(nodes_b - ligands - receptors - tfs))
display(pd.DataFrame(cmp).fillna("").T)
print("リガンドから各 TF までの最短ステップ数の分布（除いた後のネットワーク）")
dist_tf["除いた後"].value_counts().sort_index().to_frame("TFの数")

## 4d. 枝刈り後のネットワーク（表）
- 動作: 除いた後のネットワークを枝刈りしたもの（`net_lt`）を重みの大きい順に並べて見る。この表を以降の段階で使う
- 出力: 辺の表（先頭20行）

In [ ]:
# 以降の段階で使うネットワーク（ligand -> TF）
net_lt = pruned["除いた後"]
role = lambda g: "リガンド" if g in ligands else "受容体" if g in receptors else "TF" if g in tfs else "途中"
net_lt = net_lt.assign(src_role=net_lt.src.map(role), dst_role=net_lt.dst.map(role))
print("辺:", len(net_lt), " ノード:", len(set(net_lt.src) | set(net_lt.dst)))
net_lt.sort_values("weight", ascending=False).head(20)

## 4e. シグナル伝達ネットワーク（`net_lt`）の列の説明と、辺のスコア
> 指示: please explain the signaling network data columns, is there any score of the edges?

- 動作: `net_lt` の各列の意味を表にし、辺のスコア（`weight`）の分布を出典別に見る
- 出力: 列の説明の表、`weight` の統計（出典別）

**答え: 辺のスコアはある（`weight` 列、0〜1）。**
- 元のスコア: SIGNOR は `SCORE`（根拠の強さ）、OmniPath は論文の数。
- 各データベースの中で百分位（順位）に直して 0〜1 にそろえ、同じ辺が両方にあれば noisy-OR（1 − ∏(1 − s)）で合わせたものが `weight`。
- **元のスコアと百分位のスコアは、統合のときに表から外している**（`weight` だけが残っている）。必要なら列として残せる。
- スコアは辺の**順位づけ**用の目安であり、確率ではない。

In [ ]:
# net_lt の列の説明（日本語）と、weight の分布
doc_net = {
    "src": "上流のタンパク質（遺伝子名、大文字）。リガンドのときは出発点",
    "dst": "下流のタンパク質。TF のときは終点",
    "effect": "符号。+1 = 促進、-1 = 抑制、0 = 不明（データベース間で食い違う場合も 0）",
    "mechanism": "作用の仕組み。SIGNOR の仕組み（phosphorylation など）、`omnipath:` の後ろは根拠のリソース名、`complex` は複合体から展開した辺",
    "residue": "修飾されるアミノ酸残基（例 Ser15）。SIGNOR の記録があるときだけ",
    "source": "出典のデータベース。SIGNOR / OmniPath / OmniPath+SIGNOR（両方にある）",
    "weight": "辺のスコア（0〜1）。各データベース内の百分位を noisy-OR で合わせたもの。大きいほど根拠が多い・強い",
    "src_role": "上流の役割（リガンド / 受容体 / TF / 途中）",
    "dst_role": "下流の役割",
}
tab = pd.DataFrame({"説明": doc_net}).rename_axis("column")
miss = sorted(set(net_lt.columns) - set(tab.index))
display(tab)
print("説明がない列:", miss if miss else "なし")
print("\nweight の統計（出典別）")
net_lt.groupby("source").weight.describe().round(3)

## Step 5. ネットワーク構築の試験: 標的遺伝子から上流へ（例 AKT1）
> 指示: next step is to test network construction, starting from target gene x to upstream. for example AKT1. please show me the list of genes of each hops.

### 方針
- 使うネットワーク: Step 4 の **ligand → TF のネットワーク（`net_lt`）**。標的遺伝子から辺を**逆向きにたどり**、最短のステップ数（hop）で遺伝子を分ける。
- hop 1 = 標的を直接動かす遺伝子、hop 2 = その上流、…。1つの遺伝子は最短のステップ数で1回だけ数える。
- 各遺伝子について、標的の方へ向かう次の遺伝子（`next_gene`）、その辺の符号と重みを付ける（最短の経路が複数あるときは重みが最大の辺）。

**Note**
- `net_lt` はリガンド → TF の経路上の辺だけに枝刈りしたもの。標的が TF まで届かない経路（標的より下流が TF に届くまで長すぎる場合など）の上流は含まれない。枝刈り前のネットワーク（`net`）との差も 5c で確認する。

## 5a. 標的から上流へ最短ステップ数を計算する
- 動作: `net_lt` を有向グラフにして、標的から辺を逆向きにたどり、各遺伝子の最短ステップ数を求める。標的に向かう次の遺伝子と、その辺の符号・重みも付ける
- 出力: 標的がネットワークにあるか、上流の遺伝子の数（表 `up` の先頭5行）

In [ ]:
# 標的から上流へ（辺を逆向きにたどる）
import networkx as nx

def build_graph(df):
    g = nx.DiGraph()
    for r in df.itertuples(): g.add_edge(r.src, r.dst, effect=r.effect, weight=r.weight)
    return g

G = build_graph(net_lt)
print(TARGET, "は net_lt にある" if TARGET in G else "は net_lt にない")
dist = nx.single_source_shortest_path_length(G.reverse(copy=False), TARGET, cutoff=UP_HOPS)     # 標的までのステップ数

rows = []
for gname, h in dist.items():
    if h == 0: continue
    nxt = [(s, G[gname][s]["weight"], G[gname][s]["effect"]) for s in G.successors(gname) if dist.get(s) == h - 1]   # 標的に近づく辺
    s, w, e = max(nxt, key=lambda t: t[1])                                                                         # 重みが最大の辺
    rows.append(dict(hop=h, gene=gname, role=role(gname), next_gene=s, effect=e, weight=w))
up = pd.DataFrame(rows).sort_values(["hop", "weight"], ascending=[True, False]).reset_index(drop=True)
print("上流の遺伝子:", len(up))
up.head()

## 5b. 各 hop の遺伝子の数と役割
- 動作: hop ごとに遺伝子の数と、リガンド・受容体・TF・途中の内訳を数える
- 出力: hop 別の表

In [ ]:
# hop ごとの数
by_hop = up.groupby("hop").role.value_counts().unstack(fill_value=0)
by_hop.insert(0, "遺伝子数", up.groupby("hop").size())
by_hop

## 5c. 各 hop の遺伝子の一覧
- 動作: hop ごとに遺伝子名を、標的に向かう辺の重みが大きい順に並べて表示する。リガンドは `[L]`、受容体は `[R]`、TF は `[T]` を付ける
- 出力: hop ごとの遺伝子リスト。最後に、枝刈り前のネットワーク（`net`）で同じ試験をしたときの hop 別の数（比較用）

In [ ]:
# hop ごとの遺伝子リスト（重みの大きい順）
mark = {"リガンド": "[L]", "受容体": "[R]", "TF": "[T]", "途中": ""}
for h, g in up.groupby("hop"):
    names = [x + mark[r] for x, r in zip(g.gene, g.role)]
    print(f"--- hop {h}: {len(g)} 遺伝子 ---")
    print(", ".join(names), "\n")

# 比較: 枝刈り前のネットワーク（net）で同じ試験をすると
G0 = build_graph(net)
d0 = nx.single_source_shortest_path_length(G0.reverse(copy=False), TARGET, cutoff=UP_HOPS) if TARGET in G0 else {}
print("枝刈り前（net）の hop 別の遺伝子数:", pd.Series(d0).value_counts().sort_index().drop(0, errors="ignore").to_dict())

## Step 6. AKT1 の上流（リガンドまで）の図
> 指示: please make a visualization of AKT1 upstream up to ligand

### 方針
- 上流の遺伝子は1,581個あり、そのまま描くと読めない。そこで、**AKT1 への経路スコアが高い上位 `N_LIGANDS` 個のリガンド**を選び、各リガンドから AKT1 までの経路（Step 5 の `next_gene` をたどった経路）だけを描く。
- 経路スコア = 経路上の辺の重み（`weight`）の積。短く、強い辺でつながるリガンドほど高い。
- 横軸は AKT1 までのステップ数（hop）。右端が AKT1、左へ行くほど上流。リガンドは青、受容体は緑がかった灰色、途中のタンパク質は薄い灰色。辺の色は符号（促進 = 濃い青、抑制 = 赤の破線）。
- 図の文字は英語（プロジェクトの規則）。

**Note**
- 選んだ15個は「経路スコアの上位」で、AKT1 の上流のリガンド（hop 6 までで約140個）の代表であり、全部ではない。
- 各遺伝子は最短のステップ数の列に置く。1つの遺伝子から標的への経路は1本（重みが最大の辺）だけ描く。

## 6a. リガンドから AKT1 までの経路スコア
- 動作: Step 5 の表 `up` から、各遺伝子の経路スコア（標的までの辺の重みの積）を計算し、スコア上位のリガンドと、その経路を取り出す
- 出力: 上位リガンドの表（hop・経路スコア・経路）

In [ ]:
# 経路スコア = 標的までの辺の重みの積（標的に近い遺伝子から順に計算）
score = {TARGET: 1.0}
for r in up.sort_values("hop").itertuples():
    score[r.gene] = r.weight * score[r.next_gene]
up["path_score"] = up.gene.map(score)

nxt = dict(zip(up.gene, up.next_gene))                      # 各遺伝子 -> 標的に向かう次の遺伝子
hop = dict(zip(up.gene, up.hop)); hop[TARGET] = 0
eff = dict(zip(up.gene, up.effect))                          # 各遺伝子 -> 次の遺伝子への辺の符号

lig = up[up.role == "リガンド"].sort_values("path_score", ascending=False).head(N_LIGANDS)

def path_of(g):
    p = [g]
    while p[-1] != TARGET: p.append(nxt[p[-1]])
    return p

lig_table = pd.DataFrame({"ligand": lig.gene, "hop": lig.hop, "path_score": lig.path_score.round(3),
                          "path": [" > ".join(path_of(g)) for g in lig.gene]}).reset_index(drop=True)
pd.set_option("display.max_colwidth", 120)
lig_table

## 6b. 図を描く
- 動作: 選んだリガンドの経路を、AKT1 を右端、上流を左へ並べて描く。列は hop、同じ列の中の順序は、下流側の遺伝子の位置に合わせて交差を減らす
- 出力: 図（画面に表示し、PNG ファイルにも保存する）

In [ ]:
# 経路の図を描く（平らで地味な色、図中の文字は英語）
import matplotlib.pyplot as plt

nodes, edges = {TARGET}, set()
for g in lig.gene:
    for a, b in zip(path_of(g)[:-1], path_of(g)[1:]):
        nodes.update([a, b]); edges.add((a, b))

# 列（hop）ごとに縦の位置を決める。標的に近い列から順に、下流側の遺伝子の平均位置に揃える
cols = {}
for n in nodes: cols.setdefault(hop[n], []).append(n)
ypos = {TARGET: 0.0}
for h in sorted(cols):
    if h == 0: continue
    ns = sorted(cols[h], key=lambda n: (ypos[nxt[n]], n))
    for i, n in enumerate(ns): ypos[n] = (i - (len(ns) - 1) / 2) * 1.0 + 0.0 * i
pos = {n: (-hop[n], ypos[n]) for n in nodes}                # 右端が AKT1（x = 0）

colour = lambda n: "#6b6b6b" if n == TARGET else "#5b7a99" if role(n) == "リガンド" else "#7a9a8a" if role(n) == "受容体" else "#d0d0d0"
fig, ax = plt.subplots(figsize=(15, max(6, 0.5 * max(len(v) for v in cols.values()) + 2)))   # 一番縦に並ぶ列に合わせて高さを決める
for a, b in edges:
    act = eff[a] == 1
    ax.annotate("", xy=pos[b], xytext=pos[a], arrowprops=dict(arrowstyle="->", color="#5a8f6a" if act else "#b5655c", lw=1.1,
                linestyle="-" if act else "--", shrinkA=18, shrinkB=18, alpha=0.9))
for n in nodes:
    x, y = pos[n]
    ax.text(x, y, n, ha="center", va="center", fontsize=8, color="white" if colour(n) != "#d0d0d0" else "black",
            bbox=dict(boxstyle="round,pad=0.3", fc=colour(n), ec="none"), zorder=3)
H = max(hop[n] for n in nodes)
ax.set_xlim(-H - 0.6, 0.6); ax.set_xticks(range(-H, 1)); ax.set_xticklabels([f"hop {-x}" if x else TARGET for x in range(-H, 1)])
ys = list(ypos.values()); ax.set_ylim(min(ys) - 1.2, max(ys) + 1.2)      # 縦の範囲を遺伝子の位置に合わせる（これがないと一部が枠の外に出る）
ax.set_yticks([]); ax.spines[["top", "right", "left"]].set_visible(False)
ax.set_title(f"{TARGET} upstream: best paths from the top {N_LIGANDS} ligands (hop = shortest steps to {TARGET})")
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0], [0], marker="s", color="w", markerfacecolor="#5b7a99", markersize=10, label="ligand"),
                   Line2D([0], [0], marker="s", color="w", markerfacecolor="#7a9a8a", markersize=10, label="receptor"),
                   Line2D([0], [0], marker="s", color="w", markerfacecolor="#d0d0d0", markersize=10, label="other protein"),
                   Line2D([0], [0], color="#5a8f6a", label="activation"), Line2D([0], [0], color="#b5655c", linestyle="--", label="inhibition")],
          loc="upper left", frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f"network_development_{TARGET}_upstream.png", dpi=150); plt.show()

## Step 7. 辺を「経路全体のスコア」だけでなく「各ノードのスコア」でも選ぶ（AKT1 の上流）
> 指示: how about selecting edge not by overall score but also score from each node. Please show me the result of AKT1

### この指示の解釈（違っていたら知らせてほしい）
Step 5〜6 は、経路全体の重みの積（グローバルなスコア）が最大の1本だけを標的への経路としていた。今回は、**各ノードに入ってくる辺を、そのノードの中で比べる（ローカルなスコア）**:
1. ノード `v` に入ってくる辺 `u→v` を重みの大きい順に並べ、上位 `K_IN` 本だけ残す。
2. さらに、`v` の最大の重みに対する割合（`rel` = 辺の重み / `v` に入る辺の最大の重み）が `REL_MIN` 以上のものだけ残す。
3. 残した辺だけで、AKT1 から上流へ最大 `UP_HOPS` ステップたどる。

つまり、1本の最良経路ではなく、**各ノードで有力な上流を数本ずつ**残す（枝分かれする）。経路全体のスコアは、残った辺の上で図のリガンドを選ぶのに使う。

**Note**
- 辺の重みがほぼ同じ辺どうしを比べるので、重みの小さい（根拠の少ない）上流は、そのノードに強い上流が `K_IN` 本以上あれば落ちる。
- 符号は選択に使わない（辺の色で見るだけ）。

## 7a. 各ノードで入ってくる辺を選ぶ
- 動作: `net_lt` の各辺に、「その辺の下流ノードの中での順位（`rank_in`）」と「そのノードの最大の重みに対する割合（`rel`）」を付け、`K_IN` 位以内かつ `rel ≥ REL_MIN` の辺だけ残す
- 出力: 選ぶ前と後の辺の数、1ノードあたりの入ってくる辺の数の分布

In [ ]:
# ノードごとの順位と相対スコアで辺を選ぶ
nl = net_lt.copy()
nl["rank_in"] = nl.groupby("dst").weight.rank(ascending=False, method="first")           # 下流ノードの中での順位（1が最大）
nl["rel"] = nl.weight / nl.groupby("dst").weight.transform("max")                         # そのノードの最大の重みに対する割合
sel_edges = nl[(nl.rank_in <= K_IN) & (nl.rel >= REL_MIN)]
print(f"辺: {len(nl)} -> {len(sel_edges)}  （K_IN={K_IN}, REL_MIN={REL_MIN}）")
print("入ってくる辺の数（中央値 / 90%点）: 選ぶ前", nl.groupby("dst").size().median(), "/", nl.groupby("dst").size().quantile(.9),
      " 選んだ後", sel_edges.groupby("dst").size().median(), "/", sel_edges.groupby("dst").size().quantile(.9))
sel_edges.head()

## 7b. 選んだ辺で AKT1 から上流へたどる
- 動作: 選んだ辺だけで、AKT1 から辺を逆向きに最大 `UP_HOPS` ステップたどり、hop ごとの遺伝子の表 `up_l` を作る。次の遺伝子は、標的に近づく辺のうち重みが最大のもの
- 出力: hop ごとの遺伝子の数と役割の内訳。Step 5（経路全体のスコアだけ）との比較

In [ ]:
# 選んだ辺だけで上流へ
Gl = nx.DiGraph()
for r in sel_edges.itertuples(): Gl.add_edge(r.src, r.dst, effect=r.effect, weight=r.weight, rel=r.rel)
dist_l = nx.single_source_shortest_path_length(Gl.reverse(copy=False), TARGET, cutoff=UP_HOPS) if TARGET in Gl else {}

rows = []
for g, h in dist_l.items():
    if h == 0: continue
    cand = [(s, Gl[g][s]["weight"], Gl[g][s]["effect"], Gl[g][s]["rel"]) for s in Gl.successors(g) if dist_l.get(s) == h - 1]
    s, w, e, rl = max(cand, key=lambda t: t[1])
    rows.append(dict(hop=h, gene=g, role=role(g), next_gene=s, effect=e, weight=w, rel=round(rl, 3)))
up_l = pd.DataFrame(rows).sort_values(["hop", "weight"], ascending=[True, False]).reset_index(drop=True)

by_hop_l = up_l.groupby("hop").role.value_counts().unstack(fill_value=0)
by_hop_l.insert(0, "遺伝子数（ノードごと選択）", up_l.groupby("hop").size())
by_hop_l["Step 5（経路全体のスコアだけ）"] = up.groupby("hop").size()
by_hop_l

## 7c. 各 hop の遺伝子の一覧（AKT1）
- 動作: hop ごとに遺伝子名を重みの大きい順に並べる。リガンドは `[L]`、受容体は `[R]`、TF は `[T]`。最後に、AKT1 の既知の調節因子が Step 5 と今回でどの hop にあるかを並べる（点検のためで、選択には使っていない）
- 出力: hop ごとのリスト、既知の調節因子の hop の比較表

In [ ]:
# hop ごとの遺伝子リスト
for h, g in up_l.groupby("hop"):
    print(f"--- hop {h}: {len(g)} 遺伝子 ---")
    print(", ".join(x + mark[r] for x, r in zip(g.gene, g.role)), "\n")

# 点検用: AKT1 の既知の調節因子が見つかるか（選択には使っていない）
check = ["PDPK1", "PIK3CA", "PIK3R1", "RICTOR", "MTOR", "PTEN", "SRC", "IRS1", "IGF1R", "EGFR", "INSR", "PDGFRB"]
hop_g = dict(zip(up.gene, up.hop)); hop_l = dict(zip(up_l.gene, up_l.hop))
pd.DataFrame({"Step 5 の hop": [hop_g.get(g, "-") for g in check], "今回の hop": [hop_l.get(g, "-") for g in check]}, index=check)

## 7d. 図（選んだ辺の上でリガンドを選ぶ）
- 動作: 選んだ辺の上で、AKT1 への経路スコア（辺の重みの積）が高い上位 `N_LIGANDS` 個のリガンドとその経路を、Step 6 と同じ形で描く
- 出力: 図（画面と PNG に保存）。選べるリガンドが `N_LIGANDS` より少なければ全部描く

In [ ]:
# 選んだ辺の上で、リガンドから AKT1 までの経路を描く
score_l = {TARGET: 1.0}
for r in up_l.sort_values("hop").itertuples(): score_l[r.gene] = r.weight * score_l[r.next_gene]
up_l["path_score"] = up_l.gene.map(score_l)
nxt_l = dict(zip(up_l.gene, up_l.next_gene)); hop_l[TARGET] = 0; eff_l = dict(zip(up_l.gene, up_l.effect))
lig_l = up_l[up_l.role == "リガンド"].sort_values("path_score", ascending=False).head(N_LIGANDS)

def path_l(g):
    pth = [g]
    while pth[-1] != TARGET: pth.append(nxt_l[pth[-1]])
    return pth

display(pd.DataFrame({"ligand": lig_l.gene, "hop": lig_l.hop, "path_score": lig_l.path_score.round(3), "path": [" > ".join(path_l(g)) for g in lig_l.gene]}).reset_index(drop=True))

nodes_l, edges_l = {TARGET}, set()
for g in lig_l.gene:
    for a, b in zip(path_l(g)[:-1], path_l(g)[1:]): nodes_l.update([a, b]); edges_l.add((a, b))
cols_l = {}
for n in nodes_l: cols_l.setdefault(hop_l[n], []).append(n)
ypos_l = {TARGET: 0.0}
for h in sorted(cols_l):
    if h == 0: continue
    for i, n in enumerate(sorted(cols_l[h], key=lambda n: (ypos_l[nxt_l[n]], n))): ypos_l[n] = i - (len(cols_l[h]) - 1) / 2
pos_l = {n: (-hop_l[n], ypos_l[n]) for n in nodes_l}
colour = lambda n: "#6b6b6b" if n == TARGET else "#5b7a99" if role(n) == "リガンド" else "#7a9a8a" if role(n) == "受容体" else "#d0d0d0"
fig, ax = plt.subplots(figsize=(15, max(6, 0.5 * max(len(v) for v in cols_l.values()) + 2)))
for a, b in edges_l:
    act = eff_l[a] == 1
    ax.annotate("", xy=pos_l[b], xytext=pos_l[a], arrowprops=dict(arrowstyle="->", color="#5a8f6a" if act else "#b5655c", lw=1.1,
                linestyle="-" if act else "--", shrinkA=18, shrinkB=18, alpha=0.9))
for n in nodes_l:
    x, y = pos_l[n]
    ax.text(x, y, n, ha="center", va="center", fontsize=8, color="white" if colour(n) != "#d0d0d0" else "black",
            bbox=dict(boxstyle="round,pad=0.3", fc=colour(n), ec="none"), zorder=3)
Hl = max(hop_l[n] for n in nodes_l)
ax.set_xlim(-Hl - 0.6, 0.6); ax.set_xticks(range(-Hl, 1)); ax.set_xticklabels([f"hop {-x}" if x else TARGET for x in range(-Hl, 1)])
ys = list(ypos_l.values()); ax.set_ylim(min(ys) - 1.2, max(ys) + 1.2); ax.set_yticks([]); ax.spines[["top", "right", "left"]].set_visible(False)
ax.set_title(f"{TARGET} upstream, per-node edge selection (top {K_IN} incoming edges per node, relative weight >= {REL_MIN}): top {len(lig_l)} ligands")
ax.legend(handles=[Line2D([0], [0], marker="s", color="w", markerfacecolor="#5b7a99", markersize=10, label="ligand"),
                   Line2D([0], [0], marker="s", color="w", markerfacecolor="#7a9a8a", markersize=10, label="receptor"),
                   Line2D([0], [0], marker="s", color="w", markerfacecolor="#d0d0d0", markersize=10, label="other protein"),
                   Line2D([0], [0], color="#5a8f6a", label="activation"), Line2D([0], [0], color="#b5655c", linestyle="--", label="inhibition")],
          loc="upper left", frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f"network_development_{TARGET}_upstream_local.png", dpi=150); plt.show()

## Step 8. 直接の辺のスコアで、標的から1ホップずつ上流を選ぶ（AKT1）
> 指示: Yes, build it（証拠スコア `E` で、各遺伝子の上位 X 個の上流を選ぶ）

### 方法（あなたの考えのとおり）
1. **hop 1**: 標的（AKT1）に**直接入る辺**を、スコア `E` の大きい順に並べ、上位 `X_PER_NODE` 個の上流遺伝子を選ぶ。
2. **hop 2**: hop 1 の**各遺伝子**に直接入る辺を、同じく `E` の大きい順に並べ、各遺伝子で上位 `X_PER_NODE` 個を選ぶ（すでに選んだ遺伝子は除く）。
3. これを `UP_HOPS` まで繰り返す。リガンドに着いたらそこで止める。

### 辺のスコア `E`（証拠スコア）
`E = 1 − (1 − log(1+論文数) / log(1+最大の論文数)) × (1 − SIGNOR の SCORE)`
- OmniPath の論文数と SIGNOR の `SCORE` という**元のスコアを、百分位に直さずそのまま**使う（百分位にすると、論文1本の辺がすべて同じ値になるため）。
- 論文数は対数で効かせる（1本と2本の差を、49本と50本の差より大きくする）。両方あれば noisy-OR で合わせる。

**Note**
- PIK3CA / PIK3R1 → AKT1 のように、データベースに直接の記録が少ない辺は、この方法でも低い順位になる（PI3K は脂質 PIP3 を介して AKT1 に作用するため）。
- 比較の表（8b）の「既知の調節因子」は結果の点検に使うだけで、選択には使っていない。

## 8a. 辺の証拠スコア `E` を計算する
- 動作: `net_lt` の各辺に、OmniPath の論文数と SIGNOR の `SCORE` を付け、`E` を計算する
- 出力: `E` と、これまでの `weight` の値の種類の数・分布の比較

In [ ]:
# 各辺の論文数・SIGNOR スコアから証拠スコア E を作る
oe = omni_edges["除いた後"].set_index(["src", "dst"])          # OmniPath の辺（score = 論文の数）
rmax = oe.score.max()                                         # 最大の論文数
sgi = edges_sg.set_index(["src", "dst"]).score                # SIGNOR の SCORE
key = pd.MultiIndex.from_frame(net_lt[["src", "dst"]])

ev = net_lt.copy()
ev["refs"] = oe.score.reindex(key).values
ev["signor"] = sgi.reindex(key).values
A = np.log1p(ev.refs.fillna(0)) / np.log1p(rmax)             # 論文数（対数）を 0〜1 に
B = ev.signor.fillna(0)                                       # SIGNOR のスコア（なければ 0）
ev["E"] = (1 - (1 - A) * (1 - B)).round(4)                    # noisy-OR

print("値の種類の数: weight =", ev.weight.nunique(), "/ E =", ev.E.nunique(), "（辺", len(ev), "本）")
print("最も多い値の割合: weight", round(ev.weight.value_counts(normalize=True).iloc[0], 3), "/ E", round(ev.E.value_counts(normalize=True).iloc[0], 3))
ev[["src", "dst", "effect", "refs", "signor", "weight", "E"]].describe().round(3)

## 8b. 標的から1ホップずつ、直接の辺のスコアで選ぶ
- 動作: AKT1 から始め、各遺伝子に入る辺を `E` の大きい順に並べて上位 `X_PER_NODE` 個を選び、上流へ1ホップずつ進む（すでに選んだ遺伝子は除く。リガンドでは止める）
- 出力: hop ごとの遺伝子数と役割の内訳（Step 5・Step 7 と比較）、既知の調節因子の点検表

In [ ]:
# 1ホップずつ、各遺伝子の上位 X を E で選ぶ
inc = {v: g.sort_values("E", ascending=False) for v, g in ev.groupby("dst")}      # 各遺伝子に入る辺（E の大きい順）
reached, kept, frontier = {TARGET: 0}, [], [TARGET]
for h in range(1, UP_HOPS + 1):
    new = []
    for v in frontier:
        if v not in inc: continue
        cand = inc[v][~inc[v].src.isin(reached)].head(X_PER_NODE)                  # 前のホップまでに選んだ遺伝子は除く
        for r in cand.itertuples():
            kept.append(dict(hop=h, gene=r.src, to=v, effect=r.effect, E=r.E, weight=r.weight))
            if r.src not in new: new.append(r.src)
    for g in new: reached[g] = h
    frontier = [g for g in new if role(g) != "リガンド"]                              # リガンドは上流がないので止める

kept = pd.DataFrame(kept)
# 各遺伝子について、1つ下流の層への辺のうち E が最大のものを「次の遺伝子」にする
up8 = kept.sort_values("E").groupby("gene").tail(1).rename(columns={"to": "next_gene"})
up8["role"] = up8.gene.map(role)
up8 = up8.sort_values(["hop", "E"], ascending=[True, False]).reset_index(drop=True)

by_hop8 = up8.groupby("hop").role.value_counts().unstack(fill_value=0)
by_hop8.insert(0, "遺伝子数（Step 8: E で選ぶ）", up8.groupby("hop").size())
by_hop8["Step 5（経路全体のスコア）"] = up.groupby("hop").size()
by_hop8["Step 7（weight でノードごと）"] = up_l.groupby("hop").size()
display(by_hop8)

# 点検用: 既知の調節因子が見つかるか（選択には使っていない）
check = ["PDPK1", "PIK3CA", "PIK3R1", "RICTOR", "MLST8", "MTOR", "PTEN", "PHLPP1", "PPP2CA", "PTPN1", "ILK", "SRC", "IRS1", "IGF1R", "EGFR", "INSR"]
h5 = dict(zip(up.gene, up.hop)); h7 = dict(zip(up_l.gene, up_l.hop)); h8 = dict(zip(up8.gene, up8.hop))
pd.DataFrame({"Step 5": [h5.get(g, "-") for g in check], "Step 7": [h7.get(g, "-") for g in check], "Step 8": [h8.get(g, "-") for g in check]}, index=check)

## 8c. 各 hop の遺伝子の一覧（AKT1、Step 8）
- 動作: hop ごとに遺伝子名を `E` の大きい順に並べる。リガンドは `[L]`、受容体は `[R]`、TF は `[T]`
- 出力: hop ごとのリスト

In [ ]:
# hop ごとの遺伝子リスト（E の大きい順）
for h, g in up8.groupby("hop"):
    print(f"--- hop {h}: {len(g)} 遺伝子 ---")
    print(", ".join(x + mark[r] for x, r in zip(g.gene, g.role)), "\n")

## 8d. 図（Step 8 で選んだ辺の上でリガンドを選ぶ）
- 動作: 描き方が3回目になるので、描画を関数 `draw_upstream` にまとめる。選んだ辺の上で、AKT1 への経路スコア（経路上の `E` の積）が高い上位 `N_LIGANDS` 個のリガンドとその経路を描く
- 出力: 経路の表と図（画面と PNG に保存）

In [ ]:
# 描画を関数にまとめる（Step 6・7 と同じ描き方で3回目）
def draw_upstream(tbl, edge_col, title, fname, ligands=None, extra_edges=()):
    # ligands: 描くリガンドを指定（なければ経路スコア上位 N_LIGANDS 個）/ extra_edges: 追加で描く辺 (上流, 下流) のリスト
    ev_eff = ev.set_index(["src", "dst"]).effect                              # 辺ごとの符号（追加の辺の色に使う）
    sc = {TARGET: 1.0}
    for r in tbl.sort_values("hop").itertuples(): sc[r.gene] = getattr(r, edge_col) * sc[r.next_gene]      # 経路スコア = 辺のスコアの積
    t = tbl.assign(path_score=tbl.gene.map(sc))
    nx_, hp, ef = dict(zip(t.gene, t.next_gene)), dict(zip(t.gene, t.hop)), dict(zip(t.gene, t.effect)); hp[TARGET] = 0
    lg = t[t.gene.isin(ligands)] if ligands else t[t.role == "リガンド"].sort_values("path_score", ascending=False).head(N_LIGANDS)
    def pth(g):
        p = [g]
        while p[-1] != TARGET: p.append(nx_[p[-1]])
        return p
    display(pd.DataFrame({"start": lg.gene, "hop": lg.hop, "path_score": lg.path_score.round(3), "path": [" > ".join(pth(g)) for g in lg.gene]}).reset_index(drop=True))
    nd, ed = {TARGET}, set()
    for g in lg.gene:
        for a, b in zip(pth(g)[:-1], pth(g)[1:]): nd.update([a, b]); ed.add((a, b))
    skipped = 0
    for a, b in extra_edges:                                               # 追加の辺は、列が1つずつ進むものだけ描く
        if a in hp and b in hp and hp[a] == hp[b] + 1: nd.update([a, b]); ed.add((a, b))
        else: skipped += 1
    if skipped: print(f"列がそろわない辺 {skipped} 本は描かない")
    cl = {}
    for n in nd: cl.setdefault(hp[n], []).append(n)
    yp = {TARGET: 0.0}
    for h in sorted(cl):
        if h == 0: continue
        for i, n in enumerate(sorted(cl[h], key=lambda n: (yp[nx_[n]], n))): yp[n] = i - (len(cl[h]) - 1) / 2
    ps = {n: (-hp[n], yp[n]) for n in nd}
    colr = lambda n: "#6b6b6b" if n == TARGET else "#5b7a99" if role(n) == "リガンド" else "#7a9a8a" if role(n) == "受容体" else "#d0d0d0"
    fig, ax = plt.subplots(figsize=(15, max(6, 0.5 * max(len(v) for v in cl.values()) + 2)))
    for a, b in ed:
        act = ev_eff[(a, b)] == 1
        ax.annotate("", xy=ps[b], xytext=ps[a], arrowprops=dict(arrowstyle="->", color="#5a8f6a" if act else "#b5655c", lw=1.1,
                    linestyle="-" if act else "--", shrinkA=18, shrinkB=18, alpha=0.9))
    for n in nd:
        x, y = ps[n]
        ax.text(x, y, n, ha="center", va="center", fontsize=8, color="white" if colr(n) != "#d0d0d0" else "black",
                bbox=dict(boxstyle="round,pad=0.3", fc=colr(n), ec="none"), zorder=3)
    Hh = max(hp[n] for n in nd)
    ax.set_xlim(-Hh - 0.6, 0.6); ax.set_xticks(range(-Hh, 1)); ax.set_xticklabels([f"hop {-x}" if x else TARGET for x in range(-Hh, 1)])
    yy = list(yp.values()); ax.set_ylim(min(yy) - 1.2, max(yy) + 1.2); ax.set_yticks([]); ax.spines[["top", "right", "left"]].set_visible(False)
    ax.set_title(title)
    ax.legend(handles=[Line2D([0], [0], marker="s", color="w", markerfacecolor="#5b7a99", markersize=10, label="ligand"),
                       Line2D([0], [0], marker="s", color="w", markerfacecolor="#7a9a8a", markersize=10, label="receptor"),
                       Line2D([0], [0], marker="s", color="w", markerfacecolor="#d0d0d0", markersize=10, label="other protein"),
                       Line2D([0], [0], color="#5a8f6a", label="activation"), Line2D([0], [0], color="#b5655c", linestyle="--", label="inhibition")],
              loc="upper left", frameon=False, fontsize=8)
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()

draw_upstream(up8, "E", f"{TARGET} upstream: top {X_PER_NODE} direct edges per gene by evidence score E, top {N_LIGANDS} ligands", f"network_development_{TARGET}_upstream_E.png")

## Step 9. いろいろなパターンを比べる（AKT1、基準は KDR のシグナル）
> 指示: yes try evidence score. AND my image of AKT1 upstream is KDR signaling. / I guess we need to try various pattern to see which is best.

### 基準（あなたのイメージ）
**VEGFA / VEGFC / VEGFD → KDR（VEGFR2）→（PI3K・SRC・PLCG1 など）→ AKT1** という流れが、上流の図に現れること。各パターンについて次を調べる。
1. KDR と VEGF 系のリガンドが上流として選ばれるか。何ホップ目か。
2. VEGFA から AKT1 までの最良の経路（辺のスコア `E` の積が最大、符号は促進と抑制を別々に）。促進する経路があるか。
3. 選ばれたネットワークの大きさ（小さいほど読みやすい）。
4. AKT1 の既知の調節因子が2ホップ以内に何個入るか（点検用）。

### 比べるパターン
| 名前 | 選び方 | 辺のスコア |
|---|---|---|
| A | 各遺伝子の上位 5（Step 7） | `weight`（百分位 + noisy-OR） |
| B | 各遺伝子の上位 10（Step 8） | `E`（証拠スコア） |
| C | 各遺伝子の上位 20 | `E` |
| D | 各遺伝子の上位 40 | `E` |
| E | 各遺伝子の上位 10 | `E` × 特異性（出る辺の多い上流を下げる） |
| G | ホップごとの選択はしない。リガンドごとに、AKT1 まで**促進する最良経路**（`E` の積が最大）を1本ずつ選んで合わせる | `E` |

**Note**
- 基準は KDR のシグナルだけなので、これで「一番よい」と決めるのは早い。他のシグナル（IGF1R、EGFR など）でも比べる必要がある（別のセルで `TARGET` や基準を変えて試せる）。
- 既知の調節因子の点検は、選択には使わず、結果の評価だけに使う。

## 9a. パターンごとに上流の辺を選ぶ
- 動作: 1ホップずつ上位 X を選ぶ関数 `select_upstream` で A〜E を作り、G はリガンドごとの促進する最良経路から作る。各パターンの「選んだ辺」の表を `patterns` に入れる
- 出力: パターンごとの辺の数

In [ ]:
# パターンごとに「選んだ辺」(gene -> to) の表を作る
ev["spec"] = 1 / np.log2(2 + ev.src.map(ev.groupby("src").size()))      # 特異性: 出る辺が多い上流ほど小さい
ev["E_spec"] = ev.E * ev.spec

def select_upstream(df, col, X):
    # 標的から1ホップずつ、各遺伝子に入る辺を col の大きい順に並べて上位 X を選ぶ（前のホップまでに選んだ遺伝子は除く）
    inc = {v: g.sort_values(col, ascending=False) for v, g in df.groupby("dst")}
    reached, kept, frontier = {TARGET: 0}, [], [TARGET]
    for h in range(1, UP_HOPS + 1):
        new = []
        for v in frontier:
            if v not in inc: continue
            for r in inc[v][~inc[v].src.isin(reached)].head(X).itertuples():
                kept.append(dict(hop=h, gene=r.src, to=v, effect=r.effect, E=r.E, weight=r.weight))
                if r.src not in new: new.append(r.src)
        for g in new: reached[g] = h
        frontier = [g for g in new if role(g) != "リガンド"]
    return pd.DataFrame(kept)

patterns = {
    "A: weight, top5": select_upstream(ev, "weight", 5),
    "B: E, top10": select_upstream(ev, "E", 10),
    "C: E, top20": select_upstream(ev, "E", 20),
    "D: E, top40": select_upstream(ev, "E", 40),
    "E: E x specificity, top10": select_upstream(ev, "E_spec", 10),
}

# G: AKT1 から逆向きに、(遺伝子, AKT1 への効果の符号) ごとの最良経路（E の積）を UP_HOPS まで求める
inc_all = {}
for r in ev[ev.effect != 0].itertuples(): inc_all.setdefault(r.dst, []).append((r.src, r.effect, r.E))
best = {(TARGET, 1): (1.0, [TARGET])}; fr = dict(best)
for _ in range(UP_HOPS):
    new = {}
    for (v, s), (sc, pth) in fr.items():
        for u, e_, w in inc_all.get(v, []):
            if u in pth: continue
            k, val = (u, s * e_), sc * w
            if val > best.get(k, (0,))[0] and val > new.get(k, (0,))[0]: new[k] = (val, [u] + pth)
    best.update(new); fr = new
rowsG = set()
for (g, s), (sc, pth) in best.items():
    if s == 1 and role(g) == "リガンド":                                    # 促進する経路があるリガンドごとに、その最良経路の辺を集める
        for a, b in zip(pth[:-1], pth[1:]): rowsG.add((a, b))
eff_map = ev.set_index(["src", "dst"])[["effect", "E", "weight"]]
patterns["G: best activating path per ligand"] = pd.DataFrame([dict(gene=a, to=b, effect=eff_map.loc[(a, b), "effect"], E=eff_map.loc[(a, b), "E"],
                                                                  weight=eff_map.loc[(a, b), "weight"]) for a, b in rowsG])
pd.Series({k: len(v) for k, v in patterns.items()}, name="選んだ辺の数").to_frame()

## 9b. パターンの比較（KDR を基準に）
- 動作: 各パターンで、(1) 辺・遺伝子・リガンドの数、(2) KDR と VEGFA / VEGFC / VEGFD が選ばれたか（AKT1 から何ホップか）、(3) VEGFA から AKT1 までの最良経路（選んだ辺の中、`E` の積が最大で、促進する経路を優先）、(4) AKT1 の既知の調節因子が2ホップ以内に何個あるか、を数える
- 出力: 比較表

In [ ]:
# 選んだ辺の中で、ある遺伝子から標的への最良経路（符号つき、E の積）を求める
def best_in(edges, src):
    out = {}
    for r in edges.itertuples(): out.setdefault(r.gene, []).append((r.to, r.effect, r.E))      # gene -> to（標的に向かう向き）
    st = {(src, 1): (1.0, [src])}; fr = dict(st)
    for _ in range(UP_HOPS + 1):
        new = {}
        for (v, s), (sc, pth) in fr.items():
            for u, e_, w in out.get(v, []):
                if u in pth or e_ == 0: continue
                k, val = (u, s * e_), sc * w
                if val > st.get(k, (0,))[0] and val > new.get(k, (0,))[0]: new[k] = (val, pth + [u])
        st.update(new); fr = new
    return st

known = ["PDPK1", "PIK3CA", "PIK3R1", "RICTOR", "MTOR", "PTEN", "PHLPP1", "PPP2CA", "SRC", "ILK"]   # 点検用
rows, paths = [], {}
for name, kp in patterns.items():
    nodes = set(kp.gene) | {TARGET}
    hop_p = kp.groupby("gene").hop.min().to_dict() if "hop" in kp else {}
    res_ = dict(パターン=name, 辺=len(kp), 遺伝子=len(nodes), リガンド=sum(role(g) == "リガンド" for g in nodes))
    if hop_p:
        res_[f"{REF_RECEPTOR} のhop"] = hop_p.get(REF_RECEPTOR, "-")
        res_["VEGF系リガンド（hop）"] = ", ".join(f"{g}:{hop_p[g]}" for g in REF_LIGANDS if g in hop_p) or "-"
        res_["既知の調節因子（2ホップ以内）"] = sum(hop_p.get(g, 99) <= 2 for g in known)
    else:   # G は hop を持たないので、最短ステップ数で代用
        Gx = nx.DiGraph(); Gx.add_edges_from(zip(kp.gene, kp.to)); d = nx.single_source_shortest_path_length(Gx.reverse(copy=False), TARGET)
        res_[f"{REF_RECEPTOR} のhop"] = d.get(REF_RECEPTOR, "-")
        res_["VEGF系リガンド（hop）"] = ", ".join(f"{g}:{d[g]}" for g in REF_LIGANDS if g in d) or "-"
        res_["既知の調節因子（2ホップ以内）"] = sum(d.get(g, 99) <= 2 for g in known)
    b = best_in(kp, "VEGFA")
    for s, lab in [(1, "促進"), (-1, "抑制")]:
        if (TARGET, s) in b: paths[(name, lab)] = (round(b[(TARGET, s)][0], 3), " > ".join(b[(TARGET, s)][1]))
    res_["VEGFA→AKT1 促進の経路"] = f"{paths[(name, '促進')][0]}" if (name, "促進") in paths else "なし"
    rows.append(res_)
cmp9 = pd.DataFrame(rows).set_index("パターン")
cmp9

## 9c. VEGFA から AKT1 までの最良経路（パターンごと）
- 動作: 9b で求めた、VEGFA から AKT1 までの `E` の積が最大の経路を、促進・抑制に分けて表示する
- 出力: パターンごとの経路

In [ ]:
# VEGFA -> AKT1 の最良経路（選んだ辺の中）
pd.DataFrame([dict(パターン=k[0], 符号=k[1], スコア=v[0], 経路=v[1]) for k, v in paths.items()]).set_index(["パターン", "符号"])

## 9d. KDR のシグナルが、データの中でどうつながっているか
- 動作: (1) KDR に入る辺（リガンド・抑制因子）、(2) KDR から出る辺ごとに、AKT1 までの最良経路（`E` の積が最大、促進を優先）を、パターンの選択とは関係なく `net_lt` 全体で調べる
- 出力: 2つの表（KDR に入る辺、KDR から AKT1 への枝ごとの経路）

**見るポイント**: KDR → PIK3R1 → AKT1 のような短い PI3K の経路があるか、あってもスコアが低くて他の経路に負けていないか。

In [ ]:
# KDR に入る辺と、KDR から AKT1 への枝ごとの最良経路
display(ev[ev.dst == REF_RECEPTOR][["src", "effect", "E", "refs", "signor"]].sort_values("E", ascending=False).reset_index(drop=True).rename(columns={"src": "KDR に入る（上流）"}))

all_edges = ev.rename(columns={"src": "gene", "dst": "to"})                          # best_in は gene -> to の形を使う
rows = []
for r in ev[(ev.src == REF_RECEPTOR) & (ev.effect != 0)].sort_values("E", ascending=False).itertuples():
    b = best_in(all_edges, r.dst)
    for s, lab in [(1, "促進"), (-1, "抑制")]:
        if (TARGET, s) in b:
            pth = [REF_RECEPTOR] + b[(TARGET, s)][1]
            rows.append(dict(枝=r.dst, KDR_to_枝_E=r.E, 符号=lab, 経路=" > ".join(pth), ステップ数=len(pth) - 1, 経路スコア=round(r.E * b[(TARGET, s)][0], 3)))
pd.DataFrame(rows).sort_values(["枝", "符号"]).reset_index(drop=True)

## Step 10. 上位 X 個ではなく、スコアのしきい値で選ぶ（weight と E を合わせたスコア）
> 指示: Please not use top X but threshold of a score, which combines weight and E. please let us show results of some pattern.

### 方法
- Step 8 までは「各遺伝子で上位 X 個」を選んでいた。今回は、**合成スコア S がしきい値以上の辺はすべて**選ぶ（個数の上限なし）。標的から1ホップずつ上流へ進む手順は同じ。
- 合成スコア S は、`weight`（データベース内の順位をもとにした値）と `E`（論文数と SIGNOR のスコアをそのまま使った値）を合わせたもの。4通りの合わせ方を試す。

| 名前 | 式 | 意味 |
|---|---|---|
| `geo` | √(weight × E) | 幾何平均。どちらかが低いと下がる（バランス型） |
| `mean` | (weight + E) / 2 | 算術平均。片方が高ければ残りやすい（ゆるい） |
| `min` | min(weight, E) | 小さいほう。**両方**が高いときだけ残る（厳しい） |
| `prod` | weight × E | 積。値が小さくなるので、しきい値も低め |

**Note**
- しきい値は、合成スコアごとに値の分布が違う（`prod` は中央値 0.08、`geo` は 0.29）ので、同じ数字でも厳しさが違う。そこで、ネットワークの大きさが 800〜2,400 辺の範囲になるように、まず AKT1 で試して選んだ（`THRESHOLD_PATTERNS`）。PIK3R1 → AKT1 の辺のスコアは `geo` 0.29・`mean` 0.33・`prod` 0.08・`min` 0.18 なので、PI3K の経路を拾うには、それより少し低いしきい値（`geo` 0.2〜0.25、`mean` 0.3、`prod` 0.08）も試す。
- 評価は Step 9 と同じ。基準は KDR のシグナル、既知の調節因子の点検は結果の評価だけに使う。
- しきい値方式は、証拠の多い遺伝子（ハブ）に入る辺が多いと、選ばれる辺がいくらでも増えうる。上限がないので、大きさの確認が重要。

## 10a. 合成スコアを作り、しきい値で選ぶ
- 動作: `ev` の各辺に4種類の合成スコアを付け、`THRESHOLD_PATTERNS` の各組について、標的から1ホップずつ、スコアがしきい値以上の辺をすべて選ぶ（前のホップまでに選んだ遺伝子は除く）
- 出力: 合成スコアの分布（四分位）と、パターンごとの選んだ辺の数

In [ ]:
# 合成スコアを作る
ev["geo"] = np.sqrt(ev.weight * ev.E)
ev["mean"] = (ev.weight + ev.E) / 2
ev["min"] = ev[["weight", "E"]].min(axis=1)
ev["prod"] = ev.weight * ev.E
display(ev[["weight", "E", "geo", "mean", "min", "prod"]].quantile([.25, .5, .75, .9, .95]).round(3))

def select_by_threshold(df, col, th):
    # 標的から1ホップずつ、col が th 以上の入ってくる辺をすべて選ぶ（個数の上限なし）
    inc = {v: g for v, g in df[df[col] >= th].groupby("dst")}
    reached, kept, frontier = {TARGET: 0}, [], [TARGET]
    for h in range(1, UP_HOPS + 1):
        new = []
        for v in frontier:
            if v not in inc: continue
            for r in inc[v][~inc[v].src.isin(reached)].itertuples():
                kept.append(dict(hop=h, gene=r.src, to=v, effect=r.effect, E=r.E, weight=r.weight))
                if r.src not in new: new.append(r.src)
        for g in new: reached[g] = h
        frontier = [g for g in new if role(g) != "リガンド"]
    return pd.DataFrame(kept)

thr_patterns = {f"{col} >= {th}": select_by_threshold(ev, col, th) for col, th in THRESHOLD_PATTERNS}
pd.Series({k: len(v) for k, v in thr_patterns.items()}, name="選んだ辺の数").to_frame()

## 10b. パターンの比較（KDR を基準に）
- 動作: Step 9 と同じ指標で比べる。(1) 辺・遺伝子・リガンドの数、(2) KDR と VEGF 系リガンドの hop、(3) AKT1 の hop 1 の遺伝子数、(4) 既知の調節因子が2ホップ以内に何個あるか、(5) VEGFA から AKT1 への促進する最良経路のスコア
- 出力: 比較表。比較のため、Step 9 の B / C / G も並べる

In [ ]:
# しきい値パターンの比較（Step 9 の B, C, G も並べる）
rows, paths10 = [], {}
allp = {**{k: v for k, v in patterns.items() if k.startswith(("B", "C", "G"))}, **thr_patterns}
for name, kp in allp.items():
    nodes = set(kp.gene) | {TARGET}
    if "hop" in kp.columns:
        hp = kp.groupby("gene").hop.min().to_dict()
    else:                                                     # G は hop を持たないので、最短ステップ数で代用
        Gx = nx.DiGraph(); Gx.add_edges_from(zip(kp.gene, kp.to)); hp = nx.single_source_shortest_path_length(Gx.reverse(copy=False), TARGET)
    res_ = dict(パターン=name, 辺=len(kp), 遺伝子=len(nodes), リガンド=sum(role(g) == "リガンド" for g in nodes),
                hop1=sum(v == 1 for v in hp.values()), **{f"{REF_RECEPTOR} のhop": hp.get(REF_RECEPTOR, "-")},
                **{"VEGF系リガンド（hop）": ", ".join(f"{g}:{hp[g]}" for g in REF_LIGANDS if g in hp) or "-"},
                **{"既知の調節因子（2ホップ以内）": sum(hp.get(g, 99) <= 2 for g in known)},
                **{"2ホップ以内にない既知の因子": ", ".join(g for g in known if hp.get(g, 99) > 2) or "-"})
    b = best_in(kp, "VEGFA")
    for s, lab in [(1, "促進"), (-1, "抑制")]:
        if (TARGET, s) in b: paths10[(name, lab)] = (round(b[(TARGET, s)][0], 3), " > ".join(b[(TARGET, s)][1]))
    res_["VEGFA→AKT1 促進のスコア"] = paths10[(name, "促進")][0] if (name, "促進") in paths10 else "なし"
    rows.append(res_)
pd.DataFrame(rows).set_index("パターン")

## 10c. AKT1 の hop 1 と、VEGFA の経路（パターンごと）
- 動作: パターンごとに、AKT1 に直接入る遺伝子（hop 1）と、VEGFA から AKT1 への促進する最良経路を並べる
- 出力: パターンごとの hop 1 の遺伝子リスト、VEGFA の経路の表

In [ ]:
# パターンごとの hop 1 の遺伝子と VEGFA の経路
for name, kp in thr_patterns.items():
    g1 = kp[kp.hop == 1].sort_values("E", ascending=False)
    print(f"[{name}] hop 1 = {len(g1)}: " + ", ".join(g1.gene))
print()
pd.DataFrame([dict(パターン=k[0], 符号=k[1], スコア=v[0], 経路=v[1]) for k, v in paths10.items()]).set_index(["パターン", "符号"])

## Step 11. 図: `geo >= 0.25` のパターン（AKT1、KDR のシグナル）
> 指示: Draw figure

- パターン: 合成スコア √(weight × E) が 0.25 以上の辺を、標的から1ホップずつ選んだもの（Step 10）。
- **図1（KDR のシグナル）**: VEGFA・VEGFC → KDR、KDR に入る調節因子（SRC・PTPN6・PTPN1）、KDR から出る枝（PIK3R1・PIK3CA・PLCG1…）と、各枝から AKT1 までの最良経路。
- **図2（全体）**: 経路スコア上位のリガンド + VEGFA・VEGFC。
- 辺の色は符号（促進 = 濃い青、抑制 = 赤の破線）。図の文字は英語。

**Note**
- 描く辺は、選ばれたネットワークの中の最良経路（`E` の積が最大）に限る。KDR から出る枝は、そのうち AKT1 まで届くものだけ。
- 列は AKT1 までの最短ステップ数。列が1つずつ進まない追加の辺は描かない（描かなかった本数は表示する）。

## 11a. 図を描くパターンの辺の表
- 動作: `FIG_PATTERN` で選んだ辺から、描画用の表（各遺伝子の次の遺伝子など）を作る
- 出力: 辺の数、hop ごとの遺伝子の数

In [ ]:
# 図を描くパターンの辺の表
kp = thr_patterns[FIG_PATTERN]
fig_tbl = kp.sort_values("E").groupby("gene").tail(1).rename(columns={"to": "next_gene"})      # 各遺伝子について E が最大の辺を「次の遺伝子」にする
fig_tbl["role"] = fig_tbl.gene.map(role)
print(FIG_PATTERN, "| 辺", len(kp), "| 遺伝子", len(fig_tbl), "| KDR の hop:", fig_tbl[fig_tbl.gene == REF_RECEPTOR].hop.tolist())
fig_tbl.groupby("hop").size().to_frame("遺伝子数").T

## 11b. 図1: KDR のシグナル
- 動作: VEGF 系リガンド → KDR、KDR に入る調節因子、KDR から出る枝と各枝から AKT1 までの最良経路を描く
- 出力: 図（画面と PNG に保存）

In [ ]:
# 図1: KDR のシグナル
extra = set()
for r in kp[kp.to == REF_RECEPTOR].itertuples(): extra.add((r.gene, REF_RECEPTOR))                  # KDR に入る辺（リガンド・調節因子）
for r in kp[kp.gene == REF_RECEPTOR].itertuples():                                                   # KDR から出る枝
    extra.add((REF_RECEPTOR, r.to))
    b = best_in(kp, r.to)                                                                             # その枝から AKT1 までの促進する最良経路
    if (TARGET, 1) in b:
        pth_ = b[(TARGET, 1)][1]; extra.update(zip(pth_[:-1], pth_[1:]))
draw_upstream(fig_tbl, "E", f"{TARGET} upstream: {REF_RECEPTOR} signaling ({FIG_PATTERN} on sqrt(weight x E))", f"network_development_{TARGET}_upstream_geo025_KDR.png",
              ligands=[g for g in REF_LIGANDS if g in set(fig_tbl.gene)], extra_edges=extra)

## 11c. 図2: 全体（経路スコア上位のリガンド + VEGF 系）
- 動作: 経路スコア上位 `N_LIGANDS` 個のリガンドに、VEGFA・VEGFC を加えて、それぞれの最良経路を描く
- 出力: 図（画面と PNG に保存）

In [ ]:
# 図2: 全体
sc2 = {TARGET: 1.0}
for r in fig_tbl.sort_values("hop").itertuples(): sc2[r.gene] = r.E * sc2[r.next_gene]
top_l = fig_tbl[fig_tbl.role == "リガンド"].assign(s=lambda d: d.gene.map(sc2)).sort_values("s", ascending=False).head(N_LIGANDS).gene.tolist()
draw_upstream(fig_tbl, "E", f"{TARGET} upstream: top {N_LIGANDS} ligands + VEGFs ({FIG_PATTERN} on sqrt(weight x E))", f"network_development_{TARGET}_upstream_geo025_overview.png",
              ligands=sorted(set(top_l) | {g for g in REF_LIGANDS if g in set(fig_tbl.gene)}))

## Step 12. 1,439 遺伝子をすべて描く（`geo >= 0.25`、AKT1 の上流）
> 指示: please try to visualize 1439 genes

### 描き方（放射状の木）
- 中心が AKT1。同心円が hop 1〜6（外へ行くほど上流）。
- 各遺伝子は、「次の遺伝子」（標的に向かう辺のうち `E` が最大のもの）の外側に置く。**1,438 本の木の辺**を描く。木に入らない、選ばれたネットワークの残りの辺（約 960 本）は薄く描く。
- 角度は、枝ごとの葉（一番外側の遺伝子）の数に、最低限の幅（8 個分）を足した値に比例して割り当てる。同じ枝の遺伝子が近くに並び、葉の少ない遺伝子も1か所に固まらない。
- 色: リガンド = 青、受容体 = 緑がかった灰色、その他 = 薄い灰色、AKT1 = 濃い灰色。辺の色は符号（促進 = 濃い青、抑制 = 赤）。
- 図は2つ: **静止画 PNG**（印刷用）と、**ホバーで遺伝子名が見える対話式の図 HTML**（拡大できる）。

**Note**
- 1,439 個すべてに名前を付けると読めないので、PNG には「hop 1、リガンド、KDR の経路」だけ名前を付けた。対話式の図では、どの点でもポイントすると名前・hop・役割・次の遺伝子が出る。
- 木の辺（各遺伝子から次の遺伝子への1本）は「最良の1本」なので、遺伝子が複数の経路で AKT1 につながることは、薄い辺でしか見えない。

## 12a. 放射状の配置を計算する
- 動作: 11a の表 `fig_tbl`（各遺伝子の次の遺伝子）から木を作り、枝ごとの葉の数に比例した角度と、hop に応じた半径で座標を決める
- 出力: hop ごとの遺伝子数と役割、座標の表（先頭5行）

In [ ]:
# 放射状の木の座標を計算する
from collections import defaultdict
import math

children = defaultdict(list)
for r in fig_tbl.itertuples(): children[r.next_gene].append(r.gene)      # 次の遺伝子 -> それに入る遺伝子

leaves = {}
def count_leaves(n):
    leaves[n] = sum(count_leaves(c) for c in children[n]) or 1               # 葉（一番外側の遺伝子）の数
    return leaves[n]
count_leaves(TARGET)

MIN_SLOT = 8                                                                 # 各遺伝子に最低限割り当てる葉の数（角度の幅）
angle = {}
def place(n, a0, a1):
    angle[n] = (a0 + a1) / 2
    a = a0
    tot = sum(leaves[k] + MIN_SLOT for k in children[n])                       # 葉の数 + 最低限の幅（葉が少ない遺伝子が1か所に固まらないように）
    for c in sorted(children[n], key=lambda c: (-leaves[c], c)):             # 大きい枝から順に角度を割り当てる
        w = (a1 - a0) * (leaves[c] + MIN_SLOT) / tot
        place(c, a, a + w); a += w
place(TARGET, 0, 2 * math.pi)

RADIUS = {0: 0, 1: 1.0, 2: 2.6, 3: 4.4, 4: 6.0, 5: 7.4, 6: 8.6}              # hop ごとの半径（外側ほど詰まるので間隔を広げる）
hop_of = dict(zip(fig_tbl.gene, fig_tbl.hop)); hop_of[TARGET] = 0
lay = pd.DataFrame({"gene": list(hop_of), "hop": list(hop_of.values())})
lay["role"] = lay.gene.map(lambda g: "AKT1" if g == TARGET else role(g))
lay["angle"] = lay.gene.map(angle)
lay["x"] = lay.hop.map(RADIUS) * np.cos(lay.angle); lay["y"] = lay.hop.map(RADIUS) * np.sin(lay.angle)
lay["next_gene"] = lay.gene.map(dict(zip(fig_tbl.gene, fig_tbl.next_gene))).fillna("")
print("遺伝子:", len(lay))
display(lay.groupby(["hop", "role"]).size().unstack(fill_value=0))
lay.head()

## 12b. 静止画（PNG）
- 動作: 木の辺を促進（緑）・抑制（赤）で、木に入らない辺を薄い灰色で描き、遺伝子を役割の色の点で描く。名前は hop 1、リガンド、KDR の経路にだけ付ける
- 出力: 図（画面と PNG に保存）

In [ ]:
# 全遺伝子の放射状の図（静止画）
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

xy = dict(zip(lay.gene, zip(lay.x, lay.y)))
tree = set(zip(fig_tbl.gene, fig_tbl.next_gene))
other = [(a, b) for a, b in zip(kp.gene, kp.to) if (a, b) not in tree]                       # 木に入らない選ばれた辺
eff_t = {(r.gene, r.next_gene): r.effect for r in fig_tbl.itertuples()}

fig, ax = plt.subplots(figsize=(18, 18))
ax.add_collection(LineCollection([[xy[a], xy[b]] for a, b in other], colors="#dddddd", linewidths=0.3, zorder=1))
ax.add_collection(LineCollection([[xy[a], xy[b]] for (a, b), e_ in eff_t.items() if e_ != 1], colors="#b5655c", linewidths=0.6, zorder=2))   # 抑制・不明
ax.add_collection(LineCollection([[xy[a], xy[b]] for (a, b), e_ in eff_t.items() if e_ == 1], colors="#9ab8a3", linewidths=0.5, zorder=2))    # 促進
colour = {"AKT1": "#444444", "リガンド": "#5b7a99", "受容体": "#7a9a8a", "途中": "#bdbdbd"}
for rl, g in lay.groupby("role"):
    ax.scatter(g.x, g.y, s=60 if rl == "AKT1" else 14, c=colour[rl], zorder=3, linewidths=0)
for h, rr in RADIUS.items():                                                                   # hop の輪
    if h: ax.add_patch(plt.Circle((0, 0), rr, fill=False, ec="#eeeeee", lw=0.6, zorder=0)); ax.text(0, rr + 0.08, f"hop {h}", ha="center", fontsize=8, color="#888888")

label = set(lay[lay.hop == 1].gene) | set(lay[lay.role == "リガンド"].gene) | set(REF_LIGANDS) | {REF_RECEPTOR, TARGET}
for r in lay[lay.gene.isin(label)].itertuples():                                               # 名前は外向きに回転して付ける
    deg = math.degrees(r.angle); flip = 90 < deg % 360 < 270
    ax.text(r.x * (1.02 if r.hop else 1), r.y * (1.02 if r.hop else 1) + (0.2 if r.gene == TARGET else 0), r.gene, fontsize=4.5 if r.role == "リガンド" else 6, ha="right" if flip else "left", va="center",
            rotation=deg + 180 if flip else deg, rotation_mode="anchor", color="#222222" if r.hop <= 1 else "#46627f", zorder=4)
for g_ in [REF_RECEPTOR, "VEGFA", "VEGFC", "PIK3R1", "PIK3CA"]:                                 # KDR の経路を目立たせる
    if g_ in xy: ax.scatter(*xy[g_], s=70, c="none", edgecolors="#d08c2f", linewidths=1.6, zorder=5)
ax.set_xlim(-10.5, 10.5); ax.set_ylim(-10.5, 10.5); ax.set_aspect("equal"); ax.axis("off")
ax.set_title(f"{TARGET} upstream: all {len(lay)} genes selected by {FIG_PATTERN} (radial tree; rings = hop; orange rings = KDR signaling)", fontsize=12)
ax.legend(handles=[Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=8, label=l) for l, c in [("ligand", "#5b7a99"), ("receptor", "#7a9a8a"), ("other protein", "#bdbdbd")]]
          + [Line2D([0], [0], color="#9ab8a3", label="activation (best edge)"), Line2D([0], [0], color="#b5655c", label="inhibition (best edge)"), Line2D([0], [0], color="#dddddd", label="other selected edges")],
          loc="lower left", frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f"network_development_{TARGET}_upstream_all.png", dpi=150); plt.show()

## 12c. 対話式の図（HTML）
- 動作: 同じ配置を Plotly で描く。点にポイントすると、遺伝子名・hop・役割・次の遺伝子・その辺のスコアが出る。拡大・移動ができる
- 出力: 画面の図と、HTML ファイル（`network_development_AKT1_upstream_all.html`）

In [ ]:
# 全遺伝子の放射状の図（対話式）
import plotly.graph_objects as go

def seg(pairs):
    xs, ys = [], []
    for a, b in pairs: xs += [xy[a][0], xy[b][0], None]; ys += [xy[a][1], xy[b][1], None]
    return xs, ys

fig = go.Figure()
for name, pairs, colr, w in [("other selected edges", other, "#e3e3e3", 0.4),
                             ("inhibition / unknown (best edge)", [k for k, e_ in eff_t.items() if e_ != 1], "#b5655c", 0.8),
                             ("activation (best edge)", [k for k, e_ in eff_t.items() if e_ == 1], "#9ab8a3", 0.7)]:
    xs, ys = seg(pairs); fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=colr, width=w), name=name, hoverinfo="skip"))
Emap = dict(zip(fig_tbl.gene, fig_tbl.E))
for rl, g in lay.groupby("role"):
    fig.add_trace(go.Scatter(x=g.x, y=g.y, mode="markers+text", name={"AKT1": "AKT1", "リガンド": "ligand", "受容体": "receptor", "途中": "other protein"}[rl],
                             marker=dict(size=14 if rl == "AKT1" else 6, color=colour[rl]),
                             text=[n if (h <= 1 or n in REF_LIGANDS or n == REF_RECEPTOR) else "" for n, h in zip(g.gene, g.hop)], textposition="top center", textfont=dict(size=9),
                             customdata=np.stack([g.hop, g.next_gene, g.gene.map(Emap).round(3).fillna("")], axis=1),
                             hovertemplate="<b>%{hovertext}</b><br>hop %{customdata[0]}<br>next: %{customdata[1]}<br>edge E: %{customdata[2]}<extra></extra>", hovertext=g.gene))
fig.update_layout(title=f"{TARGET} upstream: all {len(lay)} genes selected by {FIG_PATTERN} (rings = hop 1-6)", width=1000, height=1000, plot_bgcolor="white",
                  xaxis=dict(visible=False, scaleanchor="y"), yaxis=dict(visible=False), legend=dict(orientation="h", y=-0.02))
fig.write_html(f"network_development_{TARGET}_upstream_all.html", include_plotlyjs="cdn")
fig.show()

## Step 13. 上位 100 遺伝子の選び方を、受容体 → 標的の経路にする
> 指示: you dont need to include ligands in the selection; important path is from receptor to the target, or target to TF (in the case of downstream)

### 変更点
- 選ぶときの出発点を**リガンドから受容体に変える**。リガンドは選ぶ対象にも、流れの計算にも入れない（注釈として、各受容体に結合するリガンドの数だけ表に付ける）。
- **流れ（flow）** = 各受容体について、その受容体から AKT1 までの最良経路（辺の `E` の積 = 経路スコア）を求め、その経路が通る遺伝子すべてに、経路スコアを足していく。受容体どうしの経路が重なる遺伝子ほど、流れが大きい。
- 流れの大きい順に上位 `TOP_N` 個を選ぶ。遺伝子の流れは、その上流（受容体に近い側）の遺伝子の流れ以上になるので、選んだ遺伝子は AKT1 までつながる。
- 下流（標的 → TF）の場合は、同じ方法で「出発点 = 標的、終点 = TF」にする（今回は上流だけ）。

**Note**
- 受容体の上流にあるタンパク質（受容体を調節する因子など）は、「受容体から標的まで」の経路の外なので、選ばない。

## 13a. 受容体 → AKT1 の流れで遺伝子を選ぶ
- 動作: 受容体ごとに AKT1 までの最良経路をたどり、通る遺伝子に経路スコアを足して流れを求める。上位 `TOP_N` 個を選ぶ
- 出力: 選んだ遺伝子の hop・役割の内訳、AKT1 につながっているかの確認、流れの上位 25 個

In [ ]:
# 受容体 -> AKT1 の流れ（flow）を計算して上位 TOP_N 個を選ぶ
nxt13 = dict(zip(fig_tbl.gene, fig_tbl.next_gene))
ps13 = {TARGET: 1.0}
for r in fig_tbl.sort_values("hop").itertuples(): ps13[r.gene] = r.E * ps13[r.next_gene]        # 経路スコア（E の積）

receptors_all = [g for g in fig_tbl.gene if role(g) == "受容体"]
flow, n_rec = {}, {}
for rc in receptors_all:                                                                         # 受容体ごとに AKT1 までの経路をたどる
    g = rc
    while True:
        flow[g] = flow.get(g, 0) + ps13[rc]; n_rec[g] = n_rec.get(g, 0) + 1
        if g == TARGET: break
        g = nxt13[g]

lig_per_rec = ev[(ev.geo >= 0.25) & (ev.src.map(role) == "リガンド")].groupby("dst").size()           # 注釈: 各受容体に結合するリガンドの数（geo >= 0.25 の辺すべて）
flow13 = pd.DataFrame({"flow": pd.Series(flow), "n_receptors": pd.Series(n_rec)}).sort_values("flow", ascending=False)
flow13["role"] = ["AKT1" if g == TARGET else role(g) for g in flow13.index]
flow13["hop"] = [0 if g == TARGET else int(hop_of[g]) for g in flow13.index]
flow13["n_ligands"] = [int(lig_per_rec.get(g, 0)) if role(g) == "受容体" else "" for g in flow13.index]
top13 = flow13.head(TOP_N)

print(f"受容体 {len(receptors_all)} 個の経路が通る遺伝子: {len(flow13)} 個 -> 上位 {TOP_N} 個を選ぶ")
print("AKT1 までつながっているか（親がすべて選んだ集合に入る）:", all(nxt13[g] in top13.index for g in top13.index if g != TARGET))
print("KDR・PIK3R1・PIK3CA が入っているか:", {g: g in top13.index for g in ["KDR", "PIK3R1", "PIK3CA"]})
display(top13.groupby(["hop", "role"]).size().unstack(fill_value=0))
top13.head(25)

## 13b. 図（受容体 → AKT1、上位 100 遺伝子）
- 動作: 選んだ遺伝子だけを、AKT1 からのステップ数の列に並べて描く。出発点は受容体（緑）。辺は最良経路の辺（促進 = 濃い青、抑制 = 赤の破線）
- 出力: 図（画面と PNG に保存）と、選んだ遺伝子の表（CSV）

In [ ]:
# 選んだ上位 TOP_N 個の図（受容体 -> AKT1）
keep13 = set(top13.index)
tbl13 = fig_tbl[fig_tbl.gene.isin(keep13)]
tree13 = [(r.gene, r.next_gene) for r in tbl13.itertuples() if r.next_gene in keep13]            # 選んだ遺伝子どうしの最良の辺
draw_upstream(tbl13, "E", f"{TARGET} upstream: top {TOP_N} genes on receptor-to-{TARGET} paths (ranked by path flow; ligands not included)",
              f"network_development_{TARGET}_upstream_top{TOP_N}.png", ligands=[g for g in keep13 if role(g) == "受容体"], extra_edges=tree13)
top13.assign(next_gene=[nxt13.get(g, "") for g in top13.index]).to_csv(f"network_development_{TARGET}_upstream_top{TOP_N}.csv")

## Step 14. 図の並びを「リガンド → 受容体 → 途中のタンパク質 → 標的」の順にする
> 指示: the figure start from ligands, receptors, intermediates and targets; from left to right.

### 変更点（Step 13 の上位 100 遺伝子の図）
- 列の並びを、AKT1 からのステップ数ではなく、**役割の順**にする: 左から **リガンド | 受容体 | 途中のタンパク質 | 標的（AKT1）**。
- 途中のタンパク質は、標的に近いほど右に置く（AKT1 からのステップ数ごとに列を分ける）。受容体は、ステップ数にかかわらず全部1列にそろえる。
- 選ぶ 100 遺伝子は Step 13 のまま（リガンドは数に入れない）。**リガンドは、選んだ受容体に結合するものを、図に添えるだけ**。受容体1つにつき、辺のスコア `E` が大きい順に `MAX_LIG_PER_REC` 個まで（`geo ≥ 0.25` の辺のみ）。
- 同じ列の中の順序は、Step 12 の木の並び（`angle`）に合わせる。同じ枝の遺伝子が近くに並び、辺の交差が減る。

**Note**
- 受容体から途中のタンパク質への辺は、列をまたぐので長くなる（色は符号: 促進 = 濃い青、抑制 = 赤の破線）。
- 添えるリガンドを `MAX_LIG_PER_REC` で絞っているので、リガンドの数は、その受容体に結合するリガンドの全部ではない（表の `n_ligands` が全部の数）。

## 14a. 列ごとの遺伝子と辺を用意する
- 動作: 13 で選んだ 100 遺伝子を役割ごとの列に分け、選んだ受容体に添えるリガンドを選ぶ。各遺伝子の縦の位置は、木の並び（`angle`）で決める
- 出力: 列ごとの遺伝子数、添えたリガンドの数

In [ ]:
# 役割ごとの列に分けて、縦の位置を決める
rc13 = [g for g in keep13 if role(g) == "受容体"]
lig_e = ev[(ev.geo >= 0.25) & (ev.src.map(role) == "リガンド") & ev.dst.isin(rc13)].sort_values("E", ascending=False)
lig_e = lig_e.groupby("dst").head(MAX_LIG_PER_REC)                                              # 受容体ごとに E が大きい順
lig13 = sorted(set(lig_e.src))

int_hops = sorted({int(hop_of[g]) for g in keep13 if role(g) == "途中" and g != TARGET}, reverse=True)         # 途中のタンパク質の列（標的から遠い順）
columns = [("ligand", lig13), ("receptor", rc13)] + [(f"intermediate (hop {h})", [g for g in keep13 if role(g) == "途中" and g != TARGET and hop_of[g] == h]) for h in int_hops] + [("target", [TARGET])]
ang = {g: angle.get(g, 0) for g in keep13}
for g in lig13:                                                                                  # リガンドは、結合する受容体の角度の平均
    ang[g] = np.mean([angle[r] for r in lig_e[lig_e.src == g].dst])
ypos14 = {}
for ci, (name, genes) in enumerate(columns):
    genes = sorted(genes, key=lambda g: (ang.get(g, 0), g))
    for i, g in enumerate(genes): ypos14[g] = (len(genes) - 1) / 2 - i                           # 列の中央を 0 にして上から並べる
xpos14 = {g: ci for ci, (name, genes) in enumerate(columns) for g in genes}
print("列ごとの遺伝子数:", {name: len(genes) for name, genes in columns})
print(f"添えたリガンド: {len(lig13)} 個（受容体 {len(rc13)} 個に、1つあたり最大 {MAX_LIG_PER_REC} 個）")

## 14b. 図を描く
- 動作: 左から リガンド | 受容体 | 途中のタンパク質 | AKT1 の順に並べて描く。辺は、リガンド → 受容体、そして Step 13 の最良の辺（受容体 → … → AKT1）
- 出力: 図（画面と PNG に保存）

In [ ]:
# リガンド -> 受容体 -> 途中のタンパク質 -> 標的 の図
edges14 = [(r.src, r.dst, r.effect) for r in lig_e.itertuples()] + [(r.gene, r.next_gene, r.effect) for r in tbl13.itertuples() if r.next_gene in keep13]
colour14 = lambda g: "#6b6b6b" if g == TARGET else "#5b7a99" if role(g) == "リガンド" else "#7a9a8a" if role(g) == "受容体" else "#d0d0d0"
nmax = max(len(genes) for name, genes in columns)
fig, ax = plt.subplots(figsize=(22, max(8, 0.34 * nmax + 2)))
for a, b, e_ in edges14:
    ax.annotate("", xy=(xpos14[b], ypos14[b]), xytext=(xpos14[a], ypos14[a]),
                arrowprops=dict(arrowstyle="->", color="#5a8f6a" if e_ == 1 else "#b5655c", lw=0.7, linestyle="-" if e_ == 1 else "--", shrinkA=16, shrinkB=16, alpha=0.7,
                                connectionstyle="arc3,rad=0.03"))
for g in xpos14:
    ax.text(xpos14[g], ypos14[g], g, ha="center", va="center", fontsize=7, color="white" if colour14(g) != "#d0d0d0" else "black",
            bbox=dict(boxstyle="round,pad=0.25", fc=colour14(g), ec="none"), zorder=3)
ax.set_xlim(-0.6, len(columns) - 0.4); ax.set_xticks(range(len(columns))); ax.set_xticklabels([f"{n}\n({len(g_)})" for n, g_ in columns], fontsize=9)
ys = list(ypos14.values()); ax.set_ylim(min(ys) - 1.5, max(ys) + 1.5); ax.set_yticks([]); ax.spines[["top", "right", "left"]].set_visible(False)
ax.set_title(f"{TARGET} upstream: ligand > receptor > intermediate > target (top {TOP_N} genes on receptor-to-{TARGET} paths; up to {MAX_LIG_PER_REC} ligands per receptor shown)", fontsize=12)
ax.legend(handles=[Line2D([0], [0], marker="s", color="w", markerfacecolor=c, markersize=10, label=l) for l, c in [("ligand (shown, not counted)", "#5b7a99"), ("receptor", "#7a9a8a"), ("intermediate protein", "#d0d0d0")]]
          + [Line2D([0], [0], color="#5a8f6a", label="activation"), Line2D([0], [0], color="#b5655c", linestyle="--", label="inhibition")], loc="upper left", frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f"network_development_{TARGET}_upstream_top{TOP_N}_layers.png", dpi=130); plt.show()

## Step 15. 影響の大きい受容体とリガンドに絞り、途中のタンパク質を多く見せる
> 指示: please select most impactful receptors and ligands for the visualization and show more genes in the middle

### 選び方
- **受容体の影響** = その受容体から AKT1 までの最良経路のスコア（辺の `E` の積）。大きい順に上位 `N_REC` 個。
- **リガンドの影響** = そのリガンドから選んだ受容体への辺の `E` × その受容体の影響（つまり、リガンドから AKT1 までの経路スコア）。大きい順に上位 `N_LIG` 個。
- **途中のタンパク質**: 選んだ受容体の最良経路に載る遺伝子は必ず入れ、さらに、Step 13 の流れ（全受容体の経路が通る量）が大きい順に追加して、合計 `N_MID` 個まで見せる。
- **辺**: 最良経路の辺だけでなく、`geo ≥ 0.25` の辺のうち、各遺伝子から標的の側へ向かうものを `MAX_OUT` 本まで（`E` が大きい順）描く。途中のタンパク質どうしのつながり（複数の経路）が見える。
- 列の並びは Step 14 と同じ（リガンド | 受容体 | 途中のタンパク質（標的に近いほど右）| 標的）。

**Note**
- 「影響」は経路スコア（辺のスコア `E` の積）で測っている。標的に近く、証拠の強い辺だけでつながる受容体・リガンドが上位に来る（RET や PDGFR など）。長い経路の受容体は、実際に重要でも下位になりうる。
- 追加した途中のタンパク質は、選んだ受容体とは別の経路で標的につながるものも含む。

## 15a. 影響の大きい受容体とリガンドを選ぶ
- 動作: 受容体と、選んだ受容体に結合するリガンドについて、標的への経路スコアを計算し、上位を選ぶ
- 出力: 選んだ受容体とリガンドの表（スコア・経路）

In [ ]:
# 影響の大きい受容体とリガンドを選ぶ
rec_score = {g: ps13[g] for g in receptors_all}                                         # 受容体の影響 = 標的までの経路スコア
sel_rec = sorted(rec_score, key=rec_score.get, reverse=True)[:N_REC]

lig_c = ev[(ev.geo >= 0.25) & (ev.src.map(role) == "リガンド") & ev.dst.isin(sel_rec)].copy()
lig_c["impact"] = lig_c.E * lig_c.dst.map(rec_score)                                    # リガンドの影響 = 辺の E × 受容体の影響
lig_best = lig_c.sort_values("impact", ascending=False).drop_duplicates("src").head(N_LIG)
sel_lig = list(lig_best.src)

def path_to_target(g):
    pth = [g]
    while pth[-1] != TARGET: pth.append(nxt13[pth[-1]])
    return pth
tab_r = pd.DataFrame({"receptor": sel_rec, "path_score": [round(rec_score[g], 3) for g in sel_rec], "path": [" > ".join(path_to_target(g)) for g in sel_rec]})
tab_l = pd.DataFrame({"ligand": lig_best.src.values, "receptor": lig_best.dst.values, "impact": lig_best.impact.round(3).values})
display(tab_r)
tab_l

## 15b. 途中のタンパク質を選び、辺を用意する
- 動作: 選んだ受容体の最良経路に載る遺伝子を入れ、流れの大きい順に追加して `N_MID` 個にする。選んだ遺伝子どうしの `geo ≥ 0.25` の辺から、標的の側へ向かうものを各遺伝子 `MAX_OUT` 本まで選ぶ
- 出力: 列ごとの遺伝子数と、辺の数

In [ ]:
# 途中のタンパク質と辺を選ぶ
on_path = set()
for g in sel_rec: on_path.update(path_to_target(g))
mid = [g for g in on_path if g not in sel_rec and g != TARGET]                                  # 選んだ受容体の経路上の遺伝子（必ず入れる）
extra = [g for g in flow13.index if role(g) == "途中" and g != TARGET and g not in on_path]      # 流れの大きい順に追加
mid += extra[:max(0, N_MID - len(mid))]

shown = set(sel_rec) | set(mid) | {TARGET}
cand = ev[(ev.geo >= 0.25) & ev.src.isin(set(sel_rec) | set(mid)) & ev.dst.isin(set(mid) | {TARGET}) & (ev.src != ev.dst)].copy()
ok = cand.src.isin(sel_rec) | (cand.src.map(hop_of) > cand.dst.map(hop_of))                      # 受容体からは自由、途中のタンパク質は標的に近づく向きだけ
cand = cand[ok].sort_values("E", ascending=False).groupby("src").head(MAX_OUT)
tree15 = [(g, nxt13[g]) for g in on_path if g != TARGET]                                          # 最良経路の辺は必ず入れる
eff_lookup = ev.set_index(["src", "dst"]).effect
edges15 = {(r.src, r.dst, r.effect) for r in cand.itertuples()} | {(a, b, eff_lookup[(a, b)]) for a, b in tree15}
lig_e15 = lig_c[lig_c.src.isin(sel_lig)]
edges15 |= {(r.src, r.dst, r.effect) for r in lig_e15.itertuples()}

int_h = sorted({int(hop_of[g]) for g in mid}, reverse=True)
cols15 = [("ligand", sel_lig), ("receptor", sel_rec)] + [(f"intermediate (hop {h})", [g for g in mid if hop_of[g] == h]) for h in int_h] + [("target", [TARGET])]
print("列ごとの遺伝子数:", {n: len(g_) for n, g_ in cols15})
print(f"辺: {len(edges15)} 本（リガンド→受容体 {len(lig_e15)}、最良経路 {len(tree15)}、そのほか {len(edges15) - len(lig_e15) - len(tree15)}）")

## 15c. 図を描く
- 動作: 左から リガンド | 受容体 | 途中のタンパク質 | AKT1 に並べて描く。同じ列の中の順序は木の並び（`angle`）
- 出力: 図（画面と PNG に保存）

In [ ]:
# リガンド -> 受容体 -> 途中のタンパク質 -> 標的 の図（影響の大きいものだけ）
ang15 = {g: angle.get(g, 0) for g in shown}
for g in sel_lig: ang15[g] = np.mean([angle[r] for r in lig_e15[lig_e15.src == g].dst])
yp15 = {}
for name, genes in cols15:
    for i, g in enumerate(sorted(genes, key=lambda g: (ang15.get(g, 0), g))): yp15[g] = (len(genes) - 1) / 2 - i
xp15 = {g: ci for ci, (name, genes) in enumerate(cols15) for g in genes}
colr = lambda g: "#6b6b6b" if g == TARGET else "#5b7a99" if role(g) == "リガンド" else "#7a9a8a" if role(g) == "受容体" else "#d0d0d0"
nmax = max(len(genes) for name, genes in cols15)
fig, ax = plt.subplots(figsize=(20, max(8, 0.42 * nmax + 2)))
for a, b, e_ in edges15:
    ax.annotate("", xy=(xp15[b], yp15[b]), xytext=(xp15[a], yp15[a]),
                arrowprops=dict(arrowstyle="->", color="#5a8f6a" if e_ == 1 else "#b5655c", lw=0.8, linestyle="-" if e_ == 1 else "--", shrinkA=16, shrinkB=16, alpha=0.75, connectionstyle="arc3,rad=0.03"))
for g in xp15:
    ax.text(xp15[g], yp15[g], g, ha="center", va="center", fontsize=8, color="white" if colr(g) != "#d0d0d0" else "black",
            bbox=dict(boxstyle="round,pad=0.28", fc=colr(g), ec="none"), zorder=3)
ax.set_xlim(-0.6, len(cols15) - 0.4); ax.set_xticks(range(len(cols15))); ax.set_xticklabels([f"{n}\n({len(g_)})" for n, g_ in cols15], fontsize=9)
yy = list(yp15.values()); ax.set_ylim(min(yy) - 1.5, max(yy) + 1.5); ax.set_yticks([]); ax.spines[["top", "right", "left"]].set_visible(False)
ax.set_title(f"{TARGET} upstream: top {N_LIG} ligands > top {N_REC} receptors > {len(mid)} intermediate proteins > target (impact = path score to {TARGET})", fontsize=12)
ax.legend(handles=[Line2D([0], [0], marker="s", color="w", markerfacecolor=c, markersize=10, label=l) for l, c in [("ligand", "#5b7a99"), ("receptor", "#7a9a8a"), ("intermediate protein", "#d0d0d0")]]
          + [Line2D([0], [0], color="#5a8f6a", label="activation"), Line2D([0], [0], color="#b5655c", linestyle="--", label="inhibition")], loc="upper left", frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f"network_development_{TARGET}_upstream_impact.png", dpi=130); plt.show()

## Step 16. 受容体の影響: 媒介する遺伝子の流れ（D）に、「直接の辺が最も高い」を加える
> 指示: D looks good. I put high score on direct link, for example: AKT1-KDR is highest but AKT1-X-KDR decreases the impact.

### 影響の定義
1. **媒介する遺伝子の流れ**: 受容体から AKT1 までの最良経路の途中にある遺伝子（受容体と AKT1 は除く）について、Step 13 の流れ（全受容体の経路が通る量）を、最大の流れ（SRC）で割って 0〜1 にし、その平均を取る。**間に遺伝子がない（直接の辺）場合は 1**。
2. **距離の減衰**: 受容体と AKT1 の間に遺伝子が1つ増えるごとに、`DECAY`（既定 0.5）をかける。
3. **影響 = 媒介する遺伝子の流れ × `DECAY` ^ （ステップ数 − 1）**。
4. **同点のときの順位（タイブレーク）**: 影響が同じ受容体は、最良経路のスコア（辺の `E` の積）が大きいほうを上にする。

つまり、直接つながる受容体（例 RET → AKT1）が最高（1.0）。間に1つ入る（KDR → PIK3R1 → AKT1）と半分以下、2つ入ると4分の1以下になる。間に入る遺伝子が、多くの受容体の経路が通るハブ（PIK3R1、SRC）であるほど、その中では高くなる。

**Note**
- 辺の証拠スコア `E` は、影響の大きさには入れず、同点のときの順位づけにだけ使う。直接の辺でも `E` が低いものは、最高点でも証拠が弱い。その辺の `E` も表に出す。
- `DECAY` を変えると、間接的な受容体の順位が変わる（0.7 にすると緩やか、0.3 にすると直接の辺をいっそう優先する）。

## 16a. 受容体の影響を計算する
- 動作: 全 130 個の受容体について、ステップ数・媒介する遺伝子の流れ・影響を計算し、上位 15 個と、Step 15（経路スコア）の上位との違いを見る
- 出力: 上位 15 個の表、基準の受容体（KDR・EGFR・IGF1R・INSR）の順位

In [ ]:
# 受容体の影響 = 媒介する遺伝子の流れ × DECAY^(ステップ数 - 1)
Eg = dict(zip(fig_tbl.gene, fig_tbl.E))                                                        # 各遺伝子から次の遺伝子への辺の E
fmax = flow13[(flow13.index != TARGET)].flow.max()                                              # 流れの最大値（正規化に使う）
rows = []
for rc in receptors_all:
    pth = path_to_target(rc); steps = len(pth) - 1; mids = pth[1:-1]
    mf = 1.0 if not mids else float(np.mean([flow13.loc[m, "flow"] / fmax for m in mids]))      # 直接の辺なら 1
    rows.append(dict(receptor=rc, steps=steps, mediator_flow=round(mf, 3), impact=mf * DECAY ** (steps - 1), path_score=ps13[rc],      # path_score は同点のときの順位づけ（タイブレーク）
                     edge_E_last=round(Eg[pth[-2]], 3), path=" > ".join(pth)))
imp16 = pd.DataFrame(rows).sort_values(["impact", "path_score"], ascending=False).reset_index(drop=True)      # 影響が同じなら、経路スコア（E の積）が大きいほうを上に
imp16["impact"] = imp16.impact.round(4); imp16["path_score"] = imp16.path_score.round(3)
imp16["rank"] = imp16.index + 1
rec_impact16 = dict(zip(imp16.receptor, imp16.impact + 1e-3 * imp16.path_score))        # リガンドの影響の計算でも同点を避ける
print("基準の受容体の順位:", {g: int(imp16[imp16.receptor == g]["rank"].iloc[0]) for g in ["KDR", "EGFR", "IGF1R", "INSR", "RET"]})
print("Step 15（経路スコア）の上位 15 個との重なり:", len(set(imp16.receptor[:15]) & set(sel_rec)), "個")
imp16.head(15)

## 16b. 図（影響の大きい受容体・リガンドと、多くの途中のタンパク質）
- 動作: Step 15 と同じ手順で、新しい受容体の影響で上位 `N_REC` 個の受容体、その受容体に結合する上位 `N_LIG` 個のリガンド（リガンドの影響 = 辺の `E` × 受容体の影響）、途中のタンパク質 `N_MID` 個を選び、同じ並びで描く
- 出力: 図（画面と PNG に保存）

In [ ]:
# 新しい影響で選び直して描く（Step 15 と同じ手順）
Eg = dict(zip(fig_tbl.gene, fig_tbl.E))
sel_rec16 = list(imp16.receptor[:N_REC])
lig_c16 = ev[(ev.geo >= 0.25) & (ev.src.map(role) == "リガンド") & ev.dst.isin(sel_rec16)].copy()
lig_c16["impact"] = lig_c16.E * lig_c16.dst.map(rec_impact16)
sel_lig16 = list(lig_c16.sort_values("impact", ascending=False).drop_duplicates("src").head(N_LIG).src)

on_path16 = set()
for g in sel_rec16: on_path16.update(path_to_target(g))
mid16 = [g for g in on_path16 if g not in sel_rec16 and g != TARGET]
mid16 += [g for g in flow13.index if role(g) == "途中" and g != TARGET and g not in on_path16][:max(0, N_MID - len(mid16))]

cand16 = ev[(ev.geo >= 0.25) & ev.src.isin(set(sel_rec16) | set(mid16)) & ev.dst.isin(set(mid16) | {TARGET}) & (ev.src != ev.dst)].copy()
cand16 = cand16[cand16.src.isin(sel_rec16) | (cand16.src.map(hop_of) > cand16.dst.map(hop_of))].sort_values("E", ascending=False).groupby("src").head(MAX_OUT)
lig_e16 = lig_c16[lig_c16.src.isin(sel_lig16)]
edges16 = ({(r.src, r.dst, r.effect) for r in cand16.itertuples()} | {(g, nxt13[g], eff_lookup[(g, nxt13[g])]) for g in on_path16 if g != TARGET}
           | {(r.src, r.dst, r.effect) for r in lig_e16.itertuples()})
cols16 = [("ligand", sel_lig16), ("receptor", sel_rec16)] + [(f"intermediate (hop {h})", [g for g in mid16 if hop_of[g] == h]) for h in sorted({int(hop_of[g]) for g in mid16}, reverse=True)] + [("target", [TARGET])]

shown16 = set(sel_rec16) | set(mid16) | {TARGET}
ang16 = {g: angle.get(g, 0) for g in shown16}
for g in sel_lig16: ang16[g] = np.mean([angle[r] for r in lig_e16[lig_e16.src == g].dst])
yp16 = {}
for name, genes in cols16:
    for i, g in enumerate(sorted(genes, key=lambda g: (ang16.get(g, 0), g))): yp16[g] = (len(genes) - 1) / 2 - i
xp16 = {g: ci for ci, (name, genes) in enumerate(cols16) for g in genes}
colr16 = lambda g: "#6b6b6b" if g == TARGET else "#5b7a99" if role(g) == "リガンド" else "#7a9a8a" if role(g) == "受容体" else "#d0d0d0"
fig, ax = plt.subplots(figsize=(20, max(8, 0.42 * max(len(g_) for n, g_ in cols16) + 2)))
for a, b, e_ in edges16:
    ax.annotate("", xy=(xp16[b], yp16[b]), xytext=(xp16[a], yp16[a]),
                arrowprops=dict(arrowstyle="->", color="#5a8f6a" if e_ == 1 else "#b5655c", lw=0.8, linestyle="-" if e_ == 1 else "--", shrinkA=16, shrinkB=16, alpha=0.75, connectionstyle="arc3,rad=0.03"))
for g in xp16:
    ax.text(xp16[g], yp16[g], g, ha="center", va="center", fontsize=8, color="white" if colr16(g) != "#d0d0d0" else "black",
            bbox=dict(boxstyle="round,pad=0.28", fc=colr16(g), ec="none"), zorder=3)
ax.set_xlim(-0.6, len(cols16) - 0.4); ax.set_xticks(range(len(cols16))); ax.set_xticklabels([f"{n}\n({len(g_)})" for n, g_ in cols16], fontsize=9)
yy = list(yp16.values()); ax.set_ylim(min(yy) - 1.5, max(yy) + 1.5); ax.set_yticks([]); ax.spines[["top", "right", "left"]].set_visible(False)
ax.set_title(f"{TARGET} upstream: top {N_LIG} ligands > top {N_REC} receptors > {len(mid16)} intermediate proteins > target (receptor impact = mediator flow x {DECAY}^(steps-1))", fontsize=12)
ax.legend(handles=[Line2D([0], [0], marker="s", color="w", markerfacecolor=c, markersize=10, label=l) for l, c in [("ligand", "#5b7a99"), ("receptor", "#7a9a8a"), ("intermediate protein", "#d0d0d0")]]
          + [Line2D([0], [0], color="#5a8f6a", label="activation"), Line2D([0], [0], color="#b5655c", linestyle="--", label="inhibition")], loc="upper left", frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f"network_development_{TARGET}_upstream_impact_decay.png", dpi=130); plt.show()

## 図の点の見た目（共通。Step 17 以降の Plotly の図に使う）
> 指示: the size of the circle reflect the correlation with the target gene. The edge of the circle is same for all genes the color of the circle reflect the function of genes.

- **点の大きさ** = 標的（AKT1）との DepMap の相関（ピアソン `r`）の**大きさ（絶対値）**。相関が強い（正でも負でも）ほど大きい。DepMap にない遺伝子は最小の大きさ。
- **点の縁取り** = **全部の遺伝子で同じ**（濃い灰色）。
- **点の色** = **遺伝子の機能**: リガンド、受容体、転写因子（TF）、キナーゼ、ホスファターゼ、ユビキチン経路、その他。分け方は、まず役割（リガンド・受容体・TF）、次に UniProt のキーワード（Kinase、Protein phosphatase、Ubl conjugation pathway）の順。
- 辺の色・線の種類（促進 = 濃い青の実線、抑制 = 赤の破線）は、これまでと同じ。以前は点の縁の色で「AKT1 への効果の符号」を出していたが、縁は同じにしたので、符号は辺で見る。

- **名前**は自分の点の**上部**に置く（点の上端に少し重なる位置。丸の間の行の間隔は広げてある（1 行 48 ピクセル。二列に分けた列の間隔も広げた））。

- **列の分け方**: 1 つの列（hop など）に遺伝子が **20 個以上**あるときは、半分ずつ**二列**に分けて並べる（上から下の順は保ち、前半を左、後半を右の列に置く）。

**Note**
- 相関は、ネットワークの全遺伝子について新しく計算する（DepMap の CRISPR 遺伝子効果の読み込みが1回増えるので、少し時間がかかる）。
- 大きさは相関の大きさだけで、相関が正か負かは出ない（`corr_net` に符号つきの値がある）。

In [ ]:
# 点の大きさ・色の準備: AKT1 との DepMap の相関と、遺伝子の機能の分類
eff_f = os.path.join(DB["depmap"], "CRISPRGeneEffect.csv")
hdr_ = pd.read_csv(eff_f, nrows=0).columns
colmap_ = {c.split(" (")[0].upper(): c for c in hdr_ if " (" in c}
genes_net = (set(net.src) | set(net.dst) | {TARGET}) & set(colmap_)
dm_net = pd.read_csv(eff_f, usecols=[colmap_[g] for g in genes_net], dtype=np.float32)
dm_net.columns = [c.split(" (")[0].upper() for c in dm_net.columns]                     # usecols は列をファイルの順に返すので、名前は読んだ列から付ける
corr_net = dm_net.corrwith(dm_net[TARGET]).drop(TARGET)                                   # AKT1 との相関（ピアソン、符号つき）

u_ = raw["uniprot"].dropna(subset=["Gene Names (primary)"]).copy()
u_["g"] = u_["Gene Names (primary)"].str.upper(); u_ = u_.drop_duplicates("g").set_index("g")
kw_ = u_.Keywords.fillna("")
uni_cat = pd.Series("other", index=u_.index)                                                # UniProt のキーワードによる機能（弱い順に上書きする）
uni_cat[kw_.str.contains("Ubl conjugation pathway")] = "ubiquitin pathway"
uni_cat[kw_.str.contains("Protein phosphatase")] = "phosphatase"
uni_cat[kw_.str.contains("Kinase")] = "kinase"
FUNC_COL = {"ligand": "#79b9a8", "receptor": "#5d84b8", "transcription factor": "#e0b84f", "kinase": "#d98c5f",
            "phosphatase": "#9b7cb6", "ubiquitin pathway": "#a7855a", "other": "#c4c4c4"}

def func_cat(g):                                                                            # まず役割、次に UniProt の機能
    r_ = role(g)
    if r_ == "リガンド": return "ligand"
    if r_ == "受容体": return "receptor"
    if g in tfs: return "transcription factor"
    return uni_cat.get(g, "other")

def add_gene_nodes(fig, genes, x, y, text_size=11, size_scale=70, size_min=8, opacity=1.0, legend=True, px_per_x=None):
    """点の大きさ = |AKT1 との相関|、縁は全部同じ、色 = 機能。機能ごとに 1 本の trace（凡例に機能が出る）。
    名前は注釈（annotation）で、自分の点の**上部**に、点の上端に少し重なる位置（点の中心から「半径 - 3 ピクセル」だけ上）に置く。文字は中央そろえ。ピクセルでずらすので、図が縮小されて表示されても位置関係が変わらない。
    キーが "遺伝子名@d" のものは、下流側のコピー（名前は @d を除いて出す）。px_per_x は互換のために残してある"""
    base = lambda k: k[:-2] if k.endswith("@d") else k
    cats = {g: func_cat(base(g)) for g in genes}
    for cat in FUNC_COL:
        gs = [g for g in genes if cats[g] == cat]
        if not gs: continue
        r_ = [corr_net.get(base(g), np.nan) for g in gs]
        sz = [size_min + size_scale * (abs(v) if v == v else 0) for v in r_]
        fig.add_trace(go.Scatter(x=[x[g] for g in gs], y=[y[g] for g in gs], mode="markers",
                                 marker=dict(size=sz, color=FUNC_COL[cat], line=dict(color="#444444", width=1.5)), name=cat,
                                 hovertext=[f"<b>{base(g)}</b><br>{cat}<br>DepMap r with {TARGET}: {('%.2f' % v) if v == v else 'n/a'}" for g, v in zip(gs, r_)], hoverinfo="text",
                                 legendgroup="function", opacity=opacity, showlegend=legend))
        for g, s in zip(gs, sz):                                                              # 名前: 自分の点の上部に、点の上端に少し重なる位置（半径 - 3 ピクセル）に置く
            fig.add_annotation(x=x[g], y=y[g], text=base(g), showarrow=False, xanchor="center", yanchor="bottom", yshift=max(s / 2 - 3, 2),
                               font=dict(size=text_size, color="#222222"), opacity=opacity)
    for v in ((0.1, 0.2, 0.4) if legend else ()):                                           # 大きさの凡例
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=size_min + size_scale * v, color="#ffffff", line=dict(color="#444444", width=1.5)),
                                 name=f"|r| = {v}", legendgroup="size"))

def rows_needed(n, thr=20):                                                                  # 列を二つに分けたときの、1 列の行数
    return (n + 1) // 2 if n >= thr else n

def wrap_columns(cols, x, y, thr=20, dx=0.3):
    """遺伝子が thr 個以上の列は、半分ずつ二列に分けて並べる（上から下の順を保ち、前半を左、後半を右に置く）。x, y の辞書を直接書き換える"""
    for ci, (n, genes) in enumerate(cols):
        if len(genes) < thr: continue
        order = sorted(genes, key=lambda g: -y[g])                                          # 上から下の順
        h = (len(order) + 1) // 2
        for part, off in ((order[:h], -dx), (order[h:], dx)):
            for i, g in enumerate(part): x[g] = ci + off; y[g] = (len(part) - 1) / 2 - i


## Step 17. 参考画像の見た目に合わせて描く（Plotly）
> 指示: please refer to the image（ligands | receptors | hop 5 … hop 1 | seed の列、丸い点と右側の遺伝子名、灰色の辺、緑・赤の縁取り）

### 参考画像から読み取ったこと（違っていたら知らせてほしい）
1. **列**: 左から ligands、receptors、hop 5、hop 4、hop 3、hop 2、hop 1、seed（AKT1）。列の見出しは上に灰色の文字。
2. **途中のタンパク質は、hop の列ごとにほぼ同じ数（約 20 個ずつ）**。これを `N_PER_HOP` 個ずつにする。
3. **点**: 丸。遺伝子名は点の**右**に付ける。リガンドは緑がかった青緑、受容体は青、途中のタンパク質は灰色。AKT1 は濃い灰色のひし形。
4. **縁取りの色**: 途中のタンパク質のうち、AKT1 を**促進する**ものは緑、**抑制する**ものは赤の縁（それ以外は縁なし）。ここでは、その遺伝子から AKT1 までの最良経路の符号（辺の符号の積）で決める。
5. **辺**: すべて薄い灰色（符号の色分けなし）、細い線。

### 選び方
- 受容体: Step 16 の影響（媒介する遺伝子の流れ × 減衰、同点は経路スコア）の上位 `N_REC` 個。リガンド: Step 16 の影響の上位 `N_LIG` 個。
- 途中のタンパク質: **hop ごとに**、受容体 → AKT1 の経路が通る遺伝子（流れが 0 より大きいもの）を、流れの大きい順に `N_PER_HOP` 個まで。その hop に該当する遺伝子が少なければ、あるだけ描く。
- 辺: 描いている遺伝子どうしの `geo ≥ 0.25` の辺のうち、各遺伝子から標的の側へ向かうものを `MAX_OUT` 本まで（`E` が大きい順）。

**Note**
- 参考画像の縁取りの意味と、列ごとの個数は私の推測。違うなら、4 と 2 を直す。
- 参考画像では辺がもっと多く見える。ここでは `MAX_OUT` で絞っている（増やすと密になる）。

## 17a. 描く遺伝子を選ぶ
- 動作: 受容体・リガンドは Step 16 の上位。途中のタンパク質は、hop 1 から順に、流れの大きい順に `N_PER_HOP` 個まで。ただし、**最良経路の「次の遺伝子」が、すでに選んだ遺伝子（前の hop）であるものだけ**を選ぶ（こうすると、選んだ遺伝子がすべて AKT1 までつながる。遺伝子がない hop の列は作らない）
- 出力: 列ごとの遺伝子数

In [ ]:
# 描く遺伝子を選ぶ（受容体・リガンドは Step 16 の結果。途中は hop ごとに N_PER_HOP 個、ただし「次の遺伝子」が描く遺伝子であるものだけ）
mid17, prev = {}, {TARGET}                                                                 # hop 1 から順に、前の hop で選んだ遺伝子につながるものだけを選ぶ
for h in range(1, 6):
    cand_h = [g for g in flow13.index if role(g) == "途中" and g != TARGET and hop_of[g] == h and nxt13[g] in prev]     # flow13 は流れの大きい順
    mid17[h] = cand_h[:N_PER_HOP]; prev = set(mid17[h])
mid17 = [(h, mid17[h]) for h in range(5, 0, -1)]                                          # 図の並び（hop 5 -> 1）
cols17 = [("ligands", sel_lig16), ("receptors", sel_rec16)] + [(f"hop {h}", g_) for h, g_ in mid17 if g_] + [("seed", [TARGET])]
print({n: len(g_) for n, g_ in cols17})

## 17b. 辺と並び順を決める
- 動作: 描く遺伝子どうしの辺を選び、各列の縦の並びを、右の列から順に、つながる先の位置の平均で決める（辺の交差を減らす）。各遺伝子の AKT1 への符号（縁取りの色）も計算する
- 出力: 辺の数、縁取りの色別の遺伝子数

In [ ]:
# 辺・並び順・縁取りの色を決める
shown17 = [g for n, g_ in cols17 for g in g_]
x17 = {g: ci for ci, (n, g_) in enumerate(cols17) for g in g_}
mids17 = {g for h, g_ in mid17 for g in g_}
e17 = ev[(ev.geo >= 0.25) & ev.src.isin(shown17) & ev.dst.isin(shown17) & (ev.src != ev.dst)].copy()
lig_part = e17[e17.src.isin(sel_lig16) & e17.dst.isin(sel_rec16)]                         # リガンド -> 受容体
rest = e17[~e17.src.isin(sel_lig16) & ~e17.dst.isin(sel_rec16)]
rest = rest[rest.src.map(x17) < rest.dst.map(x17)]                                         # 左から右へ進む辺だけ
rest = rest.sort_values("E", ascending=False).groupby("src").head(MAX_OUT)                 # 各遺伝子から最大 MAX_OUT 本
tree17 = pd.DataFrame([(g, nxt13[g]) for g in mids17], columns=["src", "dst"])                 # 途中の遺伝子は、最良経路の辺を必ず入れる
edges17 = pd.concat([lig_part[["src", "dst"]], rest[["src", "dst"]], tree17]).drop_duplicates()

y17 = {TARGET: 0.0}
for ci in range(len(cols17) - 2, -1, -1):                                                  # 右の列から順に、つながる先の平均位置で並べる
    genes = cols17[ci][1]
    def key(g):
        t = [y17[b] for a, b in zip(edges17.src, edges17.dst) if a == g and b in y17]
        return (np.mean(t) if t else 1e9, g)
    for i, g in enumerate(sorted(genes, key=key)): y17[g] = (len(genes) - 1) / 2 - i       # 列の中央を 0 にして、上から並べる

def path_sign(g):                                                                          # g から AKT1 までの最良経路の符号（辺の符号の積）
    s = 1
    while g != TARGET:
        e_ = eff_lookup[(g, nxt13[g])]; s *= e_; g = nxt13[g]
    return s
sign17 = {g: path_sign(g) for g in mids17}
alone = [g for g in shown17 if g != TARGET and g not in set(edges17.src) | set(edges17.dst)]
print("辺がない遺伝子:", alone or "なし")
print("辺:", len(edges17), "| 縁取り 緑（促進）:", sum(v == 1 for v in sign17.values()), "赤（抑制）:", sum(v == -1 for v in sign17.values()), "なし:", sum(v == 0 for v in sign17.values()))

## 17c. 描く（Plotly）
- 動作: 参考画像の見た目で描く。辺は曲線、**促進 = 濃い青の実線 + 矢じり、抑制 = 赤の破線 + 「T」字の棒、符号不明 = 灰色の点線**。凡例（点の種類・縁取りの色・辺の種類）を下に付ける。点にポイントすると、遺伝子名・役割・AKT1 への効果が出る
- 出力: 画面の図と、HTML ファイル・PNG ファイル

In [ ]:
# 参考画像の見た目で描く（Plotly、曲線の辺、促進と抑制を区別、凡例つき）
import plotly.graph_objects as go

wrap_columns(cols17, x17, y17)                                                         # 20 個以上の列は二列に分ける
ncol = len(cols17); nmax = max(rows_needed(len(g_)) for n, g_ in cols17)
W, Hpx = 1500, max(780, 48 * nmax + 230)
xr, yr = (-0.4, ncol - 0.2), (-nmax / 2 - 1.5, nmax / 2 + 2.5)
sx, sy = (W - 40) / (xr[1] - xr[0]), (Hpx - 140) / (yr[1] - yr[0])                  # 1 目盛りあたりのピクセル数（矢印の向きの計算に使う）
TT = np.linspace(0, 1, 28)

def bez(a, b):                                                                       # 横に S 字を描く 3 次ベジェ曲線
    (x0, y0), (x1, y1) = (x17[a], y17[a]), (x17[b], y17[b]); dx = x1 - x0
    P = np.array([[x0, y0], [x0 + 0.5 * dx, y0], [x1 - 0.5 * dx, y1], [x1, y1]])
    pt = lambda t: (1 - t) ** 3 * P[0] + 3 * (1 - t) ** 2 * t * P[1] + 3 * (1 - t) * t ** 2 * P[2] + t ** 3 * P[3]
    d = lambda t: 3 * (1 - t) ** 2 * (P[1] - P[0]) + 6 * (1 - t) * t * (P[2] - P[1]) + 3 * t ** 2 * (P[3] - P[2])
    return np.array([pt(t) for t in TT]), pt, d

fig = go.Figure()
kinds = {1: ("activation", "#1f3a8a", "solid"), -1: ("inhibition", "#c0504d", "dash"), 0: ("unknown sign", "#8a8a8a", "dot")}
for e_, (kname, kcol, kdash) in kinds.items():
    xs, ys, hx, hy, hang = [], [], [], [], []
    for a, b in zip(edges17.src, edges17.dst):
        if eff_lookup[(a, b)] != e_: continue
        line, pt, d = bez(a, b)
        xs += list(line[:, 0]) + [None]; ys += list(line[:, 1]) + [None]
        t_end = 0.93 if e_ == -1 else 0.90                                          # 抑制は線の端に近い所に、促進は少し手前に矢じり
        p = pt(t_end); v = d(t_end); th = math.degrees(math.atan2(v[0] * sx, v[1] * sy))      # 上を 0 度として時計回り
        hx.append(p[0]); hy.append(p[1]); hang.append(th - 90 if e_ == -1 else th)
    if not xs: continue
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=kcol, width=1.3, dash=kdash), opacity=0.75, hoverinfo="skip", name=f"{kname} edge", legendgroup=kname))
    fig.add_trace(go.Scatter(x=hx, y=hy, mode="markers", hoverinfo="skip", showlegend=False, legendgroup=kname,
                             marker=dict(symbol="line-ns" if e_ == -1 else "arrow", size=11 if e_ == -1 else 10, angle=hang, angleref="up", color=kcol,
                                         line=dict(color=kcol, width=2.5) if e_ == -1 else None)))   # 抑制は「T」字の棒、促進・不明は矢じり

add_gene_nodes(fig, [g for g in shown17 if g != TARGET], x17, y17)                           # 点: 大きさ = AKT1 との相関、縁は同じ、色 = 機能
fig.add_trace(go.Scatter(x=[x17[TARGET]], y=[y17[TARGET]], mode="markers+text", text=[TARGET], textposition="top center", textfont=dict(size=15), name="seed (target)",
                         marker=dict(size=22, color="#555555", symbol="diamond"), hoverinfo="skip"))
for ci, (n, genes) in enumerate(cols17):                                            # 列の見出し
    fig.add_annotation(x=ci, y=yr[1] - 0.6, text=n, showarrow=False, font=dict(size=14, color="#666666"))
fig.update_layout(width=W, height=Hpx, plot_bgcolor="white", margin=dict(l=20, r=20, t=30, b=110),
                  xaxis=dict(visible=False, range=list(xr)), yaxis=dict(visible=False, range=list(yr)),
                  legend=dict(orientation="h", yanchor="top", y=-0.01, xanchor="center", x=0.5, font=dict(size=11)))
fig.write_html(f"network_development_{TARGET}_upstream_refstyle.html", include_plotlyjs="cdn")
try:
    fig.write_image(f"network_development_{TARGET}_upstream_refstyle.png", scale=1.5)         # PNG は kaleido があるときだけ
except Exception as e:
    print("PNG は保存できなかった（HTML は保存済み）:", str(e)[:80])
fig.show()

## Step 18. AKT1 の上流の遺伝子リスト（いろいろなスコアと注釈つき）
> 指示: please make a list of genes upstream of AKT1 with various scores and annotations.

### 対象
- `geo ≥ 0.25` の辺で AKT1 から 6 ホップまでたどった **1,438 遺伝子**（Step 10〜11 の選び方。リガンド・受容体・途中のタンパク質を全部含む）。

### 列の種類
1. **経路**: 役割（リガンド / 受容体 / 途中）、hop、AKT1 に向かう次の遺伝子、最良経路、その経路の符号（AKT1 を促進するか抑制するか）。
2. **スコア**: 経路スコア（辺の `E` の積）、次の遺伝子への辺のスコア（`E`、`weight`、`geo`、論文数、SIGNOR の `SCORE`）、AKT1 への直接の辺の `E`、流れ（Step 13）、通る受容体の数、受容体の影響（Step 16）、リガンドの影響。
3. **辺の注釈**: 次の遺伝子への辺の出典（SIGNOR / OmniPath）、仕組み、残基。
4. **遺伝子の注釈**: UniProt（局在、キナーゼ・ホスファターゼ・ユビキチン経路などの印）、転写因子か、HPA（発現している組織の数と最大の発現量）、DepMap（必須性の平均と AKT1 との共必須性）。
5. **図との対応**: 上位 100 遺伝子（Step 13）か、Step 17 の図に描いたか。

**Note**
- DepMap の遺伝子効果は、負で小さいほど、その遺伝子を壊すと細胞が増えにくい（必須）。共必須性は、AKT1 との遺伝子効果の相関（細胞株をまたいで同じように効く遺伝子ほど 1 に近い）。
- 注釈は、遺伝子名（大文字）で UniProt・HPA・DepMap に突き合わせている。名前が合わない遺伝子は空欄になる。

## 18a. 経路とスコアの表を作る
- 動作: 11a の表 `fig_tbl`（各遺伝子の次の遺伝子など）に、最良経路・符号・辺のスコア・流れ・影響を付ける
- 出力: 遺伝子数と、役割ごとの数

In [ ]:
# 経路とスコアの表
rows = []
ed = ev.set_index(["src", "dst"])
imp_all = dict(zip(imp16.receptor, imp16.impact))                                         # 受容体の影響（全 130 個）
lig_imp = (ev[(ev.geo >= 0.25) & (ev.src.map(role) == "リガンド") & ev.dst.isin(imp_all)].assign(imp=lambda d: d.E * d.dst.map(imp_all))
           .groupby("src").imp.max())                                                      # リガンドの影響 = 辺の E × 受容体の影響の最大値
for r in fig_tbl.itertuples():
    g, n = r.gene, r.next_gene
    pth = path_to_target(g); e = ed.loc[(g, n)]
    sgn = 1
    for a, b in zip(pth[:-1], pth[1:]): sgn *= eff_lookup[(a, b)]
    rows.append(dict(gene=g, role=role(g), hop=int(r.hop), next_gene=n, path=" > ".join(pth), path_sign={1: "activates", -1: "inhibits", 0: "unknown"}[sgn],
                     path_score=round(ps13[g], 4), E_to_next=round(e.E, 3), weight_to_next=round(e.weight, 3), geo_to_next=round(e.geo, 3),
                     effect_to_next=int(e.effect), refs_to_next=e.refs, signor_to_next=e.signor, source_to_next=e.source,
                     mechanism_to_next=str(e.mechanism)[:90], residue_to_next=e.residue,
                     direct_E_to_target=round(ed.loc[(g, TARGET)].E, 3) if (g, TARGET) in ed.index else np.nan,
                     flow=round(flow13.flow.get(g, 0.0), 4), n_receptors=int(flow13.n_receptors.get(g, 0)),
                     receptor_impact=round(imp_all[g], 4) if g in imp_all else np.nan,
                     ligand_impact=round(lig_imp[g], 4) if g in lig_imp.index else np.nan,
                     in_top100=g in top13.index, in_figure17=g in set(shown17)))
up_list = pd.DataFrame(rows)
print("遺伝子:", len(up_list), up_list.role.value_counts().to_dict())
up_list.head()

## 18b. 遺伝子の注釈を付ける
- 動作: UniProt（局在・機能の印）、Lambert（転写因子）、HPA（発現）、DepMap（必須性・AKT1 との共必須性）を、遺伝子名で突き合わせて列を足す
- 出力: 注釈がついた遺伝子の数

In [ ]:
# UniProt・転写因子・HPA・DepMap の注釈を付ける
u = raw["uniprot"].dropna(subset=["Gene Names (primary)"]).copy()
u["gene"] = u["Gene Names (primary)"].str.upper(); u = u.drop_duplicates("gene").set_index("gene")
kw, loc = u.Keywords.fillna(""), u["Subcellular location [CC]"].fillna("")
places = {"membrane": "membrane", "nucleus": "nucleus", "cytoplasm|cytosol": "cytoplasm", "secreted": "secreted", "mitochondri": "mitochondrion",
          "endoplasmic reticulum|golgi": "ER/Golgi", "lysosome": "lysosome"}
loc_txt = loc.map(lambda t: ";".join(v for k, v in places.items() if re.search(k, t, re.I)))
flags = {"Kinase": "kinase", "Protein phosphatase": "phosphatase", "Ubl conjugation pathway": "ubiquitin_pathway", "Apoptosis": "apoptosis", "Cell cycle": "cell_cycle",
         "Transcription": "transcription", "Receptor": "receptor"}
flag_txt = kw.map(lambda t: ";".join(v for k, v in flags.items() if k in t))
up_list["locations"] = up_list.gene.map(loc_txt).fillna(""); up_list["uniprot_flags"] = up_list.gene.map(flag_txt).fillna("")
tfset = set(raw["tf_list"].loc[raw["tf_list"]["Is TF?"] == "Yes", "HGNC symbol"].str.upper())
up_list["is_TF"] = up_list.gene.isin(tfset)

h = raw["hpa_tissue"].assign(g=lambda d: d["Gene name"].str.upper())                                  # HPA: 発現している組織の数と最大の発現量
hg = h.groupby("g").agg(n_tissues_expressed=("nTPM", lambda s: int((s >= 1).sum())), max_nTPM=("nTPM", "max"))
hg["top_tissue"] = h.loc[h.groupby("g").nTPM.idxmax()].set_index("g").Tissue
up_list = up_list.join(hg, on="gene")

# DepMap: 必須性の平均と、AKT1 との共必須性（遺伝子効果の相関）。対象の遺伝子の列だけ読む
eff_file = os.path.join(DB["depmap"], "CRISPRGeneEffect.csv")
hdr = pd.read_csv(eff_file, nrows=0).columns
colmap = {c.split(" (")[0].upper(): c for c in hdr if " (" in c}
want = [g for g in set(up_list.gene) | {TARGET} if g in colmap]
dm = pd.read_csv(eff_file, usecols=[colmap[g] for g in want])
dm.columns = [c.split(" (")[0].upper() for c in dm.columns]               # usecols は列をファイルの順に返すので、名前は読んだ列から付け直す
up_list["depmap_mean_effect"] = up_list.gene.map(dm.mean().round(3))
up_list["depmap_corr_AKT1"] = up_list.gene.map(dm.corrwith(dm[TARGET]).round(3))
print("注釈あり: UniProt", int((up_list.locations != "").sum()), "| HPA", int(up_list.n_tissues_expressed.notna().sum()), "| DepMap", int(up_list.depmap_mean_effect.notna().sum()), "| TF", int(up_list.is_TF.sum()))
up_list.head(3).T

## 18c. リストを見る・保存する
- 動作: 流れの大きい順に並べ、上位を表示する。全体を CSV に保存する。列の説明を表にする
- 出力: 上位 40 遺伝子の表、列の説明、CSV ファイル（`network_development_AKT1_upstream_genes.csv`）

In [ ]:
# リストを並べて保存する
up_list = up_list.sort_values(["flow", "path_score"], ascending=False).reset_index(drop=True)
up_list.to_csv(f"network_development_{TARGET}_upstream_genes.csv", index=False, encoding="utf-8-sig")
print("保存:", f"network_development_{TARGET}_upstream_genes.csv", "|", up_list.shape)
show_cols = ["gene", "role", "hop", "next_gene", "path_sign", "path_score", "E_to_next", "geo_to_next", "flow", "n_receptors", "receptor_impact", "ligand_impact",
             "locations", "uniprot_flags", "is_TF", "depmap_mean_effect", "depmap_corr_AKT1", "in_top100"]
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
up_list[show_cols].head(40)

In [ ]:
# 列の説明
desc = {
 "gene": "遺伝子名", "role": "役割（ligand / receptor / intermediate）", "hop": "AKT1 までの最短ステップ数", "next_gene": "AKT1 に向かう最良の次の遺伝子（辺の E が最大）",
 "path": "最良経路", "path_sign": "最良経路の符号の積（AKT1 を促進 / 抑制 / 不明）", "path_score": "経路スコア（最良経路の辺の E の積）",
 "E_to_next": "次の遺伝子への辺の証拠スコア E", "weight_to_next": "同じ辺の weight（順位ベースのスコア）", "geo_to_next": "同じ辺の √(weight × E)",
 "effect_to_next": "次の遺伝子への辺の符号（+1 促進、-1 抑制）", "refs_to_next": "その辺の論文数（OmniPath）", "signor_to_next": "その辺の SIGNOR スコア",
 "source_to_next": "その辺の出典（SIGNOR / OmniPath / OmniPath+SIGNOR）", "mechanism_to_next": "その辺の仕組み", "residue_to_next": "その辺の修飾残基",
 "direct_E_to_target": "AKT1 への直接の辺の E（なければ空欄）", "flow": "流れ（全受容体の最良経路が通る量の合計。Step 13）", "n_receptors": "この遺伝子を通る受容体の数",
 "receptor_impact": "受容体の影響（Step 16。受容体だけ）", "ligand_impact": "リガンドの影響（辺の E × 受容体の影響の最大値。リガンドだけ）",
 "in_top100": "上位 100 遺伝子（Step 13）に入るか", "in_figure17": "Step 17 の図に描いたか",
 "locations": "局在（UniProt の文章から）", "uniprot_flags": "機能の印（kinase / phosphatase / ubiquitin_pathway / apoptosis / cell_cycle / transcription / receptor）",
 "is_TF": "転写因子か（Lambert）", "n_tissues_expressed": "発現している組織の数（nTPM ≥ 1、約 50 組織中）", "max_nTPM": "組織の中の最大の発現量", "top_tissue": "最大の発現量の組織",
 "depmap_mean_effect": "DepMap の遺伝子効果の平均（負で小さいほど必須）", "depmap_corr_AKT1": "AKT1 との遺伝子効果の相関（共必須性）"}
miss = [c for c in up_list.columns if c not in desc]
display(pd.DataFrame({"説明": desc}).rename_axis("column"))
print("説明がない列:", miss or "なし")

## Step 19. 受容体を KDR だけに限ると、遺伝子リストと経路はどう変わるか
> 指示: How does the gene list and pathway change if receptor is restricted as KDR.

### 考え方
- これまでは、**130 個すべての受容体**から AKT1 への経路を考えていた（1,438 遺伝子）。今回は、受容体を **KDR だけ**にして、**KDR から AKT1 までの経路が通る遺伝子と経路**だけを取り出す。
- 条件は同じ（`geo ≥ 0.25` の辺、リガンドは出発点だけ、TF は終点だけ）。KDR から AKT1 まで最大 `MAX_STEPS_KDR`（既定 4）ステップの**単純な経路（同じ遺伝子を2回通らない）をすべて**数える。
- 経路のスコア = 辺の `E` の積。符号 = 辺の符号の積（AKT1 を促進するか抑制するか）。
- リガンドは、KDR に入る辺（`geo ≥ 0.25`）があるものを添える。KDR を調節する遺伝子（SRC、PTPN6 など）も別の表に出す。

**Note**
- 「KDR から AKT1 への経路」は KDR の下流（AKT1 の側）の話。KDR に入る調節因子（SRC、PTPN1 など）は、KDR → AKT1 の経路の外にある。
- 比較は、全受容体の場合（Step 18 のリスト、1,438 遺伝子）との差で見る。

## 19a. KDR から AKT1 までの経路が通る遺伝子の数（ステップ数ごと）
- 動作: `geo ≥ 0.25` の辺のグラフで、KDR からのステップ数と AKT1 までのステップ数を求め、合計が最大ステップ数以内の遺伝子を数える
- 出力: 最大ステップ数 2〜6 での遺伝子の数。全受容体の場合との比較

In [ ]:
# KDR から AKT1 までの経路が通る遺伝子の数
Gk = nx.DiGraph()
for r in ev[ev.geo >= 0.25].itertuples(): Gk.add_edge(r.src, r.dst, E=r.E, effect=r.effect)
R = RESTRICT_RECEPTOR
d_from = nx.single_source_shortest_path_length(Gk, R, cutoff=6)                         # KDR からのステップ数
d_to = nx.single_source_shortest_path_length(Gk.reverse(copy=False), TARGET, cutoff=6)   # AKT1 までのステップ数
rows = []
for K in range(2, 7):
    S = {g for g in d_from if g in d_to and d_from[g] + d_to[g] <= K}
    rows.append(dict(最大ステップ数=K, 遺伝子=len(S), 受容体=sum(role(g) == "受容体" for g in S), 途中のタンパク質=sum(role(g) == "途中" and g != TARGET for g in S) ))
cmp19 = pd.DataFrame(rows).set_index("最大ステップ数")
print(f"全受容体（Step 18）: {len(up_list)} 遺伝子（受容体 {int((up_list.role == '受容体').sum())}）")
print(f"KDR から AKT1 への最短ステップ数: {d_to[R]}")
cmp19

## 19b. KDR から AKT1 への経路をすべて数える
- 動作: KDR から AKT1 まで最大 `MAX_STEPS_KDR` ステップの単純な経路をすべて数え、スコア（`E` の積）と符号を付ける
- 出力: 経路の数（符号別・ステップ数別）と、スコア上位 15 の経路

In [ ]:
# KDR -> AKT1 の経路をすべて数える
S_k = {g for g in d_from if g in d_to and d_from[g] + d_to[g] <= MAX_STEPS_KDR}                  # 最大ステップ数以内の経路が通りうる遺伝子だけに絞ってから数える（速くする）
paths = list(nx.all_simple_paths(Gk.subgraph(S_k), R, TARGET, cutoff=MAX_STEPS_KDR))
prow = []
for pth in paths:
    sc, sg = 1.0, 1
    for a, b in zip(pth[:-1], pth[1:]): sc *= Gk[a][b]["E"]; sg *= Gk[a][b]["effect"]
    prow.append(dict(path=" > ".join(pth), steps=len(pth) - 1, score=round(sc, 4), sign={1: "activates", -1: "inhibits", 0: "unknown"}[sg]))
paths19 = pd.DataFrame(prow).sort_values("score", ascending=False).reset_index(drop=True)
print(f"KDR → AKT1 の経路（最大 {MAX_STEPS_KDR} ステップ）: {len(paths19)} 本 | 符号別:", paths19.sign.value_counts().to_dict(), "| ステップ数別:", paths19.steps.value_counts().sort_index().to_dict())
paths19.head(15)

## 19c. KDR に限った遺伝子リスト
- 動作: 経路が通る遺伝子ごとに、KDR・AKT1 からのステップ数、通る経路の数（促進 / 抑制）、その遺伝子を通る最良経路のスコアを求め、Step 18 の注釈と全受容体の場合の流れ・順位を付ける。KDR に入る調節因子の表も作る
- 出力: 遺伝子の表（CSV に保存）、KDR を調節する遺伝子の表

In [ ]:
# KDR に限った遺伝子リスト
stat = {}
for pth in paths:
    sc, sg = 1.0, 1
    for a, b in zip(pth[:-1], pth[1:]): sc *= Gk[a][b]["E"]; sg *= Gk[a][b]["effect"]
    for g in pth:
        s_ = stat.setdefault(g, dict(n_paths=0, n_act=0, n_inh=0, best=0.0, best_sign=0, best_path=""))
        s_["n_paths"] += 1; s_["n_act"] += sg == 1; s_["n_inh"] += sg == -1
        if sc > s_["best"]: s_["best"], s_["best_sign"], s_["best_path"] = sc, sg, " > ".join(pth)
kdr_list = pd.DataFrame([dict(gene=g, role="AKT1" if g == TARGET else role(g), steps_from_KDR=d_from[g], steps_to_AKT1=d_to[g], n_paths=v["n_paths"],
                              n_paths_activating=int(v["n_act"]), n_paths_inhibiting=int(v["n_inh"]), best_path_score=round(v["best"], 4),
                              best_path_sign={1: "activates", -1: "inhibits", 0: "unknown"}[v["best_sign"]], best_path_through=v["best_path"]) for g, v in stat.items()])
ann = up_list.set_index("gene")[["flow", "n_receptors", "in_top100", "in_figure17", "locations", "uniprot_flags", "n_tissues_expressed", "max_nTPM", "depmap_mean_effect", "depmap_corr_AKT1"]]
kdr_list = kdr_list.join(ann.rename(columns={"flow": "flow_all_receptors", "n_receptors": "n_receptors_all"}), on="gene")
kdr_list["rank_flow_all"] = kdr_list.gene.map(up_list.reset_index().set_index("gene")["index"] + 1)                  # 全受容体の場合の順位（流れの大きい順）
kdr_list = kdr_list.sort_values(["n_paths", "best_path_score"], ascending=False).reset_index(drop=True)
kdr_list.to_csv(f"network_development_{TARGET}_upstream_genes_{R}_{MAX_STEPS_KDR}steps.csv", index=False, encoding="utf-8-sig")
print(f"遺伝子: {len(kdr_list)}（受容体 {int((kdr_list.role == '受容体').sum())} 個）| うち上位 100 遺伝子に入る: {int(kdr_list.in_top100.sum())} 個 | CSV に保存")
reg = ev[(ev.geo >= 0.25) & (ev.dst == R)][["src", "effect", "E", "refs", "signor", "source"]].assign(role=lambda d: d.src.map(role)).sort_values("E", ascending=False)
display(reg.rename(columns={"src": f"{R} に入る遺伝子"}).reset_index(drop=True))
kdr_list.head(25)

## 19d. 図（KDR に限った経路）
- 動作: 左から ligands | KDR | KDR からのステップ数の列 | AKT1 の順に並べ、KDR → AKT1 の経路の辺をすべて、曲線で描く（促進 = 濃い青の実線と矢じり、抑制 = 赤の破線と「T」字、縁取りはその遺伝子を通る最良経路の符号）
- 出力: 図（画面と HTML・PNG に保存）。列の順に進まない辺は描かない（本数を表示）

In [ ]:
# KDR に限った経路の図（Plotly）
S = set(stat)
ligs = sorted(reg[reg.role == "リガンド"].src)                                                    # KDR に結合するリガンド
cols19 = [("ligands", ligs), (R, [R])] + [(f"step {k} from {R}", sorted(g for g in S if g not in (R, TARGET) and d_from[g] == k)) for k in range(1, MAX_STEPS_KDR)] + [("seed", [TARGET])]
cols19 = [(n, g_) for n, g_ in cols19 if g_]
x19 = {g: ci for ci, (n, g_) in enumerate(cols19) for g in g_}
edges19 = {(a, b) for pth in paths for a, b in zip(pth[:-1], pth[1:])} | {(l, R) for l in ligs}
edges19 = {(a, b) for a, b in edges19 if x19.get(a, 99) < x19.get(b, -1)}                         # 左から右へ進む辺だけ
print("描かない辺（列の順に進まない）:", len({(a, b) for pth in paths for a, b in zip(pth[:-1], pth[1:])}) - (len(edges19) - len([1 for l in ligs])))

y19 = {TARGET: 0.0}
for ci in range(len(cols19) - 2, -1, -1):
    for i, g in enumerate(sorted(cols19[ci][1], key=lambda g: (np.mean([y19[b] for a, b in edges19 if a == g and b in y19] or [1e9]), g))): y19[g] = (len(cols19[ci][1]) - 1) / 2 - i
sign19 = {g: stat[g]["best_sign"] for g in S}
wrap_columns(cols19, x19, y19)
ncol, nmax = len(cols19), max(rows_needed(len(g_)) for n, g_ in cols19)
W, Hpx = 1400, max(620, 48 * nmax + 230)
xr, yr = (-0.4, ncol - 0.2), (-nmax / 2 - 1.5, nmax / 2 + 2.5)
sx, sy = (W - 40) / (xr[1] - xr[0]), (Hpx - 140) / (yr[1] - yr[0])
TT = np.linspace(0, 1, 28)
fig = go.Figure()
for e_, (kname, kcol, kdash) in kinds.items():
    xs, ys, hx, hy, hang = [], [], [], [], []
    for a, b in edges19:
        if Gk[a][b]["effect"] != e_: continue
        (x0, y0), (x1, y1) = (x19[a], y19[a]), (x19[b], y19[b]); dx = x1 - x0
        P = np.array([[x0, y0], [x0 + 0.5 * dx, y0], [x1 - 0.5 * dx, y1], [x1, y1]])
        pt = lambda t: (1 - t) ** 3 * P[0] + 3 * (1 - t) ** 2 * t * P[1] + 3 * (1 - t) * t ** 2 * P[2] + t ** 3 * P[3]
        dv = lambda t: 3 * (1 - t) ** 2 * (P[1] - P[0]) + 6 * (1 - t) * t * (P[2] - P[1]) + 3 * t ** 2 * (P[3] - P[2])
        pts = np.array([pt(t) for t in TT]); xs += list(pts[:, 0]) + [None]; ys += list(pts[:, 1]) + [None]
        te = 0.93 if e_ == -1 else 0.88; p_, v_ = pt(te), dv(te); th = math.degrees(math.atan2(v_[0] * sx, v_[1] * sy))
        hx.append(p_[0]); hy.append(p_[1]); hang.append(th - 90 if e_ == -1 else th)
    if not xs: continue
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=kcol, width=1.5, dash=kdash), opacity=0.8, hoverinfo="skip", name=f"{kname} edge", legendgroup=kname))
    fig.add_trace(go.Scatter(x=hx, y=hy, mode="markers", hoverinfo="skip", showlegend=False, legendgroup=kname,
                             marker=dict(symbol="line-ns" if e_ == -1 else "arrow", size=11 if e_ == -1 else 11, angle=hang, angleref="up", color=kcol, line=dict(color=kcol, width=2.5) if e_ == -1 else None)))
add_gene_nodes(fig, [g for g in x19 if g != TARGET], x19, y19, text_size=12)
fig.add_trace(go.Scatter(x=[x19[TARGET]], y=[y19[TARGET]], mode="markers+text", text=[TARGET], textposition="top center", textfont=dict(size=15), name="seed (target)",
                         marker=dict(size=22, color="#555555", symbol="diamond"), hoverinfo="skip"))
for ci, (n, genes) in enumerate(cols19): fig.add_annotation(x=ci, y=yr[1] - 0.6, text=n, showarrow=False, font=dict(size=14, color="#666666"))
fig.update_layout(width=W, height=Hpx, plot_bgcolor="white", margin=dict(l=20, r=20, t=30, b=110), xaxis=dict(visible=False, range=list(xr)), yaxis=dict(visible=False, range=list(yr)),
                  legend=dict(orientation="h", yanchor="top", y=-0.01, xanchor="center", x=0.5, font=dict(size=11)))
fig.write_html(f"network_development_{TARGET}_upstream_{R}_{MAX_STEPS_KDR}steps.html", include_plotlyjs="cdn")
try: fig.write_image(f"network_development_{TARGET}_upstream_{R}_{MAX_STEPS_KDR}steps.png", scale=1.5)
except Exception as e: print("PNG は保存できなかった:", str(e)[:80])
fig.show()

## Step 20. DepMap で AKT1 と相関する遺伝子のリスト（シグナルのネットワークとは独立）
> 指示: please make another gene list, independent from the signal network, regarding DepMap correlation with AKT1.

### 方法
- データは DepMap の CRISPR 遺伝子効果（Chronos、細胞株 × 遺伝子）。**シグナルのネットワークは使わず**、DepMap にある**全遺伝子**について、AKT1 の遺伝子効果との相関を、細胞株をまたいで計算する。
- 相関が正で高い遺伝子は、AKT1 と同じ細胞株で効く（**共必須**: AKT1 と同じ経路・複合体に属することが多い）。負で低い遺伝子は、AKT1 が効く細胞株では効かない（**逆の関係**）。
- 指標: ピアソン相関 `r`（外れ値に敏感）、スピアマン相関 `rho`（順位ベースで外れ値に強い）、細胞株の数、p 値と FDR（Benjamini-Hochberg）。
- 注釈: UniProt（局在・機能の印）、HPA（発現）、そして**参考として**、Step 18 のシグナルのネットワークの上流リストに入るか（順位づけには使わない）。

**Note**
- 相関は因果ではない。共通のドライバー（細胞株の増殖速度や系統）があると、無関係な遺伝子どうしでも相関する。
- AKT1 自身の遺伝子効果の平均は、必須遺伝子ほど強くは効かない（AKT1 は重複する AKT2 などにより、効果が出にくい細胞株が多い）。

## 20a. 全遺伝子について AKT1 との相関を計算する
- 動作: DepMap の遺伝子効果の表を読み、AKT1 の列との相関（ピアソン・スピアマン）を全遺伝子で計算する。p 値と FDR も求める
- 出力: 読み込んだ細胞株と遺伝子の数、AKT1 の遺伝子効果の分布

In [ ]:
# DepMap の遺伝子効果の全遺伝子と AKT1 の相関
from scipy import stats

eff_file = os.path.join(DB["depmap"], "CRISPRGeneEffect.csv")
hdr = pd.read_csv(eff_file, nrows=0).columns
dm_all = pd.read_csv(eff_file, index_col=0, dtype={c: np.float32 for c in hdr[1:]})        # 細胞株 × 遺伝子（メモリを節約するため float32）
dm_all.columns = [c.split(" (")[0].upper() for c in dm_all.columns]
a = dm_all[TARGET]
print(f"細胞株 {dm_all.shape[0]}、遺伝子 {dm_all.shape[1]}")
print(f"{TARGET} の遺伝子効果: 平均 {a.mean():.3f}、中央値 {a.median():.3f}、-0.5 以下の細胞株 {int((a <= -0.5).sum())}（{100 * (a <= -0.5).mean():.1f}%）")

r_p = dm_all.corrwith(a)                                                                     # ピアソン
r_s = dm_all.rank().corrwith(a.rank())                                                       # スピアマン（順位の相関）
n = dm_all.notna().mul(a.notna(), axis=0).sum()                                              # 両方の値がある細胞株の数
t = r_p * np.sqrt((n - 2) / (1 - r_p ** 2)); pval = pd.Series(2 * stats.t.sf(np.abs(t), n - 2), index=r_p.index)
order = pval.sort_values().index; ranks = np.empty(len(order)); ranks[:] = np.arange(1, len(order) + 1)
fdr = pd.Series(np.minimum.accumulate((pval[order].values * len(order) / ranks)[::-1])[::-1].clip(max=1), index=order)      # Benjamini-Hochberg
codep = pd.DataFrame({"pearson_r": r_p.round(3), "spearman_rho": r_s.round(3), "n_models": n.astype(int), "p_value": pval, "FDR": fdr}).drop(index=TARGET)
codep["mean_effect"] = dm_all.mean().round(3).reindex(codep.index)
print("FDR < 0.05:", int((codep.FDR < 0.05).sum()), "遺伝子 | r > 0.2:", int((codep.pearson_r > 0.2).sum()), "| r < -0.2:", int((codep.pearson_r < -0.2).sum()))

## 20b. 注釈を付けて、リストにする
- 動作: 相関の高い順に並べ、UniProt の局在・機能の印、HPA の発現、シグナルのネットワークの上流リストに入るか（参考）を付けて、CSV に保存する
- 出力: 相関が正で高い上位 40 遺伝子、負で低い上位 15 遺伝子、ネットワークとの重なり

In [ ]:
# 注釈をつけて保存する
codep = codep.sort_values("pearson_r", ascending=False)
codep["locations"] = codep.index.map(loc_txt).fillna(""); codep["uniprot_flags"] = codep.index.map(flag_txt).fillna("")
codep = codep.join(hg[["n_tissues_expressed", "max_nTPM"]], how="left")
ul = up_list.set_index("gene")
codep["in_upstream_network"] = codep.index.isin(ul.index)                                    # 参考: Step 18 の上流リストに入るか（順位づけには使っていない）
codep["network_hop"] = codep.index.map(ul.hop)
codep["network_role"] = codep.index.map(ul.role)
codep = codep.rename_axis("gene").reset_index()
codep.to_csv(f"network_development_{TARGET}_DepMap_correlation.csv", index=False, encoding="utf-8-sig")
topN = codep.head(100)
print(f"保存: network_development_{TARGET}_DepMap_correlation.csv | {len(codep)} 遺伝子")
print(f"相関が高い上位 100 遺伝子のうち、シグナルの上流ネットワーク（1,438 遺伝子）にある: {int(topN.in_upstream_network.sum())} 個 → {', '.join(topN[topN.in_upstream_network].gene.head(15))}")
show = ["gene", "pearson_r", "spearman_rho", "n_models", "FDR", "mean_effect", "locations", "uniprot_flags", "in_upstream_network", "network_hop"]
display(codep[show].head(40))
codep[show].tail(15).iloc[::-1]

## 20c. 相関が高い遺伝子の図
- 動作: 相関（ピアソン）が高い上位 `TOP_CODEP` 遺伝子の棒グラフ。シグナルのネットワークの上流リストに入る遺伝子は色を変える
- 出力: 図（画面と HTML に保存）

In [ ]:
# 相関が高い遺伝子の棒グラフ
top = codep.head(TOP_CODEP).iloc[::-1]
fig = go.Figure(go.Bar(x=top.pearson_r, y=top.gene, orientation="h", marker=dict(color=["#5d84b8" if v else "#bdbdbd" for v in top.in_upstream_network]),
                       hovertext=[f"{g}: r={r:.2f}, rho={s:.2f}, FDR={f:.1e}" for g, r, s, f in zip(top.gene, top.pearson_r, top.spearman_rho, top.FDR)], hoverinfo="text"))
fig.update_layout(title=f"Genes whose CRISPR effect correlates with {TARGET} (DepMap, top {TOP_CODEP}); blue = also in the signaling upstream network", width=900, height=max(500, 22 * TOP_CODEP + 120),
                  plot_bgcolor="white", xaxis_title="Pearson r (gene effect across cell lines)", margin=dict(l=100, r=20, t=60, b=50))
fig.write_html(f"network_development_{TARGET}_DepMap_correlation.html", include_plotlyjs="cdn")
try: fig.write_image(f"network_development_{TARGET}_DepMap_correlation.png", scale=1.5)
except Exception as e: print("PNG は保存できなかった:", str(e)[:80])
fig.show()

## Step 21. AKT1 の下流（標的 → 転写因子）
> 指示: please go ahead of downstream of AKT1.

### 上流との対応
| | 上流（Step 11〜18） | 下流（今回） |
|---|---|---|
| 出発点 | 標的 AKT1 から逆向きに | 標的 AKT1 から順向きに |
| 終点 | 受容体（リガンドは添える） | **転写因子（TF）**。図の右端 |
| 辺 | `net_lt`（リガンド → TF の経路に枝刈りしたもの） | **ライセンス判定を通った全ネットワーク**（Step 3 の 48,863 辺）。`net_lt` は AKT1 の下流の深さを浅くしてしまうため |
| 選び方 | `geo ≥ 0.25` の辺を1ホップずつ（最大6ホップ） | 同じ。TF から先へは進まない（TF は終点） |
| 影響 | 受容体の影響 | **TF の影響** = 媒介する遺伝子の流れ × `DECAY` ^ （ステップ数 − 1）。AKT1 → TF の直接の辺が最高 |

**Note**
- 辺のスコア `E`（論文数と SIGNOR のスコア）と `geo` は、全ネットワークの辺について計算し直す。
- 流れ = 各 TF について、AKT1 からその TF までの最良経路（`E` の積）が通る遺伝子すべてに、経路スコアを足したもの。
- TF の一覧（Lambert）のライセンスは「曖昧」だが、TF の定義に必要なので使っている（Step 4 と同じ）。

## 21a. 全ネットワークの辺のスコア
- 動作: Step 3 の全ネットワーク `net` の各辺に、論文数・SIGNOR スコア・`E`・`geo` を付ける。TF から出る辺は捨てる（TF は終点）
- 出力: 辺の数、`geo ≥ 0.25` の辺の数

In [ ]:
# 全ネットワークの辺に E と geo を付ける（下流の探索用）
oe_full = omni_edges["除いた後"].set_index(["src", "dst"]); sg_full = edges_sg.set_index(["src", "dst"]).score
evd = net.copy()
key = pd.MultiIndex.from_frame(evd[["src", "dst"]])
evd["refs"] = oe_full.score.reindex(key).values; evd["signor"] = sg_full.reindex(key).values
A = np.log1p(evd.refs.fillna(0)) / np.log1p(oe_full.score.max()); B = evd.signor.fillna(0)
evd["E"] = (1 - (1 - A) * (1 - B)).round(4)
evd["geo"] = np.sqrt(evd.weight * evd.E)
evd = evd[~evd.src.isin(tfs)]                                                               # TF から出る辺を捨てる
print(f"辺: {len(evd)} | geo >= 0.25: {int((evd.geo >= 0.25).sum())} | AKT1 から出る辺: {int((evd.src == TARGET).sum())}（geo >= 0.25: {int(((evd.src == TARGET) & (evd.geo >= 0.25)).sum())}）")

## 21b. AKT1 から1ホップずつ下流へたどる
- 動作: `geo ≥ 0.25` の辺を、AKT1 から順向きに1ホップずつ（最大 `UP_HOPS` ホップ）たどる。前のホップで選んだ遺伝子は除き、TF に着いたらそこで止める。各遺伝子の「前の遺伝子」は、前の層からの辺のうち `E` が最大のもの
- 出力: hop ごとの遺伝子数と、そのうちの TF の数

In [ ]:
# 下流の探索（AKT1 -> TF）
outd = {u: g.sort_values("E", ascending=False) for u, g in evd[evd.geo >= 0.25].groupby("src")}
reached_d, kept_d, front = {TARGET: 0}, [], [TARGET]
for h in range(1, UP_HOPS + 1):
    new = []
    for u in front:
        if u not in outd: continue
        for r in outd[u][~outd[u].dst.isin(reached_d)].itertuples():
            kept_d.append(dict(hop=h, gene=r.dst, prev=u, effect=r.effect, E=r.E, weight=r.weight, geo=r.geo))
            if r.dst not in new: new.append(r.dst)
    for g in new: reached_d[g] = h
    front = [g for g in new if g not in tfs]                                                # TF から先へは進まない
kept_d = pd.DataFrame(kept_d)
down = kept_d.sort_values("E").groupby("gene").tail(1).set_index("gene")                      # 各遺伝子について E が最大の辺を「前の遺伝子」にする
down["role"] = [("TF" if g in tfs else "intermediate") for g in down.index]
ps_d = {TARGET: 1.0}
for g in down.sort_values("hop").index: ps_d[g] = down.E[g] * ps_d[down.prev[g]]            # 経路スコア（E の積）
down["path_score"] = [ps_d[g] for g in down.index]
tf_reached = [g for g in down.index if down.role[g] == "TF"]
print(f"AKT1 の下流の遺伝子: {len(down)}（TF {len(tf_reached)} 個）")
down.groupby(["hop", "role"]).size().unstack(fill_value=0)

## 21c. TF の影響を計算して、描く遺伝子を選ぶ
- 動作: 到達した各 TF について、AKT1 からの最良経路が通る遺伝子に経路スコアを足して流れを求め、TF の影響（媒介する遺伝子の流れ × `DECAY` ^ （ステップ数 − 1）、直接の辺は 1、同点は経路スコア）を計算する。影響の大きい上位 `N_TF` 個の TF と、その経路の遺伝子、さらに hop ごとに流れの大きい順に `N_PER_HOP` 個まで選ぶ
- 出力: TF の上位の表、列ごとの遺伝子数

In [ ]:
# TF の影響と流れ
def path_from_target(g):                                                                   # AKT1 から g までの最良経路（AKT1 ... g）
    pth = [g]
    while pth[-1] != TARGET: pth.append(down.prev[pth[-1]])
    return pth[::-1]
flow_d, n_tf = {}, {}
for tf_ in tf_reached:
    for g in path_from_target(tf_):
        flow_d[g] = flow_d.get(g, 0) + ps_d[tf_]; n_tf[g] = n_tf.get(g, 0) + 1
fmax_d = max(v for g, v in flow_d.items() if g != TARGET)
rows = []
for tf_ in tf_reached:
    pth = path_from_target(tf_); mids = pth[1:-1]; steps = len(pth) - 1
    mf = 1.0 if not mids else float(np.mean([flow_d[m] / fmax_d for m in mids]))
    rows.append(dict(TF=tf_, steps=steps, mediator_flow=round(mf, 3), impact=mf * DECAY ** (steps - 1), path_score=ps_d[tf_], path=" > ".join(pth)))
tf_imp = pd.DataFrame(rows).sort_values(["impact", "path_score"], ascending=False).reset_index(drop=True)
tf_imp["impact"] = tf_imp.impact.round(4); tf_imp["path_score"] = tf_imp.path_score.round(4)
sel_tf = list(tf_imp.TF[:N_TF])

on_path_d = set()
for t_ in sel_tf: on_path_d.update(path_from_target(t_))
mid_d = {h: [g for g in on_path_d if g not in tfs and g != TARGET and down.hop[g] == h] for h in range(1, UP_HOPS + 1)}     # 選んだ TF の経路の遺伝子は必ず入れる
for h in range(1, UP_HOPS + 1):                                                            # 足りない分は、hop ごとに流れの大きい順（前の遺伝子が描くものに限る）
    shown_prev = set(mid_d.get(h - 1, [])) | ({TARGET} if h == 1 else set())
    extra = sorted([g for g in down.index if down.role[g] == "intermediate" and down.hop[g] == h and g not in mid_d[h] and down.prev[g] in shown_prev], key=lambda g: -flow_d.get(g, 0))
    mid_d[h] += extra[:max(0, N_PER_HOP - len(mid_d[h]))]
cols_d = [("seed", [TARGET])] + [(f"hop {h}", mid_d[h]) for h in range(1, UP_HOPS + 1) if mid_d[h]] + [("transcription factors", sel_tf)]
print({n: len(g_) for n, g_ in cols_d})
tf_imp.head(15)

## 21d. 図（AKT1 → 転写因子）
- 動作: 左から seed（AKT1）| hop 1、2、… | 転写因子の順に並べ、曲線で描く（促進 = 濃い青の実線と矢じり、抑制 = 赤の破線と「T」字）。TF は hop にかかわらず右端の列に置く。縁取りは、AKT1 からその遺伝子までの最良経路の符号
- 出力: 図（画面と HTML・PNG に保存）。凡例つき

In [ ]:
# AKT1 の下流の図（Plotly）
shown_d = [g for n, g_ in cols_d for g in g_]
x_d = {g: ci for ci, (n, g_) in enumerate(cols_d) for g in g_}
cand_d = evd[(evd.geo >= 0.25) & evd.src.isin(shown_d) & evd.dst.isin(shown_d) & (evd.src != evd.dst)].copy()
cand_d = cand_d[cand_d.src.map(x_d) < cand_d.dst.map(x_d)].sort_values("E", ascending=False).groupby("src").head(MAX_OUT)       # 左から右へ、各遺伝子から最大 MAX_OUT 本
tree_d = [(down.prev[g], g) for g in shown_d if g != TARGET and down.prev[g] in x_d and x_d[down.prev[g]] < x_d[g]]                # 最良経路の辺は必ず入れる
eff_d = evd.set_index(["src", "dst"]).effect
edges_d = {(a, b) for a, b in zip(cand_d.src, cand_d.dst)} | set(tree_d)

y_d = {TARGET: 0.0}
for ci in range(1, len(cols_d)):                                                           # 左の列から順に、つながる元の位置の平均で並べる
    genes = cols_d[ci][1]
    for i, g in enumerate(sorted(genes, key=lambda g: (np.mean([y_d[a] for a, b in edges_d if b == g and a in y_d] or [1e9]), g))): y_d[g] = (len(genes) - 1) / 2 - i

def sign_to(g):                                                                            # AKT1 からその遺伝子までの最良経路の符号
    s = 1
    for a, b in zip(path_from_target(g)[:-1], path_from_target(g)[1:]): s *= eff_d[(a, b)]
    return s
sgn_d = {g: sign_to(g) for g in shown_d if g != TARGET}
wrap_columns(cols_d, x_d, y_d)
ncol, nmax = len(cols_d), max(rows_needed(len(g_)) for n, g_ in cols_d)
W, Hpx = 1500, max(700, 48 * nmax + 230)
xr, yr = (-0.4, ncol - 0.2), (-nmax / 2 - 1.5, nmax / 2 + 2.5)
sx, sy = (W - 40) / (xr[1] - xr[0]), (Hpx - 140) / (yr[1] - yr[0])
fig = go.Figure()
for e_, (kname, kcol, kdash) in kinds.items():
    xs, ys, hx, hy, hang = [], [], [], [], []
    for a, b in edges_d:
        if eff_d[(a, b)] != e_: continue
        (x0, y0), (x1, y1) = (x_d[a], y_d[a]), (x_d[b], y_d[b]); dx = x1 - x0
        P = np.array([[x0, y0], [x0 + 0.5 * dx, y0], [x1 - 0.5 * dx, y1], [x1, y1]])
        pt = lambda t: (1 - t) ** 3 * P[0] + 3 * (1 - t) ** 2 * t * P[1] + 3 * (1 - t) * t ** 2 * P[2] + t ** 3 * P[3]
        dv = lambda t: 3 * (1 - t) ** 2 * (P[1] - P[0]) + 6 * (1 - t) * t * (P[2] - P[1]) + 3 * t ** 2 * (P[3] - P[2])
        pts = np.array([pt(t) for t in np.linspace(0, 1, 28)]); xs += list(pts[:, 0]) + [None]; ys += list(pts[:, 1]) + [None]
        te = 0.93 if e_ == -1 else 0.88; p_, v_ = pt(te), dv(te); th = math.degrees(math.atan2(v_[0] * sx, v_[1] * sy))
        hx.append(p_[0]); hy.append(p_[1]); hang.append(th - 90 if e_ == -1 else th)
    if not xs: continue
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=kcol, width=1.3, dash=kdash), opacity=0.75, hoverinfo="skip", name=f"{kname} edge", legendgroup=kname))
    fig.add_trace(go.Scatter(x=hx, y=hy, mode="markers", hoverinfo="skip", showlegend=False, legendgroup=kname,
                             marker=dict(symbol="line-ns" if e_ == -1 else "arrow", size=11, angle=hang, angleref="up", color=kcol, line=dict(color=kcol, width=2.5) if e_ == -1 else None)))
add_gene_nodes(fig, [g for g in shown_d if g != TARGET], x_d, y_d)
fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers+text", text=[TARGET], textposition="top center", textfont=dict(size=15), name="seed (target)", marker=dict(size=22, color="#555555", symbol="diamond"), hoverinfo="skip"))
for ci, (n, genes) in enumerate(cols_d): fig.add_annotation(x=ci, y=yr[1] - 0.6, text=n, showarrow=False, font=dict(size=14, color="#666666"))
fig.update_layout(width=W, height=Hpx, plot_bgcolor="white", margin=dict(l=20, r=20, t=30, b=110), xaxis=dict(visible=False, range=list(xr)), yaxis=dict(visible=False, range=list(yr)),
                  legend=dict(orientation="h", yanchor="top", y=-0.01, xanchor="center", x=0.5, font=dict(size=11)))
fig.write_html(f"network_development_{TARGET}_downstream.html", include_plotlyjs="cdn")
try: fig.write_image(f"network_development_{TARGET}_downstream.png", scale=1.5)
except Exception as e: print("PNG は保存できなかった:", str(e)[:80])
fig.show()

## 21e. 下流の遺伝子リスト（スコアと注釈つき）
- 動作: 到達した全遺伝子について、経路・スコア・流れ・TF の影響に、UniProt・HPA・DepMap の注釈を付けて CSV にする
- 出力: 上位 30 遺伝子の表、CSV ファイル（`network_development_AKT1_downstream_genes.csv`）

In [ ]:
# 下流の遺伝子リスト
cod = codep.set_index("gene")
rows = []
for g in down.index:
    pth = path_from_target(g); sg = 1
    for a, b in zip(pth[:-1], pth[1:]): sg *= eff_d[(a, b)]
    ed_ = evd.set_index(["src", "dst"]).loc[(down.prev[g], g)]
    rows.append(dict(gene=g, role=down.role[g], hop=int(down.hop[g]), prev_gene=down.prev[g], path=" > ".join(pth), path_sign={1: "activated by AKT1", -1: "inhibited by AKT1", 0: "unknown"}[sg],
                     path_score=round(ps_d[g], 4), E_from_prev=round(ed_.E, 3), weight_from_prev=round(ed_.weight, 3), geo_from_prev=round(ed_.geo, 3), refs_from_prev=ed_.refs, signor_from_prev=ed_.signor,
                     source_from_prev=ed_.source, mechanism_from_prev=str(ed_.mechanism)[:90], residue_from_prev=ed_.residue,
                     flow=round(flow_d.get(g, 0.0), 4), n_TFs=int(n_tf.get(g, 0)),
                     TF_impact=float(tf_imp.set_index("TF").impact.get(g, np.nan)) if g in tfs else np.nan, in_figure=g in set(shown_d)))
down_list = pd.DataFrame(rows)
down_list["locations"] = down_list.gene.map(loc_txt).fillna(""); down_list["uniprot_flags"] = down_list.gene.map(flag_txt).fillna("")
down_list = down_list.join(hg, on="gene")
down_list["depmap_mean_effect"] = down_list.gene.map(cod.mean_effect); down_list["depmap_corr_AKT1"] = down_list.gene.map(cod.pearson_r)
down_list = down_list.sort_values(["flow", "path_score"], ascending=False).reset_index(drop=True)
down_list.to_csv(f"network_development_{TARGET}_downstream_genes.csv", index=False, encoding="utf-8-sig")
print(f"保存: network_development_{TARGET}_downstream_genes.csv | {down_list.shape}")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
down_list[["gene", "role", "hop", "prev_gene", "path_sign", "path_score", "E_from_prev", "flow", "n_TFs", "TF_impact", "locations", "uniprot_flags", "depmap_mean_effect", "depmap_corr_AKT1", "in_figure"]].head(30)

## Step 22. 選択肢 4（ハブ遺伝子ごとにまとめる）を実際に作る（AKT1 の下流）
> 指示: how about 選択肢4は実際にどうなりますか？（what if TF selection is same as receptors）

### 選択肢 4 の中身
- 各 TF を、AKT1 からの**最良経路の最初の遺伝子（hop 1 のハブ）**でグループに分ける。AKT1 に直接つながる TF は「直接」のグループ。
- ハブは、その先の TF の数が多い順に上位 `N_HUBS` 個。ハブごとに TF を経路スコアの大きい順に `TF_PER_HUB` 個、直接のグループは `N_DIRECT` 個描く。
- 図は 3 列: **AKT1 | ハブ遺伝子 | TF（ハブごとのかたまり）**。表には、ハブの先の TF の数、ハブの機能の印（UniProt）、ハブ → TF の辺の仕組みを出す。
- **ハブの選び方は「TF の数」だけで、ノイズの除去はしていない**（選択肢 5 は含まない）。そのまま出すとどうなるかを見る。

**Note（TF の選び方を受容体と同じにしたら）**
- Step 21 の TF の影響は、受容体の影響（Step 16）と**同じ式**（媒介する遺伝子の流れ × `DECAY` ^ （ステップ数 − 1）、直接の辺が最高、同点は経路スコア）。違いはデータにある: **受容体で直接 AKT1 につながるのは RET の 1 個だけ**だが、**TF は 30 個が直接**。そのため、同じ式でも TF では上位 15 個がすべて直接になり、間接の TF が出ない。

## 22a. ハブごとの表
- 動作: 到達した各 TF の最良経路から、最初の遺伝子（ハブ）を求めて、ハブごとに TF の数・経路スコアの平均・機能の印・辺の仕組みを集計する
- 出力: ハブの表（TF の数が多い上位 15 個）

In [ ]:
# 各 TF を、最良経路の最初の遺伝子（ハブ）でグループに分ける
import collections
first_hub = {}
for t_ in tf_reached:
    pth = path_from_target(t_)
    first_hub[t_] = pth[1] if len(pth) > 2 else "(direct)"                                  # 直接の辺なら "(direct)"
tf_tab = pd.DataFrame({"TF": tf_reached, "hub": [first_hub[t] for t in tf_reached], "path_score": [ps_d[t] for t in tf_reached]})
evi = evd.set_index(["src", "dst"])

def edge_mech(a, b):                                                                        # 辺の仕組み（omnipath の出典名と complex を除いたもの）
    m = [x for x in str(evi.loc[(a, b)].mechanism).split(";") if x and not x.startswith("omnipath:") and x != "complex"]
    return m[0] if m else "(source only)"
rows = []
for h_, g in tf_tab[tf_tab.hub != "(direct)"].groupby("hub"):
    mech = collections.Counter(edge_mech(h_, t) for t in g.TF if (h_, t) in evi.index).most_common(2)
    rows.append(dict(hub=h_, n_TFs=len(g), n_TFs_one_step_from_hub=int(sum((h_, t) in evi.index for t in g.TF)), mean_path_score=round(g.path_score.mean(), 3), E_AKT1_to_hub=round(evi.loc[(TARGET, h_)].E, 3),
                     hub_flags=flag_txt.get(h_, ""), hub_to_TF_mechanism=", ".join(f"{m}({n})" for m, n in mech), example_TFs=", ".join(g.sort_values("path_score", ascending=False).TF[:5])))
hub_tab = pd.DataFrame(rows).sort_values("n_TFs", ascending=False).reset_index(drop=True)
print("直接の TF:", int((tf_tab.hub == "(direct)").sum()), "個 | ハブ（hop 1 の遺伝子）の数:", len(hub_tab), "| 上位 10 ハブが持つ TF:", int(hub_tab.n_TFs.head(10).sum()), "/", int((tf_tab.hub != '(direct)').sum()))
pd.set_option("display.max_colwidth", 70)
hub_tab.head(15)

## 22b. 図（AKT1 → ハブ → TF）
- 動作: 直接の TF のグループと、上位ハブごとの TF のグループを縦に並べ、AKT1 から、ハブ経由または直接で TF まで曲線で描く。ハブ → TF の曲線は、**間に別の遺伝子を通ることがある**（1 ステップとは限らない。何ステップかは表の `n_TFs_one_step_from_hub` で分かる）。縁取りは AKT1 からそのハブ・TF までの最良経路の符号
- 出力: 図（画面と HTML・PNG に保存）

In [ ]:
# 選択肢 4: ハブごとの TF の図（Plotly）
hubs_sel = list(hub_tab.hub[:N_HUBS])
groups = [("(direct)", list(tf_tab[tf_tab.hub == "(direct)"].sort_values("path_score", ascending=False).TF[:N_DIRECT]))] + \
         [(h_, list(tf_tab[tf_tab.hub == h_].sort_values("path_score", ascending=False).TF[:TF_PER_HUB])) for h_ in hubs_sel]
yy, y4, hub_y, cur = {}, {}, {}, 0.0
for h_, tfl in groups:
    for t_ in tfl: y4[t_] = cur; cur -= 1.0
    if h_ != "(direct)": hub_y[h_] = np.mean([y4[t_] for t_ in tfl])
    cur -= 0.8                                                                               # グループの間のすき間
shift = (max(y4.values()) + min(y4.values())) / 2
y4 = {k: v - shift for k, v in y4.items()}; hub_y = {k: v - shift for k, v in hub_y.items()}
x4 = {TARGET: 0, **{h_: 1 for h_ in hubs_sel}, **{t_: 2 for t_ in y4}}; y_all = {TARGET: 0.0, **hub_y, **y4}
edges4 = [(TARGET, h_) for h_ in hubs_sel] + [(h_, t_) for h_, tfl in groups if h_ != "(direct)" for t_ in tfl] + [(TARGET, t_) for t_ in groups[0][1]]

# ハブ → TF の曲線は、1 ステップとは限らない（間に別の遺伝子を通ることがある）。符号はハブから TF までの経路の符号の積
eff4 = {(a, b): eff_d[(a, b)] for a, b in edges4 if a == TARGET}
for a, b in edges4:
    if a != TARGET: eff4[(a, b)] = sign_to(b) * sign_to(a)
ys_ = list(y_all.values()); yr = (min(ys_) - 1.5, max(ys_) + 2.5); xr = (-0.3, 2.9)
W, Hpx = 1100, max(700, 24 * (max(ys_) - min(ys_)) + 240); sx, sy = (W - 40) / (xr[1] - xr[0]), (Hpx - 140) / (yr[1] - yr[0])
def sg4(g): return sgn_d.get(g) if g in sgn_d else sign_to(g)
fig = go.Figure()
for e_, (kname, kcol, kdash) in kinds.items():
    xs, ys, hx, hy, hang = [], [], [], [], []
    for a, b in edges4:
        if eff4[(a, b)] != e_: continue
        (x0, y0), (x1, y1) = (x4[a], y_all[a]), (x4[b], y_all[b]); dx = x1 - x0
        P = np.array([[x0, y0], [x0 + 0.5 * dx, y0], [x1 - 0.5 * dx, y1], [x1, y1]])
        pt = lambda t: (1 - t) ** 3 * P[0] + 3 * (1 - t) ** 2 * t * P[1] + 3 * (1 - t) * t ** 2 * P[2] + t ** 3 * P[3]
        dv = lambda t: 3 * (1 - t) ** 2 * (P[1] - P[0]) + 6 * (1 - t) * t * (P[2] - P[1]) + 3 * t ** 2 * (P[3] - P[2])
        pts = np.array([pt(t) for t in np.linspace(0, 1, 28)]); xs += list(pts[:, 0]) + [None]; ys += list(pts[:, 1]) + [None]
        te = 0.93 if e_ == -1 else 0.88; p_, v_ = pt(te), dv(te); th = math.degrees(math.atan2(v_[0] * sx, v_[1] * sy))
        hx.append(p_[0]); hy.append(p_[1]); hang.append(th - 90 if e_ == -1 else th)
    if not xs: continue
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=kcol, width=1.4, dash=kdash), opacity=0.8, hoverinfo="skip", name=f"{kname} edge", legendgroup=kname))
    fig.add_trace(go.Scatter(x=hx, y=hy, mode="markers", hoverinfo="skip", showlegend=False, legendgroup=kname,
                             marker=dict(symbol="line-ns" if e_ == -1 else "arrow", size=11, angle=hang, angleref="up", color=kcol, line=dict(color=kcol, width=2.5) if e_ == -1 else None)))
def nodes4(genes, name, fill, size=14):
    if genes: fig.add_trace(go.Scatter(x=[x4[g] for g in genes], y=[y_all[g] for g in genes], mode="markers+text", text=[f"{g}  ({int(hub_tab.set_index('hub').n_TFs[g])} TFs)" if g in hubs_sel else g for g in genes],
                                       textposition="middle right", textfont=dict(size=12),
                                       marker=dict(size=size, color=fill, line=dict(color=[{1: "#3f8f5f", -1: "#c0504d", 0: "#a9a9a9"}[sg4(g)] for g in genes], width=2.5)), name=name,
                                       hovertext=[f"<b>{g}</b><br>{name}<br>path score: {ps_d[g]:.3f}" for g in genes], hoverinfo="text"))
nodes4(hubs_sel, "hub gene (first gene after AKT1 on the best path to the TFs)", "#a9a9a9")
nodes4([t for g_, tfl in groups for t in tfl], "transcription factor", "#d9a441", 15)
fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers+text", text=[TARGET], textposition="top center", textfont=dict(size=15), name="seed (target)", marker=dict(size=22, color="#555555", symbol="diamond"), hoverinfo="skip"))
fig.add_annotation(x=0, y=yr[1] - 0.6, text="seed", showarrow=False, font=dict(size=14, color="#666666")); fig.add_annotation(x=1, y=yr[1] - 0.6, text="hub genes", showarrow=False, font=dict(size=14, color="#666666"))
fig.add_annotation(x=2.0, y=yr[1] - 0.6, text="transcription factors (top group: direct targets)", showarrow=False, font=dict(size=14, color="#666666"))
fig.update_layout(width=W, height=Hpx, plot_bgcolor="white", margin=dict(l=20, r=20, t=30, b=110), xaxis=dict(visible=False, range=list(xr)), yaxis=dict(visible=False, range=list(yr)),
                  legend=dict(orientation="h", yanchor="top", y=-0.01, xanchor="center", x=0.5, font=dict(size=11)))
fig.write_html(f"network_development_{TARGET}_downstream_hubs.html", include_plotlyjs="cdn")
try: fig.write_image(f"network_development_{TARGET}_downstream_hubs.png", scale=1.5)
except Exception as e: print("PNG は保存できなかった:", str(e)[:80])
fig.show()

## Step 23. 下流を、注目する TF（HIF1A、FOXO1、SMAD2、SMAD3）への流れに限る
> 指示: what if you choose the flow from AKT1 and HIF1A, FOXO1, SMAD2/3（SMAP2/3 は SMAD2/3 と解釈）

### 考え方（Step 19 の KDR と同じ作り方を、下流に当てはめる）
- 上流で受容体を KDR だけに限ったように、**終点を `FOCUS_TFS` の 4 つの TF だけ**にして、**AKT1 からそれらの TF までの経路が通る遺伝子と経路**だけを取り出す。
- 条件は Step 21 と同じ（`geo ≥ 0.25` の辺、全ネットワーク）。AKT1 から各 TF まで最大 `MAX_STEPS_TF`（既定 4）ステップの**単純な経路をすべて**数える。経路のスコアは辺の `E` の積、符号は辺の符号の積。
- したがって、図の中間の遺伝子は**すべて、選んだ TF への経路上にある**（Step 21 のように、全 TF への流れで選ぶことはしない）。

**Note**
- 4 つの TF は、ネットワークにあるか・AKT1 から何ステップで着くかを 23a で確認する。ネットワークにない、または届かない TF があれば、そう表示する。
- SMAD2 は Lambert の TF 一覧に入っていなければ、他の遺伝子と同じ「途中の遺伝子」として扱われる（この図では終点として扱う）。

## 23a. 注目する TF がネットワークにあるか、何ステップで届くか
- 動作: `geo ≥ 0.25` の辺のグラフで、AKT1 から各 TF までの最短ステップ数と、最大ステップ数ごとに経路が通る遺伝子の数を数える
- 出力: TF ごとの表

In [ ]:
# 注目する TF への最短ステップ数と、経路が通る遺伝子の数
Gd = nx.DiGraph()
for r in evd[evd.geo >= 0.25].itertuples(): Gd.add_edge(r.src, r.dst, E=r.E, effect=r.effect)
dA = nx.single_source_shortest_path_length(Gd, TARGET, cutoff=8)                              # AKT1 からのステップ数
d_to_t = {t: nx.single_source_shortest_path_length(Gd.reverse(copy=False), t, cutoff=8) for t in FOCUS_TFS if t in Gd}
rows = []
for t in FOCUS_TFS:
    r_ = dict(TF=t, in_network=t in Gd, in_Lambert_TF_list=t in tfs, steps_from_AKT1=dA.get(t, "-"))
    for K in range(2, 7):
        r_[f"genes (<= {K} steps)"] = len({g for g in dA if t in d_to_t and g in d_to_t[t] and dA[g] + d_to_t[t][g] <= K}) if t in d_to_t else 0
    rows.append(r_)
pd.DataFrame(rows).set_index("TF")

## 23b. AKT1 から各 TF への経路をすべて数える
- 動作: AKT1 から各 TF まで最大 `MAX_STEPS_TF` ステップの単純な経路をすべて数え、スコアと符号を付ける
- 出力: TF ごとの経路の数と、スコア上位の経路

In [ ]:
# AKT1 -> 注目する TF の経路をすべて数える
prow = []
for t in FOCUS_TFS:
    if t not in d_to_t or t not in dA: continue
    S_t = {g for g in dA if g in d_to_t[t] and dA[g] + d_to_t[t][g] <= MAX_STEPS_TF}
    for pth in nx.all_simple_paths(Gd.subgraph(S_t), TARGET, t, cutoff=MAX_STEPS_TF):
        sc, sg = 1.0, 1
        for a, b in zip(pth[:-1], pth[1:]): sc *= Gd[a][b]["E"]; sg *= Gd[a][b]["effect"]
        prow.append(dict(TF=t, path=" > ".join(pth), steps=len(pth) - 1, score=round(sc, 4), sign={1: "activated by AKT1", -1: "inhibited by AKT1", 0: "unknown"}[sg]))
paths23 = pd.DataFrame(prow).sort_values(["TF", "score"], ascending=[True, False]).reset_index(drop=True)
display(paths23.groupby("TF").agg(paths=("path", "size"), best_score=("score", "max"), activated=("sign", lambda s: int((s == "activated by AKT1").sum())),
                                   inhibited=("sign", lambda s: int((s == "inhibited by AKT1").sum())), unknown=("sign", lambda s: int((s == "unknown").sum()))))
paths23.groupby("TF").head(5)

## 23c. 注目する TF への経路が通る遺伝子のリスト
- 動作: 経路が通る遺伝子ごとに、AKT1 からのステップ数、通る経路の数（TF ごと）、最良経路のスコアと符号を求め、Step 21 の注釈と全 TF への流れを付ける
- 出力: 遺伝子の表（CSV に保存）

In [ ]:
# 注目する TF への経路が通る遺伝子のリスト
st = {}
for r in paths23.itertuples():
    pth = r.path.split(" > ")
    sg = {"activated by AKT1": 1, "inhibited by AKT1": -1, "unknown": 0}[r.sign]
    for g in pth:
        s_ = st.setdefault(g, dict(n_paths=0, best=0.0, best_sign=0, best_path="", **{f"n_{t}": 0 for t in FOCUS_TFS}))
        s_["n_paths"] += 1; s_[f"n_{r.TF}"] += 1
        if r.score > s_["best"]: s_["best"], s_["best_sign"], s_["best_path"] = r.score, sg, r.path
rows = [dict(gene=g, role="seed" if g == TARGET else ("TF" if g in tfs else "intermediate"), focus_TF=g in FOCUS_TFS, steps_from_AKT1=dA[g], n_paths=v["n_paths"],
             **{f"paths_to_{t}": v[f"n_{t}"] for t in FOCUS_TFS}, best_path_score=round(v["best"], 4), best_path_sign={1: "activated by AKT1", -1: "inhibited by AKT1", 0: "unknown"}[v["best_sign"]],
             best_path_through=v["best_path"]) for g, v in st.items()]
list23 = pd.DataFrame(rows)
annd = down_list.set_index("gene")[["flow", "n_TFs", "locations", "uniprot_flags", "n_tissues_expressed", "max_nTPM", "depmap_mean_effect", "depmap_corr_AKT1"]]
list23 = list23.join(annd.rename(columns={"flow": "flow_all_TFs", "n_TFs": "n_TFs_all"}), on="gene")
list23 = list23.sort_values(["n_paths", "best_path_score"], ascending=False).reset_index(drop=True)
list23.to_csv(f"network_development_{TARGET}_downstream_focusTF.csv", index=False, encoding="utf-8-sig")
print(f"遺伝子: {len(list23)}（TF {int(list23.focus_TF.sum())} 個を含む）| 保存: network_development_{TARGET}_downstream_focusTF.csv")
list23.head(25)

## 23d. 図（AKT1 → 注目する TF）
- 動作: 経路が通る遺伝子は約 300 個あり、全部描くと読めない。そこで、**注目する TF への流れ（その遺伝子を通る全経路のスコアの合計）の大きい順に、ステップごとに `N_PER_HOP` 個まで**選ぶ。TF の列から AKT1 の側へ選び、右の列に辺がある遺伝子だけを残す。さらに、AKT1 からつながらない遺伝子を除く
- 辺は、描く遺伝子どうしの経路の辺のうち、各遺伝子から出る上位 `MAX_OUT` 本と、各遺伝子に入る最良の辺。曲線で描く（促進 = 濃い青の実線と矢じり、抑制 = 赤の破線と「T」字）。縁取りは、その遺伝子を通る最良経路の符号
- 出力: 図（画面と HTML・PNG に保存）

In [ ]:
# AKT1 から注目する TF への経路の図（Plotly）
S23 = set(st)
tf_cols = [t for t in FOCUS_TFS if t in S23]
# 流れ（flow）= その遺伝子を通る全経路のスコアの合計（注目する TF への経路だけで数える）
flow23 = {}
for r in paths23.itertuples():
    for g in r.path.split(" > "): flow23[g] = flow23.get(g, 0.0) + r.score
all_edges = {tuple(pr) for r in paths23.itertuples() for pr in zip(r.path.split(" > ")[:-1], r.path.split(" > ")[1:])}
mids_all = [g for g in S23 if g != TARGET and g not in tf_cols]
# TF の列から AKT1 の側へ、ステップごとに、右の列の遺伝子へ辺がある遺伝子を、流れの大きい順に N_PER_HOP 個ずつ選ぶ
shown23 = set(tf_cols) | {TARGET}; picked = {}
for k in range(MAX_STEPS_TF - 1, 0, -1):
    cand = [g for g in mids_all if dA[g] == k and any((g, b) in all_edges and b in shown23 and (dA[b] > k or b in tf_cols) for b in Gd.successors(g))]
    picked[k] = sorted(cand, key=lambda g: -flow23[g])[:N_PER_HOP]; shown23 |= set(picked[k])
# 左の列から AKT1 までつながらない遺伝子を除く（入ってくる辺が、描く遺伝子にないもの）
changed = True
while changed:
    changed = False
    for k in sorted(picked):
        keep = [g for g in picked[k] if any((a, g) in all_edges and (a == TARGET or a in shown23) and dA[a] < k for a in Gd.predecessors(g))]
        if len(keep) != len(picked[k]): changed = True; shown23 -= set(picked[k]) - set(keep); picked[k] = keep
cols23 = [("seed", [TARGET])] + [(f"step {k} from {TARGET}", sorted(picked[k])) for k in range(1, MAX_STEPS_TF) if picked.get(k)] + [("focus TFs", tf_cols)]
x23 = {g: ci for ci, (n, g_) in enumerate(cols23) for g in g_}
mids23 = [g for g in x23 if g != TARGET and g not in tf_cols]
cand_e = [(a, b) for a, b in all_edges if a in x23 and b in x23 and x23[a] < x23[b]]
best_in = {}; out_rank = {}
for a, b in sorted(cand_e, key=lambda e: -Gd[e[0]][e[1]]["E"]):
    out_rank[a] = out_rank.get(a, 0) + 1; out_rank[(a, b)] = out_rank[a]
    best_in.setdefault(b, (a, b))
edges23 = {(a, b) for a, b in cand_e if out_rank[(a, b)] <= MAX_OUT} | set(best_in.values())     # 各遺伝子から出る上位 MAX_OUT 本 + 各遺伝子に入る最良の辺
print(f"描く遺伝子: {len(mids23)}（全体 {len(mids_all)} 個から）| 辺: {len(edges23)}（全経路の辺 {len(all_edges)} 本から）")
y23 = {TARGET: 0.0}
for ci in range(1, len(cols23)):
    for i, g in enumerate(sorted(cols23[ci][1], key=lambda g: (np.mean([y23[a] for a, b in edges23 if b == g and a in y23] or [1e9]), g))): y23[g] = (len(cols23[ci][1]) - 1) / 2 - i
sign23 = {g: st[g]["best_sign"] for g in S23}
wrap_columns(cols23, x23, y23)
ncol, nmax = len(cols23), max(rows_needed(len(g_)) for n, g_ in cols23)
W, Hpx = 1400, max(620, 48 * nmax + 230)
xr, yr = (-0.4, ncol - 0.2), (-nmax / 2 - 1.5, nmax / 2 + 2.5)
sx, sy = (W - 40) / (xr[1] - xr[0]), (Hpx - 140) / (yr[1] - yr[0])
fig = go.Figure()
for e_, (kname, kcol, kdash) in kinds.items():
    xs, ys, hx, hy, hang = [], [], [], [], []
    for a, b in edges23:
        if Gd[a][b]["effect"] != e_: continue
        (x0, y0), (x1, y1) = (x23[a], y23[a]), (x23[b], y23[b]); dx = x1 - x0
        P = np.array([[x0, y0], [x0 + 0.5 * dx, y0], [x1 - 0.5 * dx, y1], [x1, y1]])
        pt = lambda t: (1 - t) ** 3 * P[0] + 3 * (1 - t) ** 2 * t * P[1] + 3 * (1 - t) * t ** 2 * P[2] + t ** 3 * P[3]
        dv = lambda t: 3 * (1 - t) ** 2 * (P[1] - P[0]) + 6 * (1 - t) * t * (P[2] - P[1]) + 3 * t ** 2 * (P[3] - P[2])
        pts = np.array([pt(t) for t in np.linspace(0, 1, 28)]); xs += list(pts[:, 0]) + [None]; ys += list(pts[:, 1]) + [None]
        te = 0.93 if e_ == -1 else 0.88; p_, v_ = pt(te), dv(te); th = math.degrees(math.atan2(v_[0] * sx, v_[1] * sy))
        hx.append(p_[0]); hy.append(p_[1]); hang.append(th - 90 if e_ == -1 else th)
    if not xs: continue
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=kcol, width=1.5, dash=kdash), opacity=0.8, hoverinfo="skip", name=f"{kname} edge", legendgroup=kname))
    fig.add_trace(go.Scatter(x=hx, y=hy, mode="markers", hoverinfo="skip", showlegend=False, legendgroup=kname,
                             marker=dict(symbol="line-ns" if e_ == -1 else "arrow", size=11, angle=hang, angleref="up", color=kcol, line=dict(color=kcol, width=2.5) if e_ == -1 else None)))
add_gene_nodes(fig, [g for g in x23 if g != TARGET], x23, y23, text_size=12)
fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers+text", text=[TARGET], textposition="top center", textfont=dict(size=15), name="seed (target)", marker=dict(size=22, color="#555555", symbol="diamond"), hoverinfo="skip"))
for ci, (n, genes) in enumerate(cols23): fig.add_annotation(x=ci, y=yr[1] - 0.6, text=n, showarrow=False, font=dict(size=14, color="#666666"))
fig.update_layout(width=W, height=Hpx, plot_bgcolor="white", margin=dict(l=20, r=20, t=30, b=110), xaxis=dict(visible=False, range=list(xr)), yaxis=dict(visible=False, range=list(yr)),
                  legend=dict(orientation="h", yanchor="top", y=-0.01, xanchor="center", x=0.5, font=dict(size=11)))
fig.write_html(f"network_development_{TARGET}_downstream_focusTF.html", include_plotlyjs="cdn")
try: fig.write_image(f"network_development_{TARGET}_downstream_focusTF.png", scale=1.5)
except Exception as e: print("PNG は保存できなかった:", str(e)[:80])
fig.show()

## Step 24. DepMap の相関をしきい値にして遺伝子を選び、図にする（上流と下流）
> 指示: Please try to visualize this（DepMap の相関をしきい値に使った場合）

### 選び方
- **選ぶ遺伝子**: AKT1 との DepMap の相関が `|r| ≥ CORR_MIN`（既定 0.1）で、FDR < 0.05、細胞株が 1,000 以上の遺伝子。上流は Step 18 の 1,438 遺伝子、下流は Step 21 の 3,773 遺伝子から選ぶ。
- **つなぎの遺伝子**: 選んだ遺伝子だけだと、AKT1 とつながらない。そこで、各遺伝子から AKT1 までの最良経路（Step 11 / 21 の「次の遺伝子」をたどった経路）の途中の遺伝子を**薄い色**で足す。薄い点は「しきい値を満たさないが、つなぐために描いた遺伝子」。
- 点の見た目は Step 17 以降と同じ（大きさ = AKT1 との相関の大きさ、縁は同じ、色 = 機能）。辺は、最良経路の辺と、描く遺伝子どうしの `geo ≥ 0.25` の辺（各遺伝子から最大 `MAX_OUT` 本）。

**Note**
- 受容体・リガンド・TF は、DepMap の相関が低いものが大半なので、しきい値で選ぶと少なくなる（Step 前の数え直しで、上流は受容体 9 個、リガンド 6 個）。
- 選ばれた遺伝子は、相関が高い順に表にして保存する。

## 24a. 選ばれる遺伝子と、つなぎの遺伝子
- 動作: DepMap の表（Step 20）で条件を満たす遺伝子を、上流・下流のリストから選び、AKT1 までの最良経路の途中の遺伝子を「つなぎ」として足す
- 出力: 上流・下流の、選んだ遺伝子とつなぎの遺伝子の数、選んだ遺伝子の一覧

In [ ]:
# DepMap の相関で遺伝子を選び、つなぎの遺伝子を足す
cs_ = codep.set_index("gene")
ok_genes = set(cs_.index[(cs_.pearson_r.abs() >= CORR_MIN) & (cs_.FDR < 0.05) & (cs_.n_models >= 1000)])
sel_up24 = [g for g in fig_tbl.gene if g in ok_genes]                                      # 上流で選ばれた遺伝子
sel_dn24 = [g for g in down.index if g in ok_genes]                                        # 下流で選ばれた遺伝子
up_conn = set(); [up_conn.update(path_to_target(g)) for g in sel_up24]
dn_conn = set(); [dn_conn.update(path_from_target(g)) for g in sel_dn24]
up_conn -= set(sel_up24) | {TARGET}; dn_conn -= set(sel_dn24) | {TARGET}
print(f"条件を満たす遺伝子（|r| >= {CORR_MIN}、FDR < 0.05、細胞株 >= 1,000）: 全体 {len(ok_genes)} 個")
print(f"上流: 選ばれた {len(sel_up24)} 個（リガンド {sum(role(g) == 'リガンド' for g in sel_up24)}、受容体 {sum(role(g) == '受容体' for g in sel_up24)}）+ つなぎ {len(up_conn)} 個")
print(f"下流: 選ばれた {len(sel_dn24)} 個（TF {sum(g in tfs for g in sel_dn24)}）+ つなぎ {len(dn_conn)} 個")
pd.DataFrame({"upstream": pd.Series(sorted(sel_up24, key=lambda g: -abs(cs_.pearson_r[g]))[:15]), "r (up)": pd.Series([round(cs_.pearson_r[g], 2) for g in sorted(sel_up24, key=lambda g: -abs(cs_.pearson_r[g]))[:15]]),
              "downstream": pd.Series(sorted(sel_dn24, key=lambda g: -abs(cs_.pearson_r[g]))[:15]), "r (down)": pd.Series([round(cs_.pearson_r[g], 2) for g in sorted(sel_dn24, key=lambda g: -abs(cs_.pearson_r[g]))[:15]])})

## 24b. 描画の部品
- 動作: 列に分けた遺伝子と辺から、曲線の辺（促進 = 濃い青の実線と矢じり、抑制 = 赤の破線と「T」字）と、Step 17 以降と同じ点の見た目で描く関数 `draw_corr_figure`（上流と下流で2回使う）。選んだ遺伝子は濃く、つなぎの遺伝子は薄く描く
- 出力: なし（次のセルで使う）

In [ ]:
# 描画の部品（上流・下流で共通）
def draw_corr_figure(cols, edges, eff, selected, fname, direction):
    """cols: [(列の名前, 遺伝子のリスト)]、edges: {(a, b)}、eff: 辺の符号の辞書、selected: 選ばれた遺伝子の集合、direction: 'upstream'（AKT1 が右端）か 'downstream'（AKT1 が左端）"""
    x = {g: ci for ci, (n, g_) in enumerate(cols) for g in g_}
    edges = {(a, b) for a, b in edges if a in x and b in x and x[a] != x[b]}
    seed_ci = x[TARGET]; y = {TARGET: 0.0}
    order = range(len(cols) - 2, -1, -1) if direction == "upstream" else range(1, len(cols))
    for ci in order:                                                                         # AKT1 に近い列から順に、つながる先（元）の位置の平均で並べる
        genes = cols[ci][1]
        def key(g):
            t = [y[b if direction == "upstream" else a] for a, b in edges if (a == g and b in y and direction == "upstream") or (b == g and a in y and direction == "downstream")]
            return (np.mean(t) if t else 1e9, g)
        for i, g in enumerate(sorted(genes, key=key)): y[g] = (len(genes) - 1) / 2 - i
    wrap_columns(cols, x, y)
    ncol, nmax = len(cols), max(rows_needed(len(g_)) for n, g_ in cols)
    W, Hpx = 1500, max(640, 48 * nmax + 230)
    xr, yr = (-0.4, ncol - 0.2), (-nmax / 2 - 1.5, nmax / 2 + 2.5)
    sx, sy = (W - 40) / (xr[1] - xr[0]), (Hpx - 140) / (yr[1] - yr[0])
    fig = go.Figure()
    for e_, (kname, kcol, kdash) in kinds.items():
        xs, ys, hx, hy, hang = [], [], [], [], []
        for a, b in edges:
            if eff[(a, b)] != e_: continue
            (x0, y0), (x1, y1) = (x[a], y[a]), (x[b], y[b])
            if x0 > x1: (x0, y0), (x1, y1) = (x1, y1), (x0, y0); a_, b_ = b, a                 # 下流の図では、辺の向きと列の並びが逆になる
            dx = x1 - x0
            P = np.array([[x0, y0], [x0 + 0.5 * dx, y0], [x1 - 0.5 * dx, y1], [x1, y1]])
            pt = lambda t: (1 - t) ** 3 * P[0] + 3 * (1 - t) ** 2 * t * P[1] + 3 * (1 - t) * t ** 2 * P[2] + t ** 3 * P[3]
            dv = lambda t: 3 * (1 - t) ** 2 * (P[1] - P[0]) + 6 * (1 - t) * t * (P[2] - P[1]) + 3 * t ** 2 * (P[3] - P[2])
            pts = np.array([pt(t) for t in np.linspace(0, 1, 28)]); xs += list(pts[:, 0]) + [None]; ys += list(pts[:, 1]) + [None]
            te = 0.93 if e_ == -1 else 0.88; p_, v_ = pt(te), dv(te); th = math.degrees(math.atan2(v_[0] * sx, v_[1] * sy))
            hx.append(p_[0]); hy.append(p_[1]); hang.append(th - 90 if e_ == -1 else th)
        if not xs: continue
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=kcol, width=1.4, dash=kdash), opacity=0.75, hoverinfo="skip", name=f"{kname} edge", legendgroup=kname))
        fig.add_trace(go.Scatter(x=hx, y=hy, mode="markers", hoverinfo="skip", showlegend=False, legendgroup=kname,
                                 marker=dict(symbol="line-ns" if e_ == -1 else "arrow", size=11, angle=hang, angleref="up", color=kcol, line=dict(color=kcol, width=2.5) if e_ == -1 else None)))
    add_gene_nodes(fig, [g for g in x if g != TARGET and g in selected], x, y, text_size=12, px_per_x=sx)
    add_gene_nodes(fig, [g for g in x if g != TARGET and g not in selected], x, y, text_size=11, opacity=0.35, legend=False, px_per_x=sx)
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=14, color="#c4c4c4", opacity=0.35, line=dict(color="#444444", width=1.5)),
                             name=f"faded = connector (|r| < {CORR_MIN}, shown only to connect)"))
    fig.add_trace(go.Scatter(x=[x[TARGET]], y=[0], mode="markers+text", text=[TARGET], textposition="top center", textfont=dict(size=15),
                             name="seed (target)", marker=dict(size=22, color="#555555", symbol="diamond"), hoverinfo="skip"))
    for ci, (n, genes) in enumerate(cols): fig.add_annotation(x=ci, y=yr[1] - 0.6, text=n, showarrow=False, font=dict(size=14, color="#666666"))
    fig.update_layout(width=W, height=Hpx, plot_bgcolor="white", margin=dict(l=20, r=20, t=30, b=130), xaxis=dict(visible=False, range=list(xr)), yaxis=dict(visible=False, range=list(yr)),
                      legend=dict(orientation="h", yanchor="top", y=-0.01, xanchor="center", x=0.5, font=dict(size=11)))
    fig.write_html(fname + ".html", include_plotlyjs="cdn")
    try: fig.write_image(fname + ".png", scale=1.5)
    except Exception as e: print("PNG は保存できなかった:", str(e)[:80])
    fig.show()

## 24c. 上流の図（DepMap の相関で選んだ遺伝子 + つなぎ）
- 動作: 左から ligands | receptors | hop の列（遠い順）| seed。選んだ遺伝子とつなぎの遺伝子を、最良経路の辺と `geo ≥ 0.25` の辺（各遺伝子から最大 `MAX_OUT` 本）で結ぶ
- 出力: 図（画面と HTML・PNG に保存）

In [ ]:
# 上流の図: DepMap の相関で選んだ遺伝子 + つなぎ
shown_u = set(sel_up24) | up_conn | {TARGET}
rec_u = sorted(g for g in shown_u if role(g) == "受容体"); lig_u = sorted(g for g in shown_u if role(g) == "リガンド")
mid_u = [g for g in shown_u if g != TARGET and role(g) not in ("受容体", "リガンド")]
cols_u = [("ligands", lig_u), ("receptors", rec_u)] + [(f"hop {h}", sorted(g for g in mid_u if hop_of[g] == h)) for h in sorted({hop_of[g] for g in mid_u}, reverse=True)] + [("seed", [TARGET])]
cols_u = [(n, g_) for n, g_ in cols_u if g_]
xu = {g: ci for ci, (n, g_) in enumerate(cols_u) for g in g_}
tree_u = {(g, nxt13[g]) for g in shown_u if g != TARGET}
ex_u = ev[(ev.geo >= 0.25) & ev.src.isin(shown_u) & ev.dst.isin(shown_u) & (ev.src != ev.dst)]
ex_u = ex_u[ex_u.src.map(xu) < ex_u.dst.map(xu)].sort_values("E", ascending=False).groupby("src").head(MAX_OUT)
edges_u = tree_u | set(zip(ex_u.src, ex_u.dst))
edges_u = {(a, b) for a, b in edges_u if xu[a] < xu[b]}
print(f"描く遺伝子 {len(shown_u) - 1} 個（選んだ {len(sel_up24)} + つなぎ {len(up_conn)}）| 辺 {len(edges_u)} 本")
draw_corr_figure(cols_u, edges_u, {(a, b): eff_lookup[(a, b)] for a, b in edges_u}, set(sel_up24), f"network_development_{TARGET}_upstream_corr", "upstream")

## 24d. 下流の図（DepMap の相関で選んだ遺伝子 + つなぎ）
- 動作: 左から seed | hop の列 | TF（右端）。選んだ遺伝子とつなぎの遺伝子を、最良経路の辺と `geo ≥ 0.25` の辺（各遺伝子から最大 `MAX_OUT` 本）で結ぶ
- 出力: 図（画面と HTML・PNG に保存）

In [ ]:
# 下流の図: DepMap の相関で選んだ遺伝子 + つなぎ
shown_d2 = set(sel_dn24) | dn_conn | {TARGET}
tf_d2 = sorted(g for g in shown_d2 if g in tfs)
mid_d2 = [g for g in shown_d2 if g != TARGET and g not in tfs]
cols_d2 = [("seed", [TARGET])] + [(f"hop {h}", sorted(g for g in mid_d2 if down.hop[g] == h)) for h in sorted({down.hop[g] for g in mid_d2})] + [("transcription factors", tf_d2)]
cols_d2 = [(n, g_) for n, g_ in cols_d2 if g_]
xd2 = {g: ci for ci, (n, g_) in enumerate(cols_d2) for g in g_}
tree_d2 = {(down.prev[g], g) for g in shown_d2 if g != TARGET}
ex_d2 = evd[(evd.geo >= 0.25) & evd.src.isin(shown_d2) & evd.dst.isin(shown_d2) & (evd.src != evd.dst)]
ex_d2 = ex_d2[ex_d2.src.map(xd2) < ex_d2.dst.map(xd2)].sort_values("E", ascending=False).groupby("src").head(MAX_OUT)
edges_d2 = {(a, b) for a, b in (tree_d2 | set(zip(ex_d2.src, ex_d2.dst))) if xd2[a] < xd2[b]}
print(f"描く遺伝子 {len(shown_d2) - 1} 個（選んだ {len(sel_dn24)} + つなぎ {len(dn_conn)}）| 辺 {len(edges_d2)} 本")
draw_corr_figure(cols_d2, edges_d2, {(a, b): eff_d[(a, b)] for a, b in edges_d2}, set(sel_dn24), f"network_development_{TARGET}_downstream_corr", "downstream")

## Step 25. 流れ（証拠つき）の合計で選び、上流と下流を1つの図にする
> 指示: This is just a trial. Please try to visualize by total flow score with evidence.

### 「流れ（証拠つき）」とは
- **経路スコア** = AKT1 からその遺伝子までの最良経路の、辺の `E`（証拠スコア = 論文数と SIGNOR のスコア）の積。**証拠が強い辺でつながる遺伝子ほど高い。**
- **流れ** = その遺伝子を通る、受容体（上流）または TF（下流）の最良経路の**経路スコアの合計**。多くの受容体・TF の経路が、証拠の強い辺を通ってその遺伝子に集まるほど大きい。
- **合計**: 上流の流れ（Step 13）と下流の流れ（Step 21）を、1つの図に並べる。

### 選び方
- **途中のタンパク質**: 上流と下流のそれぞれで、**流れの大きい順に上位 `N_MID_SIDE` 個**（hop ごとの上限は設けない）。
- **受容体 / リガンド / TF**: Step 16・21 の影響の上位（受容体 `N_REC` 個、リガンド `N_LIG` 個、TF `N_TF` 個）。
- **つなぎの遺伝子**: 選んだ受容体・TF から AKT1 までの最良経路のうち、選ばれなかった遺伝子は薄く描く。
- **証拠は辺の太さで出す**: 辺の `E` が 0.6 以上は太く、0.3〜0.6 は中くらい、0.3 未満は細い。
- 点の見た目は Step 17 以降と同じ（大きさ = AKT1 との DepMap の相関の大きさ、縁は同じ、色 = 機能）。

**Note**
- 流れで選ぶと、遺伝子の祖先（AKT1 に近い側の遺伝子）の流れは必ず子孫以上になるので、選んだ途中のタンパク質は、つなぎなしで AKT1 まで連なる。
- 流れは、全受容体・全 TF への経路を数えるので、図に描く受容体・TF とは無関係な経路の遺伝子も入る。

## 25a. 描く遺伝子を選ぶ
- 動作: 上流・下流それぞれで、流れの大きい順に途中のタンパク質を選び、受容体・リガンド・TF は影響の上位を使う。選んだ受容体・TF の最良経路の遺伝子を、つなぎとして足す
- 出力: 選んだ数とつなぎの数、流れの上位の遺伝子

In [ ]:
# 流れ（証拠つき）の大きい順で選ぶ
up_mid_sel = [g for g in flow13.index if g != TARGET and role(g) == "途中"][:N_MID_SIDE]                         # 上流の途中のタンパク質（flow13 は大きい順）
dn_mid_sel = [g for g, v in sorted(flow_d.items(), key=lambda kv: -kv[1]) if g != TARGET and g not in tfs][:N_MID_SIDE]   # 下流の途中のタンパク質
sel_all25 = set(up_mid_sel) | set(sel_rec16) | set(sel_lig16) | set(dn_mid_sel) | set(sel_tf) | {TARGET}
up_conn25 = set(); [up_conn25.update(path_to_target(g)) for g in sel_rec16]
dn_conn25 = set(); [dn_conn25.update(path_from_target(g)) for g in sel_tf]
up_conn25 -= sel_all25; dn_conn25 -= sel_all25
print(f"上流: 途中 {len(up_mid_sel)} + 受容体 {len(sel_rec16)} + リガンド {len(sel_lig16)}、つなぎ {len(up_conn25)} | 下流: 途中 {len(dn_mid_sel)} + TF {len(sel_tf)}、つなぎ {len(dn_conn25)}")
pd.DataFrame({"上流の流れの上位": pd.Series([f"{g} ({flow13.flow[g]:.2f})" for g in up_mid_sel[:12]]), "下流の流れの上位": pd.Series([f"{g} ({flow_d[g]:.2f})" for g in dn_mid_sel[:12]])})

## 25b. 描く
- 動作: **上流と下流の両方に出る遺伝子（SRC、MTOR など）は、両側に別の点として描く**（以前は片側にしか描かず、上流の辺が AKT1 を横切って下流側の点につながっていた）。左から ligands | receptors | 上流の hop（遠い順）| **AKT1（中央）** | 下流の hop | TF（右端）に並べ、辺を曲線で描く。辺の太さは `E`（証拠）、色と線の種類は符号（促進 = 濃い青の実線と矢じり、抑制 = 赤の破線と「T」字）。選んだ遺伝子は濃く、つなぎは薄く
- 出力: 図（画面と HTML・PNG に保存）。選んだ遺伝子の表（流れ・相関つき）も CSV に保存

In [ ]:
# 上流と下流を1つにした図（Plotly）。上流と下流の両方に出る遺伝子（SRC など）は、両側に別の点として描く
up_nodes = set(up_mid_sel) | set(sel_rec16) | set(sel_lig16) | up_conn25
dn_nodes = set(dn_mid_sel) | set(sel_tf) | dn_conn25
both = sorted((up_nodes & dn_nodes) - {TARGET})
dk = lambda g: g if g == TARGET else g + "@d"                                               # 下流側の点のキー（上流側は遺伝子名そのまま）
mid_up = [g for g in up_nodes if role(g) not in ("リガンド", "受容体")]
mid_dn = [g for g in dn_nodes if g not in tfs]
cols25 = ([("ligands", sorted(g for g in up_nodes if role(g) == "リガンド")), ("receptors", sorted(g for g in up_nodes if role(g) == "受容体"))]
          + [(f"up hop {h}", sorted(g for g in mid_up if hop_of[g] == h)) for h in sorted({hop_of[g] for g in mid_up}, reverse=True)]
          + [("seed", [TARGET])]
          + [(f"down hop {h}", sorted(dk(g) for g in mid_dn if down.hop[g] == h)) for h in sorted({down.hop[g] for g in mid_dn})]
          + [("transcription factors", sorted(dk(g) for g in dn_nodes if g in tfs))])
cols25 = [(n, g_) for n, g_ in cols25 if g_]
x25 = {g: ci for ci, (n, g_) in enumerate(cols25) for g in g_}
seed_ci = x25[TARGET]
sel_keys = {g for g in sel_all25 if g in up_nodes or g == TARGET} | {dk(g) for g in sel_all25 if g in dn_nodes}
print(f"上流と下流の両方に出る遺伝子（両側に描く）: {len(both)} 個 → {', '.join(both[:20])}")

Eu, Ed = ev.set_index(["src", "dst"]).E, evd.set_index(["src", "dst"]).E
eff_u, eff_dd = ev.set_index(["src", "dst"]).effect, evd.set_index(["src", "dst"]).effect
edges25 = {}                                                                               # (a, b) -> (E, 符号)。a, b は点のキー
for g in up_nodes:                                                                         # 上流: 最良経路の辺（g -> 次の遺伝子）
    if g in nxt13 and nxt13[g] in x25 and x25[g] < x25[nxt13[g]]: edges25[(g, nxt13[g])] = (Eu[(g, nxt13[g])], eff_u[(g, nxt13[g])])
for g in dn_nodes:                                                                         # 下流: 最良経路の辺（前の遺伝子 -> g）
    if g in down.index:
        a, b = dk(down.prev[g]), dk(g)
        if a in x25 and b in x25 and x25[a] < x25[b]: edges25[(a, b)] = (Ed[(down.prev[g], g)], eff_dd[(down.prev[g], g)])
ex = ev[(ev.geo >= 0.25) & ev.src.isin(up_nodes) & ev.dst.isin(up_nodes | {TARGET}) & (ev.src != ev.dst)]       # 上流のそのほかの辺
ex = ex[ex.src.map(x25) < ex.dst.map(x25)].sort_values("E", ascending=False).groupby("src").head(MAX_OUT)
for a, b in zip(ex.src, ex.dst): edges25.setdefault((a, b), (Eu[(a, b)], eff_u[(a, b)]))
ex = evd[(evd.geo >= 0.25) & evd.src.isin(dn_nodes | {TARGET}) & evd.dst.isin(dn_nodes) & (evd.src != evd.dst)]     # 下流のそのほかの辺
ex = ex.assign(ka=ex.src.map(dk), kb=ex.dst.map(dk))
ex = ex[ex.ka.map(x25) < ex.kb.map(x25)].sort_values("E", ascending=False).groupby("src").head(MAX_OUT)
for a, b, ka, kb in zip(ex.src, ex.dst, ex.ka, ex.kb): edges25.setdefault((ka, kb), (Ed[(a, b)], eff_dd[(a, b)]))

y25 = {TARGET: 0.0}
for ci in range(seed_ci - 1, -1, -1):                                                      # 上流側: AKT1 に近い列から外へ
    genes = cols25[ci][1]
    key = lambda g: (np.mean([y25[b] for (a, b) in edges25 if a == g and b in y25] or [1e9]), g)
    for i, g in enumerate(sorted(genes, key=key)): y25[g] = (len(genes) - 1) / 2 - i
for ci in range(seed_ci + 1, len(cols25)):                                                 # 下流側: AKT1 に近い列から外へ
    genes = cols25[ci][1]
    key = lambda g: (np.mean([y25[a] for (a, b) in edges25 if b == g and a in y25] or [1e9]), g)
    for i, g in enumerate(sorted(genes, key=key)): y25[g] = (len(genes) - 1) / 2 - i

wrap_columns(cols25, x25, y25)
ncol, nmax = len(cols25), max(rows_needed(len(g_)) for n, g_ in cols25)
W, Hpx = 1900, max(700, 48 * nmax + 240)
xr, yr = (-0.4, ncol - 0.2), (-nmax / 2 - 1.5, nmax / 2 + 2.5)
sx, sy = (W - 40) / (xr[1] - xr[0]), (Hpx - 140) / (yr[1] - yr[0])
fig = go.Figure()
first_leg = set()
for e_, (kname, kcol, kdash) in kinds.items():
    for wlab, wcond in (("thick", lambda e: e >= 0.6), ("mid", lambda e: 0.3 <= e < 0.6), ("thin", lambda e: e < 0.3)):
        xs, ys, hx, hy, hang = [], [], [], [], []
        for (a, b), (E_, ef) in edges25.items():
            if ef != e_ or not wcond(E_): continue
            (x0, y0), (x1, y1) = (x25[a], y25[a]), (x25[b], y25[b]); dx = x1 - x0
            P = np.array([[x0, y0], [x0 + 0.5 * dx, y0], [x1 - 0.5 * dx, y1], [x1, y1]])
            pt = lambda t: (1 - t) ** 3 * P[0] + 3 * (1 - t) ** 2 * t * P[1] + 3 * (1 - t) * t ** 2 * P[2] + t ** 3 * P[3]
            dv = lambda t: 3 * (1 - t) ** 2 * (P[1] - P[0]) + 6 * (1 - t) * t * (P[2] - P[1]) + 3 * t ** 2 * (P[3] - P[2])
            pts = np.array([pt(t) for t in np.linspace(0, 1, 28)]); xs += list(pts[:, 0]) + [None]; ys += list(pts[:, 1]) + [None]
            te = 0.93 if e_ == -1 else 0.88; p_, v_ = pt(te), dv(te); th = math.degrees(math.atan2(v_[0] * sx, v_[1] * sy))
            hx.append(p_[0]); hy.append(p_[1]); hang.append(th - 90 if e_ == -1 else th)
        if not xs: continue
        showleg = kname not in first_leg; first_leg.add(kname)
        w_ = 1.8 if wlab == "thick" else 1.1 if wlab == "mid" else 0.6
        fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=kcol, width=w_, dash=kdash), opacity=0.75, hoverinfo="skip", name=f"{kname} edge", legendgroup=kname, showlegend=showleg))
        fig.add_trace(go.Scatter(x=hx, y=hy, mode="markers", hoverinfo="skip", showlegend=False, legendgroup=kname,
                                 marker=dict(symbol="line-ns" if e_ == -1 else "arrow", size=7 + 2 * w_, angle=hang, angleref="up", color=kcol, line=dict(color=kcol, width=2.5) if e_ == -1 else None)))
add_gene_nodes(fig, [g for g in x25 if g != TARGET and g in sel_keys], x25, y25, text_size=12, px_per_x=sx)
add_gene_nodes(fig, [g for g in x25 if g != TARGET and g not in sel_keys], x25, y25, text_size=11, opacity=0.35, legend=False, px_per_x=sx)
fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=14, color="#c4c4c4", opacity=0.35, line=dict(color="#444444", width=1.5)), name="faded = connector (not selected by flow)"))
for lab, w_ in (("evidence E >= 0.6 (thick)", 1.8), ("0.3 <= E < 0.6", 1.1), ("E < 0.3 (thin)", 0.6)):
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="#666666", width=w_), name=lab))
fig.add_trace(go.Scatter(x=[x25[TARGET]], y=[0], mode="markers+text", text=[TARGET], textposition="top center", textfont=dict(size=15), name="seed (target)",
                         marker=dict(size=22, color="#555555", symbol="diamond"), hoverinfo="skip"))
for ci, (n, genes) in enumerate(cols25): fig.add_annotation(x=ci, y=yr[1] - 0.6, text=n, showarrow=False, font=dict(size=13, color="#666666"))
fig.update_layout(width=W, height=Hpx, plot_bgcolor="white", margin=dict(l=20, r=20, t=30, b=150), xaxis=dict(visible=False, range=list(xr)), yaxis=dict(visible=False, range=list(yr)),
                  legend=dict(orientation="h", yanchor="top", y=-0.01, xanchor="center", x=0.5, font=dict(size=11)))
fig.write_html(f"network_development_{TARGET}_totalflow.html", include_plotlyjs="cdn")
try: fig.write_image(f"network_development_{TARGET}_totalflow.png", scale=1.4)
except Exception as e: print("PNG は保存できなかった:", str(e)[:80])
rows = [dict(gene=(g[:-2] if g.endswith("@d") else g), side=("seed" if g == TARGET else "downstream" if g.endswith("@d") else "upstream"), role=("seed" if g == TARGET else role(g[:-2] if g.endswith("@d") else g)),
             selected_by_flow=g in sel_keys, hop=(0 if g == TARGET else int(down.hop[g[:-2]]) if g.endswith("@d") else int(hop_of[g])),
             flow=(flow_d.get(g[:-2], np.nan) if g.endswith("@d") else flow13.flow.get(g, np.nan)), depmap_r=corr_net.get(g[:-2] if g.endswith("@d") else g, np.nan)) for g in x25]
pd.DataFrame(rows).sort_values(["side", "flow"], ascending=[True, False]).to_csv(f"network_development_{TARGET}_totalflow_genes.csv", index=False, encoding="utf-8-sig")
print(f"描く点 {len(x25) - 1} 個 | 辺 {len(edges25)} 本（E>=0.6: {sum(v[0] >= 0.6 for v in edges25.values())}、0.3-0.6: {sum(0.3 <= v[0] < 0.6 for v in edges25.values())}、<0.3: {sum(v[0] < 0.3 for v in edges25.values())}）")
fig.show()

## Step 26. 汎用の「流れの図」ノートブック用に、辺と遺伝子の表を保存する
> 指示: 遺伝子名を入力するとどんな遺伝子でも同じように作成される図（`flow_view.ipynb`）の準備

- 動作: ライセンス判定を通ったネットワーク全体（Step 3 の `net`、**TF から出る辺も残す**）の各辺に、論文数・SIGNOR スコア・`E`・`geo` を付けて `flow_view_edges.csv` に保存する。遺伝子ごとの機能（点の色）・局在・UniProt の印を `flow_view_genes.csv` に保存する
- 出力: 2 つの CSV（`flow_view.ipynb` が読む）。どちらも、特定の遺伝子（AKT1）には依存しない

In [ ]:
# flow_view.ipynb 用に、辺と遺伝子の表を保存する（AKT1 に依存しない）
evall = net.copy()
key_ = pd.MultiIndex.from_frame(evall[["src", "dst"]])
evall["refs"] = oe_full.score.reindex(key_).values; evall["signor"] = sg_full.reindex(key_).values
A_ = np.log1p(evall.refs.fillna(0)) / np.log1p(oe_full.score.max()); B_ = evall.signor.fillna(0)
evall["E"] = (1 - (1 - A_) * (1 - B_)).round(4)
evall["geo"] = np.sqrt(evall.weight * evall.E).round(4)
evall["mechanism"] = evall.mechanism.astype(str).str[:90]
evall[["src", "dst", "effect", "mechanism", "residue", "source", "weight", "refs", "signor", "E", "geo"]].to_csv("flow_view_edges.csv", index=False)

genes_all = sorted(set(evall.src) | set(evall.dst))
gt = pd.DataFrame({"gene": genes_all, "func": [func_cat(g) for g in genes_all], "locations": [loc_txt.get(g, "") for g in genes_all], "uniprot_flags": [flag_txt.get(g, "") for g in genes_all]})
gt.to_csv("flow_view_genes.csv", index=False, encoding="utf-8-sig")
print("辺:", len(evall), "| 遺伝子:", len(gt), "| 機能の内訳:", gt.func.value_counts().to_dict())